## Block 1：导入依赖、设置绘图环境、固定随机种子

### 这个代码块在做什么
- 导入整份 notebook 后面会用到的库。
- 统一 `matplotlib / seaborn` 的画图风格。
- 固定随机种子，尽量让实验结果可重复。

### 你运行后应该看到什么
- 正常情况下只会打印依赖和随机种子设置成功的信息。
- 如果这里报错，通常是缺少 Python 包，后面的所有 block 都会受影响。


In [1]:
# 中文导读：这个代码块负责导入整份实验要用到的库，并统一随机种子与绘图环境。
# =============================================================================
# Block 1. 导入依赖与固定随机种子
# 作用：
# 1) 统一导入整个 notebook 需要用到的库；
# 2) 固定随机种子，减少实验结果漂移；
# 3) 关闭不必要的警告，提升 notebook 可读性。
# =============================================================================
from __future__ import annotations  

import gc  # garbage collection support
import io  # in-memory buffer helpers for profiling
import json  # JSON 编码和解码模块，用于处理 JSON 数据
import math  # 数学函数库，提供数学运算和常数
import os  # 操作系统接口模块，用于与操作系统进行交互，如文件和目录操作
import copy  # 深拷贝模块（统一在 Block 1 导入，后续不再重复）
import random
import re  # 正则表达式模块，用于字符串匹配和处理
import sys  # 系统特定参数和函数模块，用于与操作系统进行交互
import time  # 时间模块，用于处理时间相关的函数
import warnings
from collections import Counter, defaultdict  # 用于计数和创建默认字典
from dataclasses import asdict, dataclass, field  # 用于创建数据类，简化类的定义
from pathlib import Path  # 用于处理文件路径，提供面向对象的路径操作
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import joblib   # 用于并行计算和模型持久化
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy.signal import butter, medfilt, sosfiltfilt  # 用于信号处理的函数
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

print("✅ Block 1 完成：依赖导入成功，随机种子已固定。")



✅ Block 1 完成：依赖导入成功，随机种子已固定。


### Block 1 运行后怎么判断正常
- 没有 `ModuleNotFoundError` 就说明依赖层基本没问题。
- 如果后面画图风格异常，通常也可以回到这个 block 检查。


## Block 2：定义全局实验配置

### 这个代码块在做什么
- 用 `dataclass` 统一管理数据路径、特征参数、评估参数和导出参数。
- 把后面所有实验共享的默认设置集中起来，避免参数散落在各个代码块里。

### 运行前你需要重点确认
- `data_root` 是否指向你的 CHB-MIT 数据目录。
- `patient_ids`、`history_epochs`、滤波参数是否符合当前实验目标。

### 输出怎么解读
- 这里打印的是“当前全局配置摘要”。
- 如果路径、病人数、历史窗长度看起来不对，应该先在这里改，再继续往后跑。


In [2]:
# 中文导读：这里集中定义全局实验参数，后续所有 block 都会从 CFG 或其变体读取配置。
# =============================================================================
# Block 2. 全局配置
# 作用：
# 1) 用 dataclass 管理实验参数；
# 2) 统一 XGBoost 调参、验证、导出配置；
# 3) 默认面向 10 位病人（chb01-chb10）。
# =============================================================================
@dataclass
class FeatureConfig:
    epoch_len_s: int = 2
    bandpass_low_hz: float = 0.5
    bandpass_high_hz: float = 50.0
    bandpass_method: str = "butter_sos"   # "butter_sos" or "fir_zero"
    butter_order: int = 4
    band_edges_hz: Tuple[int, ...] = tuple(range(0, 42, 2))   # 0,2,4,...,40
    synchrony_pairs_names: Tuple[Tuple[str, str], ...] = (
        ("FP1-F3", "FP2-F4"),
        ("F7-T7", "F8-T8"),
        ("C3-P3", "C4-P4"),
    )
    history_epochs: int = 3
    scale_to_uV: bool = True
    channel_missing_policy: str = "strict"   # "strict" or "zero_fill"
    channel_subset_names: Optional[Tuple[str, ...]] = None


@dataclass
class EvalConfig:
    patient_ids: Tuple[str, ...] = tuple(f"chb{i:02d}" for i in range(1, 11))
    threshold_grid: Tuple[float, ...] = tuple(np.round(np.linspace(0.10, 0.95, 50), 3))
    min_duration_epochs: int = 3
    min_acceptable_sensitivity: float = 0.70
    neg_to_pos_ratio: int = 10
    min_neg_samples: int = 3000
    default_threshold: float = 0.50
    fixed_threshold_mode: bool = True
    fixed_threshold_value: float = 0.50

    svm_c: float = 1.0
    svm_gamma: str = "scale"
    rf_n_estimators: int = 300
    rf_max_depth: Optional[int] = None
    rf_min_samples_leaf: int = 1
    rf_max_features: str = "sqrt"
    rf_n_jobs: int = -1

    # XGBoost params
    xgb_n_estimators: int = 100
    xgb_max_depth: int = 6
    xgb_learning_rate: float = 0.05
    xgb_subsample: float = 0.9
    xgb_colsample_bytree: float = 0.9
    xgb_reg_lambda: float = 1.0
    xgb_min_child_weight: float = 1.0
    xgb_gamma: float = 0.0
    xgb_tree_method: str = "hist"
    xgb_n_jobs: int = -1

    top_k_features: int = 50
    selector_n_estimators: int = 80
    selector_max_depth: int = 4
    random_state: int = GLOBAL_SEED


@dataclass
class ExportConfig:
    export_subdir: str = "exported_models"
    model_filename: str = "rf_lightweight_model.joblib"
    metadata_filename: str = "rf_lightweight_metadata.json"
    service_script_filename: str = "serve_model.py"
    client_script_filename: str = "example_client.py"


@dataclass
class ExperimentConfig:
    data_root: str = r"D:\EEG_Data\chb-mit-scalp-eeg-database-1.0.0"
    cache_subdir: str = "feature_cache_refactored"
    feature: FeatureConfig = field(default_factory=FeatureConfig)
    eval: EvalConfig = field(default_factory=EvalConfig)
    export: ExportConfig = field(default_factory=ExportConfig)

    @property
    def channels(self) -> List[str]:
        """Cached: avoids rebuilding the list on every access (optimization #10+#17)."""
        if not hasattr(self, "_channels_cached"):
            ref_ch_names = [
                "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
                "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
                "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
                "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
                "FZ-CZ", "CZ-PZ", "P7-T7", "T7-FT9",
                "FT9-FT10", "FT10-T8",
            ]
            base_channels = list(dict.fromkeys(ref_ch_names))
            subset_names = getattr(self.feature, "channel_subset_names", None)
            if subset_names:
                def _norm_channel_local(name: str) -> str:
                    return re.sub(r"\s+", "", str(name).upper())

                base_lookup = {_norm_channel_local(ch): ch for ch in base_channels}
                subset_norm = [_norm_channel_local(ch) for ch in subset_names]
                missing_subset = [ch for ch in subset_norm if ch not in base_lookup]
                if missing_subset:
                    raise ValueError(f"channel_subset_names contains unknown channels: {missing_subset}")
                self._channels_cached = [base_lookup[ch] for ch in subset_norm]
            else:
                self._channels_cached = base_channels
        return self._channels_cached

    @property
    def cache_dir(self) -> Path:
        return Path(self.data_root) / self.cache_subdir

    @property
    def export_dir(self) -> Path:
        return Path(self.data_root) / self.export.export_subdir

    def variant(self, **overrides) -> "ExperimentConfig":
        """Create a variant of this config with specified overrides.
        Replaces scattered CFG.variant() + manual field assignment pattern.
        Supports nested overrides via dot notation keys, e.g.:
            CFG.variant(eval__top_k_features=30, feature__history_epochs=5)
        """
        import copy as _copy
        new_cfg = _copy.deepcopy(self)
        for key, value in overrides.items():
            parts = key.split("__")
            obj = new_cfg
            for part in parts[:-1]:
                obj = getattr(obj, part)
            setattr(obj, parts[-1], value)
        # Invalidate hash cache for the new instance
        if hasattr(new_cfg, "_channels_cached"):
            delattr(new_cfg, "_channels_cached")
        return new_cfg



CFG = ExperimentConfig()

print("当前配置摘要：")
print(f"  data_root              = {CFG.data_root}")
print(f"  cache_dir              = {CFG.cache_dir}")
print(f"  patient_ids            = {CFG.eval.patient_ids}")
print(f"  channel_missing_policy = {CFG.feature.channel_missing_policy}")
print(f"  history_epochs         = {CFG.feature.history_epochs} (总上下文 = {(CFG.feature.history_epochs + 1) * CFG.feature.epoch_len_s} 秒)")
print(f"  base_channels          = {len(CFG.channels)}")
print(f"  spectral_bands         = {len(CFG.feature.band_edges_hz) - 1}")
print(f"  xgb_tree_method        = {CFG.eval.xgb_tree_method}")
print(f"  fixed_threshold_mode   = {CFG.eval.fixed_threshold_mode} (value={CFG.eval.fixed_threshold_value:.2f})")
print("✅ Block 2 完成：全局配置已创建。")




当前配置摘要：
  data_root              = D:\EEG_Data\chb-mit-scalp-eeg-database-1.0.0
  cache_dir              = D:\EEG_Data\chb-mit-scalp-eeg-database-1.0.0\feature_cache_refactored
  patient_ids            = ('chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb08', 'chb09', 'chb10')
  channel_missing_policy = strict
  history_epochs         = 3 (总上下文 = 8 秒)
  base_channels          = 22
  spectral_bands         = 20
  xgb_tree_method        = hist
  fixed_threshold_mode   = True (value=0.50)
✅ Block 2 完成：全局配置已创建。


### Block 2 输出解读补充
- `history_epochs = 3` 表示模型输入会使用“当前 epoch + 3 个历史 epoch”。
- 这里的配置是全局默认值，后面的搜索 block 会在这个基础上生成变体。


## Block 2B：运行开关与快速回归入口

### 这个代码块在做什么
- 决定哪些重型步骤要真实执行，哪些步骤直接复用已经验证过的结果。
- 给 Phase A / Phase B 的最优参数提供“预计算 winner”，让我们在不重跑整套搜索的情况下完成主验证链。

### 输出怎么解读
- 这里会打印所有 `RUN_*` 开关的当前值。
- 如果你只是复查最终结果，通常让搜索类开关为 `False` 更省时间。
- 如果你想完全从零重跑，再把对应开关改成 `True`。


In [3]:
# Block 2B. Run Control
RUN_BUILD_CACHE            = False
RUN_PHASE_A_SEARCH         = False
RUN_PHASE_B_SEARCH         = False
RUN_MULTI_MODEL_BENCHMARK  = False
RUN_SVM_BENCHMARK          = False
RUN_RF_FULL_COMPARE        = False
RUN_LATENCY_PROFILING      = False
RUN_C_EXPORT               = False
RUN_PATIENT_EXPORT         = False
RUN_STREAM_DEMO            = False
RUN_CLINICAL_ERROR_ANALYSIS = True

PRECOMPUTED_PHASE_A_WINNER = {
    "history_epochs": 3,
    "bandpass_method": "butter_sos",
    "butter_order": 4,
    "top_k_fixed": 50,
    "mean_sensitivity": 0.9550000000000001,
    "median_far_per_hour": 0.26165729497180734,
    "mean_delay_s": 7.630952380952381,
    "patients": 10,
}

PRECOMPUTED_PHASE_B_WINNER = {
    "top_k": 30,
    "rf_n_estimators": 500,
    "rf_max_depth": 12,
    "rf_min_samples_leaf": 1,
    "rf_max_features": "sqrt",
    "mean_sensitivity": 0.9800000000000001,
    "median_far_per_hour": 0.24547462194392256,
    "mean_delay_s": 10.64452380952381,
    "patients": 10,
}


def build_precomputed_best_feature_cfg() -> ExperimentConfig:
    cfg = CFG.variant()
    cfg.feature.history_epochs = int(PRECOMPUTED_PHASE_A_WINNER["history_epochs"])
    cfg.feature.bandpass_method = str(PRECOMPUTED_PHASE_A_WINNER["bandpass_method"])
    cfg.feature.butter_order = int(PRECOMPUTED_PHASE_A_WINNER["butter_order"])
    cfg.eval.fixed_threshold_mode = False
    return cfg


def build_precomputed_tuned_rf_cfg() -> ExperimentConfig:
    cfg = build_precomputed_best_feature_cfg().variant()
    cfg.eval.top_k_features = int(PRECOMPUTED_PHASE_B_WINNER["top_k"])
    cfg.eval.rf_n_estimators = int(PRECOMPUTED_PHASE_B_WINNER["rf_n_estimators"])
    cfg.eval.rf_max_depth = int(PRECOMPUTED_PHASE_B_WINNER["rf_max_depth"])
    cfg.eval.rf_min_samples_leaf = int(PRECOMPUTED_PHASE_B_WINNER["rf_min_samples_leaf"])
    cfg.eval.rf_max_features = str(PRECOMPUTED_PHASE_B_WINNER["rf_max_features"])
    cfg.eval.fixed_threshold_mode = CFG.eval.fixed_threshold_mode
    cfg.eval.fixed_threshold_value = CFG.eval.fixed_threshold_value
    return cfg


print("Block 2B ready.")
print(f"  RUN_BUILD_CACHE            = {RUN_BUILD_CACHE}")
print(f"  RUN_PHASE_A_SEARCH         = {RUN_PHASE_A_SEARCH}")
print(f"  RUN_PHASE_B_SEARCH         = {RUN_PHASE_B_SEARCH}")
print(f"  RUN_MULTI_MODEL_BENCHMARK  = {RUN_MULTI_MODEL_BENCHMARK}")
print(f"  RUN_SVM_BENCHMARK          = {RUN_SVM_BENCHMARK}")
print(f"  RUN_RF_FULL_COMPARE        = {RUN_RF_FULL_COMPARE}")
print(f"  RUN_LATENCY_PROFILING      = {RUN_LATENCY_PROFILING}")
print(f"  RUN_C_EXPORT               = {RUN_C_EXPORT}")
print(f"  RUN_PATIENT_EXPORT         = {RUN_PATIENT_EXPORT}")
print(f"  RUN_STREAM_DEMO            = {RUN_STREAM_DEMO}")
print(f"  RUN_CLINICAL_ERROR_ANALYSIS = {RUN_CLINICAL_ERROR_ANALYSIS}")

PIPELINE_RESULTS = {}
gc.collect()


Block 2B ready.
  RUN_BUILD_CACHE            = False
  RUN_PHASE_A_SEARCH         = False
  RUN_PHASE_B_SEARCH         = False
  RUN_MULTI_MODEL_BENCHMARK  = False
  RUN_SVM_BENCHMARK          = False
  RUN_RF_FULL_COMPARE        = False
  RUN_LATENCY_PROFILING      = False
  RUN_C_EXPORT               = False
  RUN_PATIENT_EXPORT         = False
  RUN_STREAM_DEMO            = False
  RUN_CLINICAL_ERROR_ANALYSIS = True


0

### Block 2B 输出解读补充
- 建议优先把“高耗时但已经完成”的步骤关掉。
- 这样你可以把时间留给真正要复查的 benchmark 或最终验证。


## Block 3：基础工具函数

### 这个代码块在做什么
- 定义目录创建、JSON 读写、配置哈希、缓存命名等基础工具。
- 这些函数本身不训练模型，但几乎所有后续 block 都依赖它们。

### 输出怎么解读
- 这里会打印当前缓存版本号和当前配置哈希。
- 如果后面出现“缓存配置不一致”，通常要回到这里理解哈希变化来自哪里。


In [4]:
# 中文导读：这里放的是最底层工具函数，主要服务于缓存、哈希、导出和文件组织。
# =============================================================================
# Block 3. 基础工具函数
# 作用：
# 1) 统一创建目录；
# 2) 生成配置哈希；
# 3) 保存/加载 JSON；
# 4) 给缓存和导出文件提供稳定命名；
# 5) 通过 CACHE_SCHEMA_VERSION 强制区分不同代码版本产生的缓存。
# =============================================================================
CACHE_SCHEMA_VERSION = "v2_summary_parser_fix"

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def make_json_safe(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, tuple):
        return [make_json_safe(x) for x in obj]
    if isinstance(obj, list):
        return [make_json_safe(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    return obj


_CONFIG_HASH_CACHE: Dict[int, str] = {}

def config_to_hash(cfg: ExperimentConfig) -> str:
    """Cached: avoids re-serializing config on every call (optimization #8).
    Clinical error analysis compatibility note:
    - channel_subset_names = None should behave like the original notebook and
      should not invalidate existing baseline caches.
    - non-empty channel_subset_names must still produce a distinct hash.
    """
    obj_id = id(cfg)
    cached = _CONFIG_HASH_CACHE.get(obj_id)
    if cached is not None:
        return cached
    cfg_dict = make_json_safe(asdict(cfg))
    feature_dict = cfg_dict.get("feature", {})
    if feature_dict.get("channel_subset_names") in (None, [], ()):
        feature_dict.pop("channel_subset_names", None)
    hash_payload = {
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "config": cfg_dict,
    }
    cfg_str = json.dumps(hash_payload, sort_keys=True, ensure_ascii=False)
    result = joblib.hash(cfg_str)
    _CONFIG_HASH_CACHE[obj_id] = result
    return result


def save_json(data: Dict[str, Any], path: Path) -> None:
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(data), f, ensure_ascii=False, indent=2)


def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_patient_cache_path(cfg: ExperimentConfig, patient_id: str) -> Path:
    cfg_hash = config_to_hash(cfg)[:12]
    ensure_dir(cfg.cache_dir)
    return cfg.cache_dir / f"{patient_id}_features_{cfg_hash}.joblib"


def get_export_paths(cfg: ExperimentConfig) -> Dict[str, Path]:
    export_dir = ensure_dir(cfg.export_dir)
    return {
        "model": export_dir / cfg.export.model_filename,
        "metadata": export_dir / cfg.export.metadata_filename,
        "service_script": export_dir / cfg.export.service_script_filename,
        "client_script": export_dir / cfg.export.client_script_filename,
    }


print("当前缓存版本:", CACHE_SCHEMA_VERSION)
print("当前配置哈希:", config_to_hash(CFG))
print("✅ Block 3 完成：工具函数可用。")



当前缓存版本: v2_summary_parser_fix
当前配置哈希: 42bda03c8a1c833100a17747b4947648
✅ Block 3 完成：工具函数可用。


### Block 3 输出解读补充
- 配置哈希变了不一定是坏事，但意味着旧缓存通常不能直接复用。
- 如果你做了参数修改，看到哈希变化是正常现象。


## Block 4：EDF 读取、通道对齐、标签解析

### 这个代码块在做什么
- 负责从原始 EDF 中读取 EEG。
- 统一通道命名、处理缺失通道、把 `summary.txt` 里的发作区间转换成标签。

### 为什么重要
- 这一层如果对齐错了，后面的特征、模型、指标都会一起偏掉。
- 这也是“数据工程正确性”最关键的一层。

### 输出怎么解读
- 正常情况下这里只是定义函数，不会产生大表格。
- 如果这里报错，通常和 EDF 文件、通道名、summary 文件格式有关。


In [5]:
# 中文导读：这里负责把原始 EDF 和 summary 文件转换成后续可建模的数据结构。
# =============================================================================
# Block 4. 数据读取与标签辅助函数
# 作用：
# 1) 解析 CHB-MIT 的 summary 标注；
# 2) 规范化 EDF 通道名；
# 3) 严格/鲁棒地对齐目标导联；
# 4) 基于半开区间生成 epoch 级标签。
# =============================================================================
def parse_summary_to_dict(summary_path: Path) -> Dict[str, List[Tuple[int, int]]]:
    """
    解析 CHB-MIT 的 patient summary，兼容以下常见变体：
    1) File Name: chb01_03.edf
    2) File Name: chb02_16+.edf
    3) Seizure Start Time: 2996 seconds
    4) Seizure 1 Start Time: 2996 seconds
    """
    seizure_dict: Dict[str, List[Tuple[int, int]]] = {}
    current_file: Optional[str] = None
    current_start: Optional[int] = None

    if not summary_path.exists():
        return seizure_dict

    file_pattern = re.compile(r"File Name:\s*([^\s]+\.edf)", flags=re.IGNORECASE)
    start_pattern = re.compile(
        r"Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s*(?:seconds?)?",
        flags=re.IGNORECASE,
    )
    end_pattern = re.compile(
        r"Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s*(?:seconds?)?",
        flags=re.IGNORECASE,
    )

    with open(summary_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw_line in f:
            line = raw_line.strip()

            file_match = file_pattern.search(line)
            if file_match:
                current_file = Path(file_match.group(1)).name
                seizure_dict.setdefault(current_file, [])
                current_start = None
                continue

            start_match = start_pattern.search(line)
            if start_match:
                current_start = int(start_match.group(1))
                continue

            end_match = end_pattern.search(line)
            if end_match and current_file is not None and current_start is not None:
                current_end = int(end_match.group(1))
                if current_end > current_start:
                    seizure_dict[current_file].append((current_start, current_end))
                current_start = None

    return seizure_dict

def normalize_channel_name(name: str) -> str:
    name = name.strip().upper()
    name = re.sub(r"-\d+$", "", name)
    name = re.sub(r"\s+", "", name)
    return name


def deduplicate_and_normalize_raw(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    original_names = raw.ch_names
    normalized_names = [normalize_channel_name(ch) for ch in original_names]

    seen = set()
    to_drop = []
    rename_map = {}
    for old_name, new_name in zip(original_names, normalized_names):
        if new_name in seen:
            to_drop.append(old_name)
        else:
            seen.add(new_name)
            rename_map[old_name] = new_name

    if to_drop:
        raw.drop_channels(to_drop)
    raw.rename_channels(rename_map)
    return raw


def align_channels(
    raw: mne.io.BaseRaw,
    target_channels: Sequence[str],
    policy: str = "strict",
) -> Tuple[np.ndarray, Dict[str, Any]]:
    normalized_targets = [normalize_channel_name(ch) for ch in target_channels]

    # Fetch full EDF matrix once to avoid repeated disk IO per channel.
    raw_data = raw.get_data()
    existing_names = [normalize_channel_name(ch) for ch in raw.ch_names]
    name_to_idx = {name: idx for idx, name in enumerate(existing_names)}

    n_ch = len(normalized_targets)
    missing_channels = []
    reversed_channels = []
    n_times = raw_data.shape[1]

    # Pre-allocate output matrix instead of list + vstack
    data = np.zeros((n_ch, n_times), dtype=raw_data.dtype)

    for i, target in enumerate(normalized_targets):
        idx = name_to_idx.get(target)
        if idx is not None:
            data[i] = raw_data[idx]
            continue

        if "-" in target:
            reverse_target = "-".join(target.split("-")[::-1])
            reverse_idx = name_to_idx.get(reverse_target)
            if reverse_idx is not None:
                np.negative(raw_data[reverse_idx], out=data[i])
                reversed_channels.append(target)
                continue

        if policy == "zero_fill":
            # data[i] is already zeros
            missing_channels.append(target)
        else:
            raise ValueError(f"Missing target channel: {target}")

    info = {
        "missing_channels": missing_channels,
        "missing_count": len(missing_channels),
        "reversed_channels": reversed_channels,
    }
    return data, info
def build_epoch_labels(
    n_epochs: int,
    epoch_len_s: int,
    seizure_intervals: Sequence[Tuple[int, int]],
) -> np.ndarray:
    """Vectorized: broadcast over epochs instead of Python for-loop."""
    labels = np.zeros(n_epochs, dtype=np.int8)
    if not seizure_intervals:
        return labels
    epoch_starts = np.arange(n_epochs) * epoch_len_s
    epoch_ends = epoch_starts + epoch_len_s
    for sz_start, sz_end in seizure_intervals:
        overlap = (epoch_starts < sz_end) & (sz_start < epoch_ends)
        labels[overlap] = 1
    return labels


print("✅ Block 4 完成：数据读取/通道对齐/标签工具已定义。")



✅ Block 4 完成：数据读取/通道对齐/标签工具已定义。


### Block 4 输出解读补充
- 这个 block 没有大输出并不代表不重要。
- 它属于“静默但关键”的数据准备层。


## Block 5：滤波、特征提取与时序堆叠

### 这个代码块在做什么
- 对 EEG 做带通滤波。
- 提取频带能量、统计量等基础特征。
- 用 `history_epochs` 把当前 epoch 和历史 epoch 堆叠成最终输入特征。

### 输出怎么解读
- 这个 block 主要是定义特征工程函数。
- 真正的特征维度、缓存大小，会在后面的缓存构建和评测结果里体现出来。


In [6]:
# 中文导读：这里把 EEG 变成模型能吃的特征，是“原始信号 -> 特征矩阵”的关键桥梁。
# =============================================================================
# Block 5. 滤波、特征提取与时序堆叠
# 作用：
# 1) 对整段 EEG 做带通滤波；
# 2) 以“整文件向量化”的方式提取 epoch 特征；
# 3) 明确 history_epochs 的语义，生成最终堆叠特征。
# =============================================================================
_SOS_CACHE: Dict[Tuple[int, float, float, int], np.ndarray] = {}
_BAND_MASK_CACHE: Dict[Tuple[int, int, Tuple[int, ...]], List[np.ndarray]] = {}
_SYNC_INDEX_CACHE: Dict[Tuple[Tuple[str, ...], Tuple[Tuple[str, str], ...]], List[Tuple[int, int]]] = {}

def get_band_tuples(cfg: ExperimentConfig) -> List[Tuple[str, Tuple[int, int]]]:
    edges = cfg.feature.band_edges_hz
    bands = []
    for low, high in zip(edges[:-1], edges[1:]):
        bands.append((f"{low}-{high}Hz", (int(low), int(high))))
    return bands


def get_synchrony_index_pairs(cfg: ExperimentConfig) -> List[Tuple[int, int]]:
    channels_norm = tuple(normalize_channel_name(ch) for ch in cfg.channels)
    pairs_norm = tuple((normalize_channel_name(a), normalize_channel_name(b)) for a, b in cfg.feature.synchrony_pairs_names)
    key = (channels_norm, pairs_norm)

    cached = _SYNC_INDEX_CACHE.get(key)
    if cached is not None:
        return cached

    channel_to_idx = {ch: i for i, ch in enumerate(channels_norm)}
    pairs = []
    for left_name, right_name in pairs_norm:
        pairs.append((channel_to_idx[left_name], channel_to_idx[right_name]))

    _SYNC_INDEX_CACHE[key] = pairs
    return pairs

def bandpass_filter_multich(
    data: np.ndarray,
    fs: int,
    lowcut: float,
    highcut: float,
    method: str = "butter_sos",
    butter_order: int = 4,
) -> np.ndarray:
    method = str(method).lower()

    if method == "butter_sos":
        key = (int(fs), float(lowcut), float(highcut), int(butter_order))
        sos = _SOS_CACHE.get(key)
        if sos is None:
            nyquist = 0.5 * fs
            if highcut >= nyquist:
                raise ValueError(f"highcut={highcut} must be < Nyquist {nyquist}.")
            sos = butter(int(butter_order), [lowcut / nyquist, highcut / nyquist], btype="bandpass", output="sos")
            _SOS_CACHE[key] = sos
        return sosfiltfilt(sos, data, axis=1)

    if method == "fir_zero":
        return mne.filter.filter_data(
            data=data,
            sfreq=float(fs),
            l_freq=float(lowcut),
            h_freq=float(highcut),
            method="fir",
            phase="zero",
            verbose=False,
        )

    raise ValueError(f"Unsupported bandpass method: {method}")

def _get_band_masks(fs: int, epoch_samples: int, band_edges: Sequence[int]) -> List[np.ndarray]:
    edges = tuple(int(x) for x in band_edges)
    key = (int(fs), int(epoch_samples), edges)
    cached = _BAND_MASK_CACHE.get(key)
    if cached is not None:
        return cached

    freqs = np.fft.rfftfreq(epoch_samples, d=1.0 / fs)
    masks = []
    for idx, (low, high) in enumerate(zip(edges[:-1], edges[1:])):
        is_last = idx == len(edges) - 2
        if is_last:
            mask = (freqs >= low) & (freqs <= high)
        else:
            mask = (freqs >= low) & (freqs < high)
        masks.append(mask)

    _BAND_MASK_CACHE[key] = masks
    return masks

def extract_base_features_for_file(
    data_bp: np.ndarray,
    fs: int,
    cfg: ExperimentConfig,
) -> Tuple[np.ndarray, np.ndarray]:
    epoch_len_s = cfg.feature.epoch_len_s
    epoch_samples = epoch_len_s * fs
    n_epochs = data_bp.shape[1] // epoch_samples

    if n_epochs == 0:
        return np.empty((0, 0), dtype=float), np.empty((0,), dtype=int)

    usable = data_bp[:, : n_epochs * epoch_samples]
    epochs = usable.reshape(data_bp.shape[0], n_epochs, epoch_samples).transpose(1, 0, 2)
    # Center once, reuse for both FFT and sync (optimization #2)
    epochs_centered = epochs - epochs.mean(axis=2, keepdims=True)

    fft_vals = np.fft.rfft(epochs_centered, axis=-1)
    # One-step power: avoid sqrt inside np.abs then squaring back (optimization #5)
    power = fft_vals.real ** 2 + fft_vals.imag ** 2

    band_masks = _get_band_masks(fs, epoch_samples, cfg.feature.band_edges_hz)
    band_features = [power[:, :, mask].sum(axis=-1) for mask in band_masks]

    band_cube = np.stack(band_features, axis=-1)
    band_flat = band_cube.reshape(n_epochs, -1)

    sync_values = []
    for idx_a, idx_b in get_synchrony_index_pairs(cfg):
        sig_a = epochs_centered[:, idx_a, :]
        sig_b = epochs_centered[:, idx_b, :]

        numerator = np.sum(sig_a * sig_b, axis=1)
        denominator = np.sqrt(np.sum(sig_a ** 2, axis=1) * np.sum(sig_b ** 2, axis=1))
        corr = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 0)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
        sync_values.append(corr)

    sync_matrix = np.stack(sync_values, axis=1) if sync_values else np.empty((n_epochs, 0), dtype=float)
    X_base = np.hstack([band_flat, sync_matrix]).astype(np.float32)

    return X_base, np.arange(n_epochs, dtype=int)


def temporal_stack_features(
    X_base: np.ndarray,
    y: np.ndarray,
    history_epochs: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Pre-allocate output matrix instead of vstack + hstack (optimization #4)."""
    if X_base.shape[0] != len(y):
        raise ValueError("X_base and y length mismatch.")

    n_epochs, base_dim = X_base.shape
    total_dim = base_dim * (history_epochs + 1)
    X_stacked = np.zeros((n_epochs, total_dim), dtype=np.float32)

    col = 0
    for lag in range(history_epochs, -1, -1):
        dst = X_stacked[:, col : col + base_dim]
        if lag == 0:
            dst[:] = X_base
        else:
            dst[lag:] = X_base[:-lag]
        col += base_dim

    return X_stacked, y.copy()

_BASE_FEATURE_NAMES_CACHE: Dict[int, List[str]] = {}

def build_base_feature_names(cfg: ExperimentConfig) -> List[str]:
    """Cached: result depends only on cfg (optimization #9)."""
    obj_id = id(cfg)
    cached = _BASE_FEATURE_NAMES_CACHE.get(obj_id)
    if cached is not None:
        return cached
    names = []
    bands = get_band_tuples(cfg)

    for ch in cfg.channels:
        for band_name, _ in bands:
            names.append(f"{ch} [{band_name}]")

    for idx_a, idx_b in get_synchrony_index_pairs(cfg):
        names.append(f"Sync: {cfg.channels[idx_a]} & {cfg.channels[idx_b]}")

    _BASE_FEATURE_NAMES_CACHE[obj_id] = names
    return names


_STACKED_FEATURE_NAMES_CACHE: Dict[int, List[str]] = {}

def build_stacked_feature_names(cfg: ExperimentConfig) -> List[str]:
    """Cached: result depends only on cfg (optimization #9)."""
    obj_id = id(cfg)
    cached = _STACKED_FEATURE_NAMES_CACHE.get(obj_id)
    if cached is not None:
        return cached
    base_names = build_base_feature_names(cfg)
    names = []
    history = cfg.feature.history_epochs

    for lag in range(history, 0, -1):
        for name in base_names:
            names.append(f"{name} @ t-{lag}")
    for name in base_names:
        names.append(f"{name} @ t")

    _STACKED_FEATURE_NAMES_CACHE[obj_id] = names
    return names


base_dim = len(build_base_feature_names(CFG))
stacked_dim = len(build_stacked_feature_names(CFG))
print(f"base_dim    = {base_dim}")
print(f"stacked_dim = {stacked_dim}")
print("✅ Block 5 完成：特征工程函数已定义。")

base_dim    = 443
stacked_dim = 1772
✅ Block 5 完成：特征工程函数已定义。


### Block 5 输出解读补充
- 如果你后面看到特征维度和预期不一致，通常要回来检查这里的频带数、通道数和时间堆叠规则。


## Block 6：指标、负采样与阈值选择

### 这个代码块在做什么
- 定义事件级 `Sensitivity / FAR_per_Hour / Delay` 的计算规则。
- 定义训练时的正负样本采样策略。
- 定义验证集阈值搜索逻辑，避免直接用测试集挑阈值。

### 输出怎么解读
- 这里本身不会给你最终模型表现，但它决定了后面指标怎么算。
- 如果你想解释论文表格里的三个核心指标，这个 block 就是规则来源。


In [7]:
# 中文导读：这里定义了整份实验最核心的指标计算与阈值选择规则。
# =============================================================================
# Block 6. 评估指标、负采样与阈值选择
# 作用：
# 1) 统一事件级 sensitivity / FAR/hr / latency 计算；
# 2) 统一最短持续时间约束；
# 3) 统一训练集负样本抽样；
# 4) 用验证集选择阈值，避免测试集泄漏。
# =============================================================================
def apply_duration_constraint(binary_preds: np.ndarray, min_epochs: int = 3) -> np.ndarray:
    cleaned = np.asarray(binary_preds, dtype=np.int8).copy()
    if cleaned.size == 0 or min_epochs <= 1:
        return cleaned

    padded = np.pad(cleaned, (1, 1), mode="constant", constant_values=0)
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]

    short_runs = (ends - starts) < min_epochs
    for run_start, run_end in zip(starts[short_runs], ends[short_runs]):
        cleaned[run_start:run_end] = 0

    return cleaned

def count_events(labels: np.ndarray) -> int:
    labels = np.asarray(labels)
    if labels.size == 0:
        return 0
    return int((labels[0] == 1) + np.sum((labels[1:] == 1) & (labels[:-1] == 0)))


def extract_binary_runs(labels: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    labels = np.asarray(labels, dtype=np.int8)
    if labels.size == 0:
        empty = np.empty((0,), dtype=int)
        return empty, empty

    padded = np.pad(labels, (1, 1), mode="constant", constant_values=0)
    diff = np.diff(padded)
    starts = np.flatnonzero(diff == 1).astype(int)
    ends = np.flatnonzero(diff == -1).astype(int)
    return starts, ends


def compute_event_metrics(
    y_true: np.ndarray,
    y_pred_binary: np.ndarray,
    epoch_len_s: int,
) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred_binary = np.asarray(y_pred_binary, dtype=np.int8)

    true_starts, true_ends = extract_binary_runs(y_true)
    total_events = int(len(true_starts))
    detected_events = 0
    delays_s = []

    for event_start, event_end in zip(true_starts, true_ends):
        pred_positions = np.flatnonzero(y_pred_binary[event_start:event_end] == 1)
        if pred_positions.size > 0:
            detected_events += 1
            delays_s.append(int(pred_positions[0]) * epoch_len_s)

    false_alarm_mask = ((y_pred_binary == 1) & (y_true == 0)).astype(np.int8)
    false_alarm_events = int(len(extract_binary_runs(false_alarm_mask)[0]))

    total_hours = len(y_true) * epoch_len_s / 3600.0
    sensitivity = detected_events / total_events if total_events > 0 else np.nan
    far_per_hour = false_alarm_events / total_hours if total_hours > 0 else np.nan

    return {
        "events": float(total_events),
        "detected_events": float(detected_events),
        "sensitivity": float(sensitivity) if not np.isnan(sensitivity) else np.nan,
        "false_alarm_events": float(false_alarm_events),
        "far_per_hour": float(far_per_hour) if not np.isnan(far_per_hour) else np.nan,
        "mean_delay_s": float(np.mean(delays_s)) if delays_s else np.nan,
        "median_delay_s": float(np.median(delays_s)) if delays_s else np.nan,
        "hours": float(total_hours),
    }


def sample_training_rows(
    X: np.ndarray,
    y: np.ndarray,
    cfg: ExperimentConfig,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    """Optimized: single index array avoids multiple data copies (optimization #12)."""
    pos_idx = np.flatnonzero(y == 1)
    neg_idx = np.flatnonzero(y == 0)

    if len(pos_idx) == 0:
        return X.copy(), y.copy()

    target_neg = max(cfg.eval.min_neg_samples, len(pos_idx) * cfg.eval.neg_to_pos_ratio)
    target_neg = min(target_neg, len(neg_idx))

    if target_neg < len(neg_idx):
        neg_idx = rng.choice(neg_idx, size=target_neg, replace=False)

    combined = np.concatenate([pos_idx, neg_idx])
    rng.shuffle(combined)
    return X[combined], y[combined]


def choose_threshold_from_validation(
    y_true: np.ndarray,
    y_score: np.ndarray,
    cfg: ExperimentConfig,
) -> Dict[str, float]:
    best = None
    fallback = None

    threshold_grid = cfg.eval.threshold_grid
    min_duration = cfg.eval.min_duration_epochs
    epoch_len_s = cfg.feature.epoch_len_s
    min_sens = cfg.eval.min_acceptable_sensitivity

    for thr in threshold_grid:
        y_pred = (y_score >= thr).astype(np.int8)
        y_pred = apply_duration_constraint(y_pred, min_duration)
        metrics = compute_event_metrics(y_true, y_pred, epoch_len_s)

        row = {
            "threshold": float(thr),
            "sensitivity": metrics["sensitivity"],
            "far_per_hour": metrics["far_per_hour"],
            "median_delay_s": metrics["median_delay_s"],
        }

        # Optimization #11: helper to avoid repeated np.nan_to_num calls
        _s = row["sensitivity"] if not np.isnan(row["sensitivity"]) else -1.0
        _f = row["far_per_hour"] if not np.isnan(row["far_per_hour"]) else float("inf")
        _d = row["median_delay_s"] if not np.isnan(row["median_delay_s"]) else float("inf")
        row_rank = (-_s, _f, _d)

        if fallback is None:
            fallback = row
            fallback_rank = row_rank
        else:
            if row_rank < fallback_rank:
                fallback = row
                fallback_rank = row_rank

        if (
            not np.isnan(metrics["sensitivity"])
            and metrics["sensitivity"] >= min_sens
        ):
            best_row_rank = (_f, _d, -_s)
            if best is None:
                best = row
                best_rank = best_row_rank
            else:
                if best_row_rank < best_rank:
                    best = row
                    best_rank = best_row_rank

    return best if best is not None else fallback


def split_inner_validation_files(
    train_seizure_files: List[str],
    train_bg_files: List[str],
    seed: int,
) -> Tuple[List[str], List[str], List[str], List[str]]:
    rng = np.random.default_rng(seed)

    if len(train_seizure_files) >= 2:
        shuffled_seizures = train_seizure_files.copy()
        rng.shuffle(shuffled_seizures)
        val_seizure_files = [shuffled_seizures[0]]
        inner_train_seizure_files = shuffled_seizures[1:]
    else:
        val_seizure_files = []
        inner_train_seizure_files = train_seizure_files.copy()

    shuffled_bg = train_bg_files.copy()
    rng.shuffle(shuffled_bg)
    n_val_bg = max(1, len(shuffled_bg) // max(2, len(train_seizure_files) + 1)) if len(shuffled_bg) > 0 else 0
    val_bg_files = shuffled_bg[:n_val_bg]
    inner_train_bg_files = shuffled_bg[n_val_bg:]

    return inner_train_seizure_files, inner_train_bg_files, val_seizure_files, val_bg_files


print("✅ Block 6 完成：评估与阈值选择函数已定义。")

✅ Block 6 完成：评估与阈值选择函数已定义。


### Block 6 输出解读补充
- 后面的所有 `Sensitivity / FAR/hr / Delay` 都是按这里定义出来的。
- 所以这部分规则最好不要中途随意改，否则前后结果不可比。


## Block 7：构建特征缓存

### 这个代码块在做什么
- 把“原始 EDF -> 特征矩阵 + 标签”这一整条昂贵流程保存成缓存文件。
- 这样后面的搜索、benchmark、最终验证都能直接复用缓存，而不是重复提特征。

### 输出怎么解读
- 正常会看到每个病人的缓存构建状态。
- 如果某个病人或 EDF 被跳过，要重点看报错原因，通常是内存、EDF 异常或通道不匹配。


In [8]:
# 中文导读：这里把昂贵的特征提取结果存成缓存，后面的搜索和评测都依赖这些缓存。
# =============================================================================
# Block 7. 构建单病人 / 多病人特征缓存
# 作用：
# 1) 从 EDF + summary.txt 构建 epoch 级特征；
# 2) 把缓存与 config_hash 绑定；
# 3) 让后续评估完全依赖缓存，避免重复提特征。
# =============================================================================
def build_patient_cache(cfg: ExperimentConfig, patient_id: str, overwrite: bool = False) -> Path:
    patient_dir = Path(cfg.data_root) / patient_id
    if not patient_dir.exists():
        raise FileNotFoundError(f"找不到病人目录: {patient_dir}")

    cache_path = get_patient_cache_path(cfg, patient_id)
    if cache_path.exists() and not overwrite:
        print(f"[跳过] {patient_id}: 发现同配置缓存 -> {cache_path.name}")
        return cache_path

    summary_path = patient_dir / f"{patient_id}-summary.txt"
    seizure_dict = parse_summary_to_dict(summary_path)

    file_payload = {}
    processed_files = 0
    skipped_files = 0

    for edf_path in sorted(patient_dir.glob("*.edf")):
        raw = None
        try:
            raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
            raw = deduplicate_and_normalize_raw(raw)

            aligned_data, align_info = align_channels(
                raw=raw,
                target_channels=cfg.channels,
                policy=cfg.feature.channel_missing_policy,
            )

            if cfg.feature.scale_to_uV:
                aligned_data = aligned_data * 1e6

            fs = int(raw.info["sfreq"])
            data_bp = bandpass_filter_multich(
                aligned_data,
                fs=fs,
                lowcut=cfg.feature.bandpass_low_hz,
                highcut=cfg.feature.bandpass_high_hz,
                method=cfg.feature.bandpass_method,
                butter_order=cfg.feature.butter_order,
            )

            X_base, kept_epoch_indices = extract_base_features_for_file(data_bp, fs, cfg)
            if X_base.size == 0:
                skipped_files += 1
                continue

            seizure_intervals = seizure_dict.get(edf_path.name, [])
            y_base = build_epoch_labels(
                n_epochs=len(kept_epoch_indices),
                epoch_len_s=cfg.feature.epoch_len_s,
                seizure_intervals=seizure_intervals,
            )

            X_stacked, y_stacked = temporal_stack_features(
                X_base=X_base,
                y=y_base,
                history_epochs=cfg.feature.history_epochs,
            )

            file_payload[edf_path.name] = {
                "X": X_stacked.astype(np.float32),
                "y": y_stacked.astype(np.int8),
                "has_seizure": bool(len(seizure_intervals) > 0),
                "has_positive_epoch": bool(np.any(y_stacked == 1)),
                "fs": fs,
                "n_epochs": int(len(y_stacked)),
                "missing_channels": align_info["missing_channels"],
                "missing_count": align_info["missing_count"],
                "reversed_channels": align_info["reversed_channels"],
            }

            processed_files += 1

        except Exception as exc:
            skipped_files += 1
            print(f"[警告] {patient_id} / {edf_path.name} 跳过，原因: {exc}")
        finally:
            if raw is not None:
                raw.close()

    payload = {
        "meta": {
            "patient_id": patient_id,
            "config_hash": config_to_hash(cfg),
            "base_feature_dim": len(build_base_feature_names(cfg)),
            "stacked_feature_dim": len(build_stacked_feature_names(cfg)),
            "channel_missing_policy": cfg.feature.channel_missing_policy,
            "history_epochs": cfg.feature.history_epochs,
            "epoch_len_s": cfg.feature.epoch_len_s,
            "cache_schema_version": CACHE_SCHEMA_VERSION,
            "processed_files": processed_files,
            "skipped_files": skipped_files,
        },
        "files": file_payload,
    }

    ensure_dir(cache_path.parent)
    joblib.dump(payload, cache_path)
    print(f"[完成] {patient_id}: processed={processed_files}, skipped={skipped_files}, cache={cache_path.name}")
    return cache_path


def build_all_caches(cfg: ExperimentConfig, overwrite: bool = False, n_jobs: int = 1) -> List[Path]:
    if not Path(cfg.data_root).exists():
        raise FileNotFoundError(
            f"Current data_root does not exist: {cfg.data_root}\n"
            "Please set CFG.data_root to your local CHB-MIT directory."
        )

    n_jobs = int(max(1, n_jobs))
    patient_ids = list(cfg.eval.patient_ids)

    if n_jobs == 1:
        cache_paths = []
        for patient_id in patient_ids:
            cache_paths.append(build_patient_cache(cfg, patient_id, overwrite=overwrite))
            gc.collect()
        return cache_paths

    print(f"Building caches in parallel: n_jobs={n_jobs}, patients={len(patient_ids)}")
    cache_paths = joblib.Parallel(n_jobs=n_jobs, backend="loky")(
        joblib.delayed(build_patient_cache)(cfg, patient_id, overwrite=overwrite)
        for patient_id in patient_ids
    )
    return list(cache_paths)

### Block 7 输出解读补充
- 缓存构建成功后，后面多数实验会明显快很多。
- 如果你要换参数，建议保留旧缓存，不要直接覆盖，便于回溯。


## Block 8：加载缓存并检查一致性

### 这个代码块在做什么
- 读取已经构建好的缓存。
- 检查缓存里的配置哈希和当前 notebook 的配置是否一致。

### 输出怎么解读
- 如果这里报“配置不一致”，说明你改过参数，但还在读取旧缓存。
- 这种情况下应该重新构建缓存，而不是继续用旧结果。


In [9]:
# 中文导读：这里负责把缓存读回来，并确认缓存确实和当前配置一致。
# =============================================================================
# Block 8. 加载缓存并检查一致性
# 作用：
# 1) 读取单病人/多病人的缓存；
# 2) 检查 config_hash 是否一致；
# 3) 给后续评估提供统一的缓存对象格式。
# =============================================================================
def load_patient_cache(cfg: ExperimentConfig, patient_id: str) -> Dict[str, Any]:
    cache_path = get_patient_cache_path(cfg, patient_id)
    if not cache_path.exists():
        raise FileNotFoundError(f"找不到缓存文件: {cache_path}")

    payload = joblib.load(cache_path)
    meta = payload.get("meta", {})
    files = payload.get("files", {})

    expected_hash = config_to_hash(cfg)
    if meta.get("config_hash") != expected_hash:
        raise ValueError(
            f"{patient_id} 的缓存配置与当前 notebook 不一致。\n"
            f"缓存 hash = {meta.get('config_hash')}\n"
            f"当前 hash = {expected_hash}\n"
            "请重新构建缓存。"
        )

    if not isinstance(files, dict):
        raise ValueError(f"{patient_id} 的缓存结构异常：缺少 files 字段。")

    return payload


def load_all_caches(cfg: ExperimentConfig) -> Dict[str, Dict[str, Any]]:
    caches = {}
    for patient_id in cfg.eval.patient_ids:
        cache_path = get_patient_cache_path(cfg, patient_id)
        if cache_path.exists():
            caches[patient_id] = load_patient_cache(cfg, patient_id)
    return caches


def summarize_caches(caches: Dict[str, Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for patient_id, payload in caches.items():
        files = payload["files"]
        n_files = len(files)
        n_seizure_files = sum(int(d["has_seizure"]) for d in files.values())
        n_positive_epoch_files = sum(int(d.get("has_positive_epoch", d["has_seizure"])) for d in files.values())
        n_epochs = sum(int(d["n_epochs"]) for d in files.values())
        rows.append(
            {
                "Patient": patient_id,
                "Files": n_files,
                "Seizure_Files": n_seizure_files,
                "Positive_Epoch_Files": n_positive_epoch_files,
                "Epochs": n_epochs,
                "Feature_Dim": payload["meta"]["stacked_feature_dim"],
            }
        )
    if not rows:
        return pd.DataFrame(
            columns=[
                "Patient",
                "Files",
                "Seizure_Files",
                "Positive_Epoch_Files",
                "Epochs",
                "Feature_Dim",
            ]
        )
    return pd.DataFrame(rows).sort_values("Patient").reset_index(drop=True)


print("✅ Block 8 完成：缓存加载函数已定义。")



✅ Block 8 完成：缓存加载函数已定义。


### Block 8 输出解读补充
- “能加载缓存”不等于“缓存一定可用”，还要看配置哈希是否匹配。


## Block 9：模型工厂与 LOSO 评估核心逻辑

### 这个代码块在做什么
- 统一创建 `svm_rbf / random_forest / xgboost` 三类模型。
- 定义单病人 LOSO、阈值选择、fold 级 top-k 特征选择、汇总输出。
- 这里是整份 notebook 最核心的评测逻辑。

### 为什么重要
- 只要这里有 bug，后面所有表格即使能跑出来，也不可信。
- 这次修复里最关键的低内存改动也在这个 block。

### 输出怎么解读
- 这里通常只打印“核心函数已定义”。
- 真正的模型效果会在 Block 14 以后体现。


In [10]:
# 中文导读：这里是 LOSO 评测主引擎，负责训练、验证、选阈值、测试和结果汇总。
# =============================================================================
# Block 9. 模型工厂与 LOSO 评估核心函数
# 作用：
# 1) 统一创建 SVM / RF / XGB 三类模型；
# 2) 统一训练/预测接口；
# 3) 支持固定阈值快速模式与标准 LOSO 评估模式；
# 4) 用单病人矩阵切片与 fold TopK 缓存降低重复计算。
# =============================================================================
def make_model(model_name: str, cfg: ExperimentConfig, scale_pos_weight: float = 1.0):
    if model_name == "svm_rbf":
        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "clf",
                    SVC(
                        kernel="rbf",
                        C=cfg.eval.svm_c,
                        gamma=cfg.eval.svm_gamma,
                        probability=True,
                        class_weight="balanced",
                        random_state=cfg.eval.random_state,
                    ),
                ),
            ]
        )

    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=cfg.eval.rf_n_estimators,
            max_depth=cfg.eval.rf_max_depth,
            min_samples_leaf=cfg.eval.rf_min_samples_leaf,
            max_features=cfg.eval.rf_max_features,
            class_weight="balanced_subsample",
            n_jobs=cfg.eval.rf_n_jobs,
            random_state=cfg.eval.random_state,
        )

    if model_name == "xgboost":
        return XGBClassifier(
            n_estimators=cfg.eval.xgb_n_estimators,
            max_depth=cfg.eval.xgb_max_depth,
            learning_rate=cfg.eval.xgb_learning_rate,
            subsample=cfg.eval.xgb_subsample,
            colsample_bytree=cfg.eval.xgb_colsample_bytree,
            reg_lambda=cfg.eval.xgb_reg_lambda,
            min_child_weight=cfg.eval.xgb_min_child_weight,
            gamma=cfg.eval.xgb_gamma,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            tree_method=cfg.eval.xgb_tree_method,
            n_jobs=cfg.eval.xgb_n_jobs,
            random_state=cfg.eval.random_state,
        )

    raise ValueError(f"Unsupported model_name: {model_name}")


def compute_scale_pos_weight(y: np.ndarray) -> float:
    y = np.asarray(y)
    pos = int(np.sum(y == 1))
    neg = int(np.sum(y == 0))
    return float(neg / max(1, pos))


def build_patient_matrix_index(
    patient_payload: Dict[str, Any],
) -> Tuple[np.ndarray, np.ndarray, Dict[str, Tuple[int, int]]]:
    cache_key = "_matrix_index_cache"
    cached = patient_payload.get(cache_key)
    if cached is not None:
        return cached["X_all"], cached["y_all"], cached["file_row_spans"]

    file_payload = patient_payload["files"]
    ordered_files = sorted(file_payload.keys())

    X_blocks = []
    y_blocks = []
    file_row_spans = {}
    row_start = 0

    for file_name in ordered_files:
        item = file_payload[file_name]
        X_block = np.asarray(item["X"], dtype=np.float32)
        y_block = np.asarray(item["y"], dtype=np.int8)

        row_end = row_start + len(y_block)
        file_row_spans[file_name] = (row_start, row_end)

        X_blocks.append(X_block)
        y_blocks.append(y_block)
        row_start = row_end

    if len(X_blocks) == 0:
        X_all = np.empty((0, 0), dtype=np.float32)
        y_all = np.empty((0,), dtype=np.int8)
    else:
        X_all = np.vstack(X_blocks).astype(np.float32, copy=False)
        y_all = np.concatenate(y_blocks).astype(np.int8, copy=False)

    patient_payload[cache_key] = {
        "X_all": X_all,
        "y_all": y_all,
        "file_row_spans": file_row_spans,
    }
    return X_all, y_all, file_row_spans


def collect_rows_from_matrix_index(
    X_all: np.ndarray,
    y_all: np.ndarray,
    file_row_spans: Dict[str, Tuple[int, int]],
    file_names: Sequence[str],
    feature_indices: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    if len(file_names) == 0:
        feature_dim = X_all.shape[1] if feature_indices is None else int(len(feature_indices))
        return np.empty((0, feature_dim), dtype=np.float32), np.empty((0,), dtype=np.int8)

    blocks_X = []
    blocks_y = []
    for file_name in file_names:
        if file_name not in file_row_spans:
            raise KeyError(f"Unknown file in row index: {file_name}")

        start, end = file_row_spans[file_name]
        X_block = X_all[start:end]
        if feature_indices is not None:
            X_block = X_block[:, feature_indices]
        blocks_X.append(X_block)
        blocks_y.append(y_all[start:end])

    if len(blocks_X) == 1:
        return blocks_X[0], blocks_y[0]

    return np.concatenate(blocks_X, axis=0), np.concatenate(blocks_y, axis=0)


def collect_rows_from_files(
    patient_payload: Dict[str, Any],
    file_names: Sequence[str],
    feature_indices: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """Convenience wrapper; callers with existing matrix index should use
    collect_rows_from_matrix_index directly to avoid redundant lookups (optimization #20)."""
    X_all, y_all, file_row_spans = build_patient_matrix_index(patient_payload)
    return collect_rows_from_matrix_index(
        X_all=X_all,
        y_all=y_all,
        file_row_spans=file_row_spans,
        file_names=file_names,
        feature_indices=feature_indices,
    )


def sample_rows_from_matrix_index(
    X_all: np.ndarray,
    y_all: np.ndarray,
    file_row_spans: Dict[str, Tuple[int, int]],
    file_names: Sequence[str],
    cfg: ExperimentConfig,
    rng: np.random.Generator,
    feature_indices: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    if len(file_names) == 0:
        feature_dim = X_all.shape[1] if feature_indices is None else int(len(feature_indices))
        return np.empty((0, feature_dim), dtype=np.float32), np.empty((0,), dtype=np.int8)

    row_index_blocks = []
    for file_name in file_names:
        if file_name not in file_row_spans:
            raise KeyError(f"Unknown file in row index: {file_name}")
        start, end = file_row_spans[file_name]
        row_index_blocks.append(np.arange(start, end, dtype=np.int64))

    combined_idx = row_index_blocks[0] if len(row_index_blocks) == 1 else np.concatenate(row_index_blocks)
    y_subset = y_all[combined_idx]
    pos_local = np.flatnonzero(y_subset == 1)
    neg_local = np.flatnonzero(y_subset == 0)

    if len(pos_local) == 0:
        selected_idx = combined_idx
    else:
        target_neg = max(cfg.eval.min_neg_samples, len(pos_local) * cfg.eval.neg_to_pos_ratio)
        target_neg = min(target_neg, len(neg_local))
        if target_neg < len(neg_local):
            neg_local = rng.choice(neg_local, size=target_neg, replace=False)
        keep_local = np.concatenate([pos_local, neg_local])
        rng.shuffle(keep_local)
        selected_idx = combined_idx[keep_local]

    X_selected = X_all[selected_idx]
    if feature_indices is not None:
        X_selected = X_selected[:, feature_indices]
    y_selected = y_all[selected_idx]
    return X_selected, y_selected


def topk_feature_cache_hash(cfg: ExperimentConfig) -> str:
    payload = {
        "feature": make_json_safe(asdict(cfg.feature)),
        "selector_n_estimators": cfg.eval.selector_n_estimators,
        "selector_max_depth": cfg.eval.selector_max_depth,
        "rf_max_features": cfg.eval.rf_max_features,
        "rf_min_samples_leaf": cfg.eval.rf_min_samples_leaf,
        "xgb_tree_method": cfg.eval.xgb_tree_method,
        "xgb_n_jobs": cfg.eval.xgb_n_jobs,
        "random_state": cfg.eval.random_state,
    }
    payload_str = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return joblib.hash(payload_str)


def select_top_k_features_tree(
    X_train: np.ndarray,
    y_train: np.ndarray,
    top_k: int,
    cfg: ExperimentConfig,
    selector_model_name: str = "random_forest",
) -> np.ndarray:
    if X_train.shape[1] <= top_k:
        return np.arange(X_train.shape[1], dtype=int)

    if selector_model_name == "xgboost":
        scale_pos_weight = compute_scale_pos_weight(y_train)
        selector = XGBClassifier(
            n_estimators=cfg.eval.selector_n_estimators,
            max_depth=cfg.eval.selector_max_depth,
            learning_rate=0.08,
            subsample=0.9,
            colsample_bytree=0.9,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            tree_method=cfg.eval.xgb_tree_method,
            n_jobs=cfg.eval.xgb_n_jobs,
            random_state=cfg.eval.random_state,
        )
    else:
        selector = RandomForestClassifier(
            n_estimators=cfg.eval.selector_n_estimators,
            max_depth=cfg.eval.selector_max_depth,
            min_samples_leaf=cfg.eval.rf_min_samples_leaf,
            max_features=cfg.eval.rf_max_features,
            class_weight="balanced_subsample",
            n_jobs=cfg.eval.rf_n_jobs,
            random_state=cfg.eval.random_state,
        )

    selector.fit(X_train, y_train)
    importances = selector.feature_importances_
    top_idx = np.argsort(importances)[-top_k:]
    return np.sort(top_idx)


def evaluate_patient_loso(
    patient_payload: Dict[str, Any],
    cfg: ExperimentConfig,
    model_name: str = "xgboost",
    top_k: Optional[int] = None,
    patient_seed_offset: int = 0,
    fold_topk_cache: Optional[Dict[Tuple[Any, ...], np.ndarray]] = None,
) -> Dict[str, Any]:
    patient_id = patient_payload["meta"]["patient_id"]
    file_payload = patient_payload["files"]
    seizure_files = sorted([f for f, d in file_payload.items() if d["has_seizure"]])
    bg_files = sorted([f for f, d in file_payload.items() if not d["has_seizure"]])

    if len(seizure_files) < 2:
        raise ValueError("Seizure file count < 2; cannot run LOSO.")

    X_all, y_all, file_row_spans = build_patient_matrix_index(patient_payload)
    feature_cfg_hash = topk_feature_cache_hash(cfg)

    rng = np.random.default_rng(cfg.eval.random_state + patient_seed_offset)
    shuffled_bg = bg_files.copy()
    rng.shuffle(shuffled_bg)
    bg_chunks = np.array_split(shuffled_bg, len(seizure_files))

    fold_rows = []
    all_y_true = []
    all_y_pred = []
    all_y_score = []
    all_selected_features = []
    topk_cache_hits = 0
    topk_cache_misses = 0

    # 外层循环：每次拿 1 个发作文件做测试，其余发作文件留作训练/验证。
    for outer_idx, test_seizure_file in enumerate(seizure_files):
        outer_train_seizure_files = seizure_files[:outer_idx] + seizure_files[outer_idx + 1 :]
        outer_test_bg_files = list(bg_chunks[outer_idx])
        outer_test_bg_set = set(outer_test_bg_files)
        outer_train_bg_files = [f for f in bg_files if f not in outer_test_bg_set]

        inner_train_seizure_files, inner_train_bg_files, val_seizure_files, val_bg_files = split_inner_validation_files(
            outer_train_seizure_files,
            outer_train_bg_files,
            seed=cfg.eval.random_state + patient_seed_offset + outer_idx,
        )

        inner_train_files = inner_train_seizure_files + inner_train_bg_files
        if len(inner_train_files) == 0:
            raise ValueError("No inner-train files available in this fold.")

        feature_indices = None
        # 如果启用了 top-k 特征选择，就在当前 fold 的 inner-train 上先选特征。
        if top_k is not None:
            selector_seed = cfg.eval.random_state + 1000 + outer_idx
            cache_key = (
                patient_id,
                feature_cfg_hash,
                int(top_k),
                int(outer_idx),
                tuple(inner_train_files),
                int(selector_seed),
            )

            if fold_topk_cache is not None and cache_key in fold_topk_cache:
                feature_indices = fold_topk_cache[cache_key]
                topk_cache_hits += 1
            else:
                selector_rng = np.random.default_rng(selector_seed)
                X_selector, y_selector = sample_rows_from_matrix_index(
                    X_all=X_all,
                    y_all=y_all,
                    file_row_spans=file_row_spans,
                    file_names=inner_train_files,
                    cfg=cfg,
                    rng=selector_rng,
                    feature_indices=None,
                )
                if len(y_selector) == 0:
                    raise ValueError("No inner-train samples available in this fold.")
                selector_model_name = model_name if model_name in {"random_forest", "xgboost"} else "random_forest"
                feature_indices = select_top_k_features_tree(
                    X_selector,
                    y_selector,
                    top_k=top_k,
                    cfg=cfg,
                    selector_model_name=selector_model_name,
                )
                if fold_topk_cache is not None:
                    fold_topk_cache[cache_key] = feature_indices
                topk_cache_misses += 1

            all_selected_features.append(feature_indices)

        # 阈值策略：可以直接用固定阈值，也可以在当前 outer-train 内部再划验证集挑阈值。
        use_fixed_threshold = bool(getattr(cfg.eval, "fixed_threshold_mode", False))
        if use_fixed_threshold:
            threshold = float(getattr(cfg.eval, "fixed_threshold_value", cfg.eval.default_threshold))
        else:
            threshold = cfg.eval.default_threshold
            if len(val_seizure_files) > 0:
                train_rng = np.random.default_rng(cfg.eval.random_state + 2000 + outer_idx)
                X_train_inner, y_train_inner = sample_rows_from_matrix_index(
                    X_all=X_all,
                    y_all=y_all,
                    file_row_spans=file_row_spans,
                    file_names=inner_train_files,
                    cfg=cfg,
                    rng=train_rng,
                    feature_indices=feature_indices,
                )

                inner_model = make_model(
                    model_name,
                    cfg,
                    scale_pos_weight=compute_scale_pos_weight(y_train_inner),
                )
                inner_model.fit(X_train_inner, y_train_inner)

                X_val, y_val = collect_rows_from_matrix_index(
                    X_all=X_all,
                    y_all=y_all,
                    file_row_spans=file_row_spans,
                    file_names=val_seizure_files + val_bg_files,
                    feature_indices=feature_indices,
                )
                val_scores = inner_model.predict_proba(X_val)[:, 1]
                threshold_info = choose_threshold_from_validation(y_val, val_scores, cfg)
                threshold = threshold_info["threshold"]

        # 外层训练集也使用受控随机采样，避免负样本数量过大导致内存和训练时间失控。
        outer_rng = np.random.default_rng(cfg.eval.random_state + 3000 + outer_idx)
        X_train_outer, y_train_outer = sample_rows_from_matrix_index(
            X_all=X_all,
            y_all=y_all,
            file_row_spans=file_row_spans,
            file_names=outer_train_seizure_files + outer_train_bg_files,
            cfg=cfg,
            rng=outer_rng,
            feature_indices=feature_indices,
        )

        model = make_model(
            model_name,
            cfg,
            scale_pos_weight=compute_scale_pos_weight(y_train_outer),
        )
        model.fit(X_train_outer, y_train_outer)

        X_test, y_test = collect_rows_from_matrix_index(
            X_all=X_all,
            y_all=y_all,
            file_row_spans=file_row_spans,
            file_names=sorted([test_seizure_file] + outer_test_bg_files),
            feature_indices=feature_indices,
        )
        y_score = model.predict_proba(X_test)[:, 1]
        y_score_smoothed = medfilt(y_score, kernel_size=5)
        y_pred = (y_score_smoothed >= threshold).astype(np.int8)
        y_pred = apply_duration_constraint(y_pred, cfg.eval.min_duration_epochs)

        fold_metrics = compute_event_metrics(y_test, y_pred, cfg.feature.epoch_len_s)
        fold_rows.append(
            {
                "fold": outer_idx,
                "threshold": threshold,
                "test_seizure_file": test_seizure_file,
                "n_test_bg_files": len(outer_test_bg_files),
                "sensitivity": fold_metrics["sensitivity"],
                "far_per_hour": fold_metrics["far_per_hour"],
                "median_delay_s": fold_metrics["median_delay_s"],
            }
        )

        all_y_true.append(y_test)
        all_y_pred.append(y_pred)
        all_y_score.append(y_score_smoothed)

    y_true_cat = np.concatenate(all_y_true)
    y_pred_cat = np.concatenate(all_y_pred)
    y_score_cat = np.concatenate(all_y_score)

    patient_metrics = compute_event_metrics(y_true_cat, y_pred_cat, cfg.feature.epoch_len_s)
    summary = {
        "Patient": patient_id,
        "Model": model_name,
        "TopK": top_k if top_k is not None else -1,
        "Hours": patient_metrics["hours"],
        "True_Seizures": patient_metrics["events"],
        "Sensitivity": patient_metrics["sensitivity"],
        "FAR_per_Hour": patient_metrics["far_per_hour"],
        "Mean_Delay_s": patient_metrics["mean_delay_s"],
        "Median_Delay_s": patient_metrics["median_delay_s"],
        "Median_Threshold": float(np.median([row["threshold"] for row in fold_rows])),
    }

    return {
        "summary": summary,
        "folds": pd.DataFrame(fold_rows),
        "y_true": y_true_cat,
        "y_pred": y_pred_cat,
        "y_score": y_score_cat,
        "selected_features": all_selected_features,
        "topk_cache_hits": int(topk_cache_hits),
        "topk_cache_misses": int(topk_cache_misses),
    }


def evaluate_many_patients(
    caches: Dict[str, Dict[str, Any]],
    cfg: ExperimentConfig,
    model_name: str = "xgboost",
    top_k: Optional[int] = None,
    fold_topk_cache: Optional[Dict[Tuple[Any, ...], np.ndarray]] = None,
) -> Dict[str, Any]:
    patient_outputs = {}
    summary_rows = []
    total_cache_hits = 0
    total_cache_misses = 0

    for patient_idx, patient_id in enumerate(cfg.eval.patient_ids):
        if patient_id not in caches:
            continue

        try:
            result = evaluate_patient_loso(
                patient_payload=caches[patient_id],
                cfg=cfg,
                model_name=model_name,
                top_k=top_k,
                patient_seed_offset=patient_idx * 100,
                fold_topk_cache=fold_topk_cache,
            )
            patient_outputs[patient_id] = result
            summary_rows.append(result["summary"])
            total_cache_hits += int(result.get("topk_cache_hits", 0))
            total_cache_misses += int(result.get("topk_cache_misses", 0))
            print(
                f"[{patient_id}] {model_name} | "
                f"Sens={result['summary']['Sensitivity']:.2%} | "
                f"FAR/hr={result['summary']['FAR_per_Hour']:.4f} | "
                f"MedianThr={result['summary']['Median_Threshold']:.3f}"
            )
        except Exception as exc:
            print(f"[警告] {patient_id} 评估失败: {exc}")

    summary_df = pd.DataFrame(summary_rows)
    return {
        "patient_outputs": patient_outputs,
        "summary_df": summary_df,
        "topk_cache_hits": int(total_cache_hits),
        "topk_cache_misses": int(total_cache_misses),
    }


print("✅ Block 9 完成：统一评估入口已定义。")

✅ Block 9 完成：统一评估入口已定义。

### Block 9 输出解读补充
- 这个 block 已经加了更明确的中文导读和关键逻辑注释。
- 如果你想继续改 LOSO 评测，建议优先从这里入手，而不是直接改后面的展示 block。


## Block 10：构建或加载缓存，并检查病人覆盖情况

### 这个代码块在做什么
- 根据 `RUN_BUILD_CACHE` 决定是否重新构建缓存。
- 加载可用缓存，并统计 `chb01` 到 `chb10` 是否齐全。

### 输出怎么解读
- `cache_summary_df`：看每位病人文件数、发作文件数、epoch 数和特征维度。
- 如果 `missing_patients` 非空，后面的 benchmark 结果就不完整。
- 如果某位病人的 `Seizure_Files < 2`，LOSO 结果会不稳定甚至无法做。


In [11]:
# 中文导读：这里会根据运行开关构建或加载缓存，并检查目标病人是否齐全。
# =============================================================================
# Block 10. 构建缓存并检查覆盖情况
# =============================================================================
cache_build_jobs = min(6, max(1, (os.cpu_count() or 1) // 2))
print(f"Cache build jobs: {cache_build_jobs}")

if RUN_BUILD_CACHE:
    cache_paths = build_all_caches(CFG, overwrite=False, n_jobs=cache_build_jobs)
    print(f"Built / verified cache files: {len(cache_paths)}")
else:
    print("⏭ Block 10 cache build skipped (RUN_BUILD_CACHE = False). Loading existing caches only.")

caches = load_all_caches(CFG)
PIPELINE_RESULTS["caches"] = caches

cache_summary_df = summarize_caches(caches)
PIPELINE_RESULTS["cache_summary_df"] = cache_summary_df
display(cache_summary_df)

target_patients = tuple(f"chb{i:02d}" for i in range(1, 11))
missing_patients = [pid for pid in target_patients if pid not in caches]
available_patients = [pid for pid in target_patients if pid in caches]

print(f"Target patients: {target_patients}")
print(f"Available in cache: {available_patients}")
if missing_patients:
    print(f"[警告] 缓存缺失病人: {missing_patients}")
else:
    print("✅ chb01-chb10 缓存已全部就绪。")

few_seizure_df = cache_summary_df[cache_summary_df["Seizure_Files"] < 2]
if not few_seizure_df.empty:
    print("[提示] 以下病人发作文件较少（<2），LOSO 结果可能波动：")
    display(few_seizure_df[["Patient", "Seizure_Files", "Files"]])

print("✅ Block 10 完成：缓存构建与覆盖检查已完成。")
print("V5 perf workflow enabled.")
print("Includes: cached feature pipeline, baseline benchmark, RF validation, and deployment-oriented outputs.")


Cache build jobs: 6
⏭ Block 10 cache build skipped (RUN_BUILD_CACHE = False). Loading existing caches only.


,Patient,Files,Seizure_Files,Positive_Epoch_Files,Epochs,Feature_Dim
0,chb01,42,7,7,72993,1772
1,chb02,36,3,3,63479,1772
2,chb03,38,7,7,68403,1772
3,chb04,42,3,3,280914,1772
4,chb05,39,5,5,70205,1772
5,chb06,18,7,7,120122,1772
6,chb07,19,3,3,120693,1772
7,chb08,20,5,5,36011,1772
8,chb09,19,3,3,122165,1772
9,chb10,25,7,7,90041,1772


Target patients: ('chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb08', 'chb09', 'chb10')
Available in cache: ['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb08', 'chb09', 'chb10']
✅ chb01-chb10 缓存已全部就绪。
✅ Block 10 完成：缓存构建与覆盖检查已完成。
V5 perf workflow enabled.
Includes: cached feature pipeline, baseline benchmark, RF validation, and deployment-oriented outputs.


### Block 10 输出解读补充
- 这里最重要的是确认 `chb01-chb10` 是否齐全。
- 如果病人不全，后面的宏观结果会失真。


## Block 12A：特征搜索（历史窗 / 滤波器 / 阶数）

### 这个代码块在做什么
- 在一个小网格里比较不同 `history_epochs`、滤波方式和巴特沃斯阶数。
- 用固定的 RF + top-k 策略找出更合适的特征配置。

### 输出怎么解读
- 结果表里，`mean_sensitivity` 越高越好。
- `median_far_per_hour` 和 `mean_delay_s` 越低越好。
- 第一名会进入 `best_feature_cfg`，作为后面 Phase B 和正式 benchmark 的基础配置。


In [12]:
# 中文导读：这里做的是特征层面的搜索，而不是模型超参数搜索。
# =============================================================================
# Block 12A. Feature Search (history/bandpass/order)
# =============================================================================

def ensure_caches_ready(
    cfg: ExperimentConfig,
    patient_ids: Optional[Sequence[str]] = None,
    build_if_missing: bool = False,
    label: str = "cache set",
    n_jobs: Optional[int] = None,
) -> Tuple[Dict[str, Dict[str, Any]], List[str]]:
    cfg_local = cfg.variant()
    if patient_ids is not None:
        cfg_local.eval.patient_ids = tuple(patient_ids)

    caches_local = load_all_caches(cfg_local)
    missing_patients = [pid for pid in cfg_local.eval.patient_ids if pid not in caches_local]

    if missing_patients and build_if_missing:
        build_cfg = cfg_local.variant()
        build_cfg.eval.patient_ids = tuple(missing_patients)
        jobs = n_jobs
        if jobs is None:
            jobs = min(4, max(1, min(len(missing_patients), (os.cpu_count() or 1) // 2 or 1)))
        print(f"[Cache] Missing {label}: {missing_patients}. Building now (n_jobs={jobs})...")
        _ = build_all_caches(build_cfg, overwrite=False, n_jobs=jobs)
        caches_local = load_all_caches(cfg_local)
        missing_patients = [pid for pid in cfg_local.eval.patient_ids if pid not in caches_local]

    if missing_patients:
        print(f"[Cache] Still missing for {label}: {missing_patients}")
    else:
        print(f"[Cache] Ready for {label}: {len(caches_local)} patients")

    return caches_local, missing_patients


if not RUN_PHASE_A_SEARCH:
    print("Block 12A: Feature Search skipped (RUN_PHASE_A_SEARCH = False)")
    best_feature_cfg = build_precomputed_best_feature_cfg()
    best_feature_caches, best_feature_missing = ensure_caches_ready(
        best_feature_cfg,
        patient_ids=CFG.eval.patient_ids,
        build_if_missing=True,
        label="Phase A winner",
    )
    if best_feature_missing:
        raise FileNotFoundError(f"Phase A winner caches are still missing: {best_feature_missing}")
    best_feature = PRECOMPUTED_PHASE_A_WINNER.copy()
    PIPELINE_RESULTS["best_feature_cfg"] = best_feature_cfg
    PIPELINE_RESULTS["best_feature_caches"] = best_feature_caches
    PIPELINE_RESULTS["phase_a_winner"] = best_feature
    print("Using precomputed Phase A winner:", best_feature)
else:
    def _safe_float(x, default=0.0):
        try:
            return float(x) if x is not None and not np.isnan(x) else float(default)
        except (TypeError, ValueError):
            return float(default)


    def _aggregate_summary(summary_df: pd.DataFrame) -> Dict[str, float]:
        if summary_df is None or summary_df.empty:
            return {
                "mean_sensitivity": np.nan,
                "median_far_per_hour": np.nan,
                "mean_delay_s": np.nan,
                "patients": 0,
            }
        return {
            "mean_sensitivity": float(summary_df["Sensitivity"].mean()),
            "median_far_per_hour": float(summary_df["FAR_per_Hour"].median()),
            "mean_delay_s": float(summary_df["Mean_Delay_s"].mean()),
            "patients": int(len(summary_df)),
        }


    def _priority_sort(df: pd.DataFrame) -> pd.DataFrame:
        if df is None or df.empty:
            return pd.DataFrame() if df is None else df

        work = df.copy()
        work["_sens"] = work["mean_sensitivity"].apply(lambda v: _safe_float(v, -1.0))
        work["_far"] = work["median_far_per_hour"].apply(lambda v: _safe_float(v, 1e9))
        work["_delay"] = work["mean_delay_s"].apply(lambda v: _safe_float(v, 1e9))
        work = work.sort_values(["_sens", "_far", "_delay"], ascending=[False, True, True]).reset_index(drop=True)
        work["priority_rank"] = np.arange(1, len(work) + 1)
        return work.drop(columns=["_sens", "_far", "_delay"])


    def _run_rf_eval(
        cfg_variant: ExperimentConfig,
        caches_variant: Dict[str, Dict[str, Any]],
        top_k: int,
        fold_topk_cache: Optional[Dict[Tuple[Any, ...], np.ndarray]] = None,
    ):
        t0 = time.perf_counter()
        result = evaluate_many_patients(
            caches=caches_variant,
            cfg=cfg_variant,
            model_name="random_forest",
            top_k=top_k,
            fold_topk_cache=fold_topk_cache,
        )
        summary_df = result.get("summary_df", pd.DataFrame())
        agg = _aggregate_summary(summary_df)
        agg["elapsed_s"] = float(time.perf_counter() - t0)
        agg["fold_topk_cache_entries"] = int(len(fold_topk_cache)) if fold_topk_cache is not None else 0
        agg["topk_cache_hits"] = int(result.get("topk_cache_hits", 0))
        agg["topk_cache_misses"] = int(result.get("topk_cache_misses", 0))
        return result, agg


    phase_a_top_k = 50
    cache_build_jobs_tune = min(4, max(1, (os.cpu_count() or 1) // 2))
    feature_search_space = [
        {"history_epochs": 3, "bandpass_method": "butter_sos", "butter_order": 4},
        {"history_epochs": 4, "bandpass_method": "butter_sos", "butter_order": 4},
        {"history_epochs": 4, "bandpass_method": "butter_sos", "butter_order": 6},
        {"history_epochs": 4, "bandpass_method": "fir_zero", "butter_order": 4},
    ]

    print("Phase A candidates:", len(feature_search_space))
    print("Phase A top_k fixed:", phase_a_top_k)

    feature_rows = []
    feature_cache_map = {}
    phase_a_fold_topk_cache = {}
    phase_a_base_cfg = CFG.variant()
    phase_a_base_cfg.eval.fixed_threshold_mode = False

    # 逐个候选配置评测：这里只改特征层参数，RF 主体保持一致。
    for cand in feature_search_space:
        cfg_a = phase_a_base_cfg.variant()
        cfg_a.feature.history_epochs = int(cand["history_epochs"])
        cfg_a.feature.bandpass_method = str(cand["bandpass_method"])
        cfg_a.feature.butter_order = int(cand["butter_order"])

        print("\n[Phase A]", cand)
        _ = build_all_caches(cfg_a, overwrite=False, n_jobs=cache_build_jobs_tune)
        caches_a = load_all_caches(cfg_a)

        _, agg = _run_rf_eval(
            cfg_variant=cfg_a,
            caches_variant=caches_a,
            top_k=phase_a_top_k,
            fold_topk_cache=phase_a_fold_topk_cache,
        )

        row = {
            "history_epochs": cfg_a.feature.history_epochs,
            "bandpass_method": cfg_a.feature.bandpass_method,
            "butter_order": cfg_a.feature.butter_order,
            "top_k_fixed": phase_a_top_k,
            **agg,
        }
        feature_rows.append(row)
        key = (cfg_a.feature.history_epochs, cfg_a.feature.bandpass_method, cfg_a.feature.butter_order)
        feature_cache_map[key] = caches_a

    feature_df = pd.DataFrame(feature_rows)
    feature_df = _priority_sort(feature_df)
    PIPELINE_RESULTS["feature_search_df"] = feature_df
    display(feature_df)

    if feature_df.empty:
        raise ValueError("Phase A failed: no valid feature candidate.")

    best_feature = feature_df.iloc[0].to_dict()
    best_feature_cfg = phase_a_base_cfg.variant()
    best_feature_cfg.feature.history_epochs = int(best_feature["history_epochs"])
    best_feature_cfg.feature.bandpass_method = str(best_feature["bandpass_method"])
    best_feature_cfg.feature.butter_order = int(best_feature["butter_order"])

    best_key = (
        best_feature_cfg.feature.history_epochs,
        best_feature_cfg.feature.bandpass_method,
        best_feature_cfg.feature.butter_order,
    )
    best_feature_caches = feature_cache_map.get(best_key)
    if best_feature_caches is None:
        _ = build_all_caches(best_feature_cfg, overwrite=False, n_jobs=cache_build_jobs_tune)
        best_feature_caches = load_all_caches(best_feature_cfg)

    PIPELINE_RESULTS["best_feature_cfg"] = best_feature_cfg
    PIPELINE_RESULTS["best_feature_caches"] = best_feature_caches
    PIPELINE_RESULTS["phase_a_winner"] = best_feature
    gc.collect()


Block 12A: Feature Search skipped (RUN_PHASE_A_SEARCH = False)


[Cache] Ready for Phase A winner: 10 patients
Using precomputed Phase A winner: {'history_epochs': 3, 'bandpass_method': 'butter_sos', 'butter_order': 4, 'top_k_fixed': 50, 'mean_sensitivity': 0.9550000000000001, 'median_far_per_hour': 0.26165729497180734, 'mean_delay_s': 7.630952380952381, 'patients': 10}


### Block 12A 输出解读补充
- 第一名不是绝对真理，而是当前小网格下的最优配置。
- 如果你后续扩大搜索空间，winner 可能会变。


## Block 12B：RF 参数搜索（top-k + RF 超参数）

### 这个代码块在做什么
- 固定 Phase A 选出来的特征配置。
- 继续搜索 `top_k_features` 和 RF 超参数组合，得到最终部署版 RF 配置。

### 输出怎么解读
- 第一行就是本阶段最优 RF 参数。
- 这个 winner 会写入 `TUNED_RF_CFG`，后面 Block 14、15、16 都会优先复用它。


In [13]:
# 中文导读：这里在固定特征配置后，继续搜索 RF 的 top-k 与超参数组合。
# =============================================================================
# Block 12B. RF Model Search (top_k + rf params)
# =============================================================================
if not RUN_PHASE_B_SEARCH:
    print("Block 12B: RF Param Search skipped (RUN_PHASE_B_SEARCH = False)")
    if "best_feature_cfg" not in globals():
        best_feature_cfg = build_precomputed_best_feature_cfg()
        best_feature_caches, best_feature_missing = ensure_caches_ready(
            best_feature_cfg,
            patient_ids=CFG.eval.patient_ids,
            build_if_missing=True,
            label="Phase A winner",
        )
        if best_feature_missing:
            raise FileNotFoundError(f"Phase A winner caches are still missing: {best_feature_missing}")
        PIPELINE_RESULTS["best_feature_cfg"] = best_feature_cfg
        PIPELINE_RESULTS["best_feature_caches"] = best_feature_caches

    TUNED_RF_CFG = build_precomputed_tuned_rf_cfg()
    TUNED_CACHES, tuned_missing = ensure_caches_ready(
        TUNED_RF_CFG,
        patient_ids=CFG.eval.patient_ids,
        build_if_missing=True,
        label="Phase B tuned RF",
    )
    if tuned_missing:
        raise FileNotFoundError(f"Phase B tuned RF caches are still missing: {tuned_missing}")
    PIPELINE_RESULTS["TUNED_RF_CFG"] = TUNED_RF_CFG
    PIPELINE_RESULTS["TUNED_CACHES"] = TUNED_CACHES
    PIPELINE_RESULTS["phase_b_winner"] = PRECOMPUTED_PHASE_B_WINNER.copy()
    print("Using precomputed Phase B winner:", PRECOMPUTED_PHASE_B_WINNER)
else:
    print("Phase B search started...")
    topk_grid = [30, 50]
    rf_param_grid = [
        {"rf_n_estimators": 300, "rf_max_depth": None, "rf_min_samples_leaf": 1, "rf_max_features": "sqrt"},
        {"rf_n_estimators": 500, "rf_max_depth": 12, "rf_min_samples_leaf": 1, "rf_max_features": "sqrt"},
        {"rf_n_estimators": 500, "rf_max_depth": None, "rf_min_samples_leaf": 2, "rf_max_features": "sqrt"},
    ]

    model_rows = []
    phase_b_fold_topk_cache = {}
    # 逐个扫描 top-k 与 RF 超参数组合，记录每组的宏观表现。
    for top_k in topk_grid:
        for params in rf_param_grid:
            cfg_b = best_feature_cfg.variant()
            cfg_b.eval.fixed_threshold_mode = False
            cfg_b.eval.rf_n_estimators = int(params["rf_n_estimators"])
            cfg_b.eval.rf_max_depth = None if params["rf_max_depth"] is None else int(params["rf_max_depth"])
            cfg_b.eval.rf_min_samples_leaf = int(params["rf_min_samples_leaf"])
            cfg_b.eval.rf_max_features = str(params["rf_max_features"])
            _, agg = _run_rf_eval(cfg_variant=cfg_b, caches_variant=best_feature_caches, top_k=int(top_k), fold_topk_cache=phase_b_fold_topk_cache)
            model_rows.append({"top_k": int(top_k), **params, **agg})

    model_df = pd.DataFrame(model_rows)
    model_df = _priority_sort(model_df)
    PIPELINE_RESULTS["phase_b_search_df"] = model_df
    display(model_df)
    if model_df.empty:
        raise ValueError("Phase B failed: no valid model candidate.")

    best_model = model_df.iloc[0].to_dict()
    TUNED_RF_CFG = best_feature_cfg.variant()
    TUNED_RF_CFG.eval.top_k_features = int(best_model["top_k"])
    TUNED_RF_CFG.eval.rf_n_estimators = int(best_model["rf_n_estimators"])
    TUNED_RF_CFG.eval.rf_max_depth = None if pd.isna(best_model["rf_max_depth"]) else int(best_model["rf_max_depth"])
    TUNED_RF_CFG.eval.rf_min_samples_leaf = int(best_model["rf_min_samples_leaf"])
    TUNED_RF_CFG.eval.rf_max_features = str(best_model["rf_max_features"])
    TUNED_RF_CFG.eval.fixed_threshold_mode = CFG.eval.fixed_threshold_mode
    TUNED_RF_CFG.eval.fixed_threshold_value = CFG.eval.fixed_threshold_value
    TUNED_CACHES = best_feature_caches
    PIPELINE_RESULTS["TUNED_RF_CFG"] = TUNED_RF_CFG
    PIPELINE_RESULTS["TUNED_CACHES"] = TUNED_CACHES
    PIPELINE_RESULTS["phase_b_winner"] = best_model


Block 12B: RF Param Search skipped (RUN_PHASE_B_SEARCH = False)


[Cache] Ready for Phase B tuned RF: 10 patients
Using precomputed Phase B winner: {'top_k': 30, 'rf_n_estimators': 500, 'rf_max_depth': 12, 'rf_min_samples_leaf': 1, 'rf_max_features': 'sqrt', 'mean_sensitivity': 0.9800000000000001, 'median_far_per_hour': 0.24547462194392256, 'mean_delay_s': 10.64452380952381, 'patients': 10}


### Block 12B 输出解读补充
- 这个 winner 更偏“工程上好用”的组合，不一定是全宇宙唯一最优。
- 但它是后面最终 RF 验证和部署导出的直接来源。


## Block 13：最终评估辅助函数

### 这个代码块在做什么
- 定义汇总宏观指标、事件矩阵、最终 RF 上下文解析等辅助函数。
- 它把前面分散的结果组织成后面最终验证能直接复用的格式。

### 输出怎么解读
- 这里本身只是“把最后评估要用的工具准备好”。
- 如果这个 block 正常完成，说明后续 RF 最终验证链路已经能接上。


In [14]:
# 中文导读：这里准备的是最终评估辅助函数，方便后续正式验证与部署复用。
# =============================================================================
# Block 13. Final evaluation helper functions
# =============================================================================
def _build_macro_row(summary_df: pd.DataFrame, model_name: str, top_k: Optional[int]) -> Dict[str, Any]:
    if summary_df is None or summary_df.empty:
        return {
            "Model": model_name,
            "TopK": top_k if top_k is not None else -1,
            "Patients": 0,
            "Sensitivity": np.nan,
            "FAR_per_Hour": np.nan,
            "Mean_Delay_s": np.nan,
            "Median_Delay_s": np.nan,
            "Median_Threshold": np.nan,
        }
    return {
        "Model": model_name,
        "TopK": top_k if top_k is not None else -1,
        "Patients": int(len(summary_df)),
        "Sensitivity": float(summary_df["Sensitivity"].mean()),
        "FAR_per_Hour": float(summary_df["FAR_per_Hour"].median()),
        "Mean_Delay_s": float(summary_df["Mean_Delay_s"].mean()),
        "Median_Delay_s": float(summary_df["Median_Delay_s"].median()),
        "Median_Threshold": float(summary_df["Median_Threshold"].median()),
    }


def _sort_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df
    return df.copy().sort_values(["Sensitivity", "FAR_per_Hour", "Mean_Delay_s"], ascending=[False, True, True]).reset_index(drop=True)


def build_event_detection_matrix(result_bundle: Dict[str, Any], cfg: ExperimentConfig, label: str) -> pd.DataFrame:
    patient_outputs = result_bundle.get("patient_outputs", {})
    tp_events = 0.0
    fn_events = 0.0
    fp_events = 0.0
    total_hours = 0.0
    delay_weighted_sum = 0.0
    delay_weight = 0.0

    for _, patient_res in patient_outputs.items():
        metrics = compute_event_metrics(patient_res["y_true"], patient_res["y_pred"], cfg.feature.epoch_len_s)
        events = float(metrics["events"])
        detected = float(metrics["detected_events"])
        false_alarms = float(metrics["false_alarm_events"])
        hours = float(metrics["hours"])
        tp_events += detected
        fn_events += max(0.0, events - detected)
        fp_events += false_alarms
        total_hours += hours
        if (not np.isnan(metrics["mean_delay_s"])) and detected > 0:
            delay_weighted_sum += float(metrics["mean_delay_s"]) * detected
            delay_weight += detected

    sensitivity = tp_events / max(1.0, tp_events + fn_events)
    far_per_hour = fp_events / max(1e-9, total_hours)
    mean_delay_s = delay_weighted_sum / delay_weight if delay_weight > 0 else np.nan
    return pd.DataFrame([{
        "Variant": label,
        "TP_events": float(tp_events),
        "FN_events": float(fn_events),
        "FP_events": float(fp_events),
        "Sensitivity": float(sensitivity),
        "FAR_per_Hour": float(far_per_hour),
        "Mean_Delay_s": float(mean_delay_s) if not np.isnan(mean_delay_s) else np.nan,
        "Hours": float(total_hours),
        "Patients": int(len(patient_outputs)),
    }])


def resolve_final_rf_context() -> Tuple[ExperimentConfig, Dict[str, Dict[str, Any]], int, str]:
    if "TUNED_RF_CFG" in PIPELINE_RESULTS:
        cfg = PIPELINE_RESULTS["TUNED_RF_CFG"].variant()
        cfg_source = "TUNED_RF_CFG"
    else:
        cfg = CFG.variant()
        cfg_source = "CFG"
    cfg.eval.patient_ids = tuple(f"chb{i:02d}" for i in range(1, 11))
    cfg.eval.fixed_threshold_mode = False
    expected_hash = config_to_hash(cfg)

    for cache_key in ["deploy_caches", "benchmark_caches", "best_feature_caches", "TUNED_CACHES", "caches"]:
        candidate = PIPELINE_RESULTS.get(cache_key)
        if not isinstance(candidate, dict):
            continue
        if not all(pid in candidate for pid in cfg.eval.patient_ids):
            continue
        sample_pid = next(iter(cfg.eval.patient_ids))
        sample_hash = candidate.get(sample_pid, {}).get("meta", {}).get("config_hash")
        if sample_hash == expected_hash:
            print(f"Reusing in-memory caches from {cache_key} for final RF context.")
            return cfg, candidate, int(cfg.eval.top_k_features), cache_key

    caches_local, missing_patients = ensure_caches_ready(
        cfg,
        patient_ids=cfg.eval.patient_ids,
        build_if_missing=True,
        label=f"{cfg_source} final RF context",
    )
    if missing_patients:
        raise FileNotFoundError(f"Missing caches for final RF context: {missing_patients}")
    return cfg, caches_local, int(cfg.eval.top_k_features), cfg_source


print("Block 13 done: final evaluation helpers are ready.")


Block 13 done: final evaluation helpers are ready.


### Block 13 输出解读补充
- 这里只准备“最后一公里”的辅助函数，真正的最终结果从下一组 block 开始看。


## Block 14：三模型 baseline benchmark

### 这个代码块在做什么
- 在同一套特征配置下，对 `svm_rbf / random_forest / xgboost` 做统一 LOSO 评测。
- 同时输出每位病人的结果和宏观汇总结果。

### 输出怎么解读
- `multi_model_per_patient_df`：看不同病人在不同模型下的表现差异。
- `multi_model_macro_df`：看整体平均/中位水平，通常论文表格主要看这里。
- 这是传统 ML 三个 baseline 的主比较结果。


In [15]:
# Block 14. Multi-model benchmark
if not RUN_MULTI_MODEL_BENCHMARK:
    print("Block 14 skipped (RUN_MULTI_MODEL_BENCHMARK = False)")
    benchmark_cfg = None
    benchmark_caches = {}
    multi_model_runs = {}
    multi_model_per_patient_df = pd.DataFrame(
        columns=[
            "Patient",
            "Model",
            "TopK",
            "Hours",
            "True_Seizures",
            "Sensitivity",
            "FAR_per_Hour",
            "Mean_Delay_s",
            "Median_Delay_s",
            "Median_Threshold",
        ]
    )
    multi_model_macro_df = pd.DataFrame(
        columns=[
            "Model",
            "TopK",
            "Patients",
            "Mean_Sensitivity",
            "Median_FAR_per_Hour",
            "Mean_Delay_s",
            "Median_Delay_s",
            "Median_Threshold",
        ]
    )
else:
    if "TUNED_RF_CFG" in PIPELINE_RESULTS:
        benchmark_cfg = PIPELINE_RESULTS["TUNED_RF_CFG"].variant()
        print("Using tuned feature configuration for multi-model benchmark.")
    else:
        benchmark_cfg = CFG.variant()
        print("Using default configuration for multi-model benchmark.")

    benchmark_cfg.eval.patient_ids = tuple(f"chb{i:02d}" for i in range(1, 11))
    benchmark_cfg.eval.fixed_threshold_mode = False
    benchmark_caches, benchmark_missing = ensure_caches_ready(
        benchmark_cfg,
        patient_ids=benchmark_cfg.eval.patient_ids,
        build_if_missing=True,
        label="multi-model benchmark",
    )
    if benchmark_missing:
        raise FileNotFoundError(f"Benchmark caches are still missing: {benchmark_missing}")

    benchmark_models = []
    if RUN_SVM_BENCHMARK:
        benchmark_models.append(("svm_rbf", None))
    else:
        print("SVM benchmark skipped inside Block 14 (RUN_SVM_BENCHMARK = False)")
    benchmark_models.extend([("random_forest", None), ("xgboost", None)])

    multi_model_runs = {}
    multi_model_patient_frames = []
    multi_model_macro_rows = []
    for model_name, top_k in benchmark_models:
        print(f"\n===== Multi-Model Benchmark: {model_name}, top_k={top_k} =====")
        res = evaluate_many_patients(caches=benchmark_caches, cfg=benchmark_cfg, model_name=model_name, top_k=top_k)
        multi_model_runs[model_name] = res
        summary_df = res.get("summary_df", pd.DataFrame()).copy()
        if not summary_df.empty:
            summary_df["Model"] = model_name
            summary_df["TopK"] = -1 if top_k is None else int(top_k)
            multi_model_patient_frames.append(summary_df)
        multi_model_macro_rows.append(_build_macro_row(summary_df, model_name, top_k))

    multi_model_per_patient_df = (
        pd.concat(multi_model_patient_frames, ignore_index=True)
        if multi_model_patient_frames
        else pd.DataFrame(
            columns=[
                "Patient",
                "Model",
                "TopK",
                "Hours",
                "True_Seizures",
                "Sensitivity",
                "FAR_per_Hour",
                "Mean_Delay_s",
                "Median_Delay_s",
                "Median_Threshold",
            ]
        )
    )
    multi_model_macro_df = _sort_benchmark(pd.DataFrame(multi_model_macro_rows))

PIPELINE_RESULTS["benchmark_cfg"] = benchmark_cfg
PIPELINE_RESULTS["benchmark_caches"] = benchmark_caches
PIPELINE_RESULTS["multi_model_runs"] = multi_model_runs
PIPELINE_RESULTS["multi_model_per_patient_df"] = multi_model_per_patient_df
PIPELINE_RESULTS["multi_model_macro_df"] = multi_model_macro_df

display(multi_model_per_patient_df)
display(multi_model_macro_df)
gc.collect()
print("Block 14 complete.")


Block 14 skipped (RUN_MULTI_MODEL_BENCHMARK = False)


,Patient,Model,TopK,Hours,True_Seizures,Sensitivity,FAR_per_Hour,Mean_Delay_s,Median_Delay_s,Median_Threshold


,Model,TopK,Patients,Mean_Sensitivity,Median_FAR_per_Hour,Mean_Delay_s,Median_Delay_s,Median_Threshold


Block 14 complete.


### Block 14 输出解读补充
- 如果三模型都能出现在 `multi_model_macro_df` 里，说明 baseline 主链路是完整的。
- 如果某个模型缺失，优先检查对应模型的训练/缓存/内存问题。


## Block 14B：三模型可视化对比

### 这个代码块在做什么
- 把 Block 14 的结果做成柱状图和热图。
- 便于快速看三模型的宏观差异和病人间差异。

### 输出怎么解读
- 左上：`Sensitivity` 越高越好。
- 右上：`FAR/hr` 越低越好。
- 左下：平均延迟越低越好。
- 右下：每位病人的灵敏度热图，用来看是否有“某个病人特别难”的情况。


In [16]:
# Block 14B. Baseline comparison plots
if "multi_model_macro_df" not in PIPELINE_RESULTS or "multi_model_per_patient_df" not in PIPELINE_RESULTS:
    raise ValueError("Please run Block 14 first.")

MODEL_LABELS = {"svm_rbf": "SVM-RBF", "random_forest": "Random Forest", "xgboost": "XGBoost"}
MODEL_COLORS = {"SVM-RBF": "#4C78A8", "Random Forest": "#59A14F", "XGBoost": "#E15759"}

macro_df = PIPELINE_RESULTS["multi_model_macro_df"].copy()
per_patient_df = PIPELINE_RESULTS["multi_model_per_patient_df"].copy()
if macro_df.empty or "Sensitivity" not in macro_df.columns or "FAR_per_Hour" not in macro_df.columns:
    print("Block 14B skipped because the multi-model benchmark is disabled in this v2 workflow.")
    final_compare_table = pd.DataFrame(
        columns=["Model", "Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "Median_Threshold"]
    )
    PIPELINE_RESULTS["baseline_compare_table"] = final_compare_table
else:
    available_models = [m for m in ["svm_rbf", "random_forest", "xgboost"] if m in macro_df["Model"].tolist()]
    macro_df = macro_df[macro_df["Model"].isin(available_models)].copy()
    macro_df["ModelLabel"] = macro_df["Model"].map(MODEL_LABELS)
    if "Model" in per_patient_df.columns:
        per_patient_df = per_patient_df[per_patient_df["Model"].isin(available_models)].copy()
        per_patient_df["ModelLabel"] = per_patient_df["Model"].map(MODEL_LABELS)
    else:
        per_patient_df = pd.DataFrame(columns=["Patient", "Model", "ModelLabel", "Sensitivity"])

    final_compare_table = macro_df[
        ["ModelLabel", "Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "Median_Threshold"]
    ].rename(columns={"ModelLabel": "Model"})
    PIPELINE_RESULTS["baseline_compare_table"] = final_compare_table
    display(final_compare_table)

    def annotate_bars(ax, fmt="{:.3f}", offset=4):
        for patch in ax.patches:
            h = patch.get_height()
            if np.isfinite(h):
                ax.annotate(
                    fmt.format(h),
                    (patch.get_x() + patch.get_width() / 2, h),
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    xytext=(0, offset),
                    textcoords="offset points",
                )

    sns.set_theme(style="whitegrid", font_scale=1.05)
    bar_colors = [MODEL_COLORS.get(label, "#808080") for label in final_compare_table["Model"]]
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    ax1, ax2, ax3, ax4 = axes.flatten()
    ax1.bar(final_compare_table["Model"], final_compare_table["Sensitivity"], color=bar_colors, edgecolor="black", linewidth=0.8)
    ax1.set_title("(A) Macro Sensitivity", fontweight="bold")
    ax1.set_ylabel("Sensitivity")
    annotate_bars(ax1, "{:.3f}")
    ax2.bar(final_compare_table["Model"], final_compare_table["FAR_per_Hour"], color=bar_colors, edgecolor="black", linewidth=0.8)
    ax2.set_title("(B) Macro FAR per Hour", fontweight="bold")
    annotate_bars(ax2, "{:.3f}")
    ax3.bar(final_compare_table["Model"], final_compare_table["Mean_Delay_s"], color=bar_colors, edgecolor="black", linewidth=0.8)
    ax3.set_title("(C) Macro Mean Delay", fontweight="bold")
    annotate_bars(ax3, "{:.2f}")
    if per_patient_df.empty:
        ax4.axis("off")
        ax4.text(0.5, 0.5, "Per-patient benchmark results are unavailable.", ha="center", va="center", fontsize=12)
        ax4.set_title("(D) Per-Patient Sensitivity", fontweight="bold")
    else:
        patient_order = sorted(per_patient_df["Patient"].dropna().unique().tolist())
        heatmap_df = (
            per_patient_df.pivot_table(index="Patient", columns="ModelLabel", values="Sensitivity", aggfunc="mean")
            .reindex(index=patient_order, columns=[MODEL_LABELS[m] for m in available_models])
        )
        sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=0.5, cbar_kws={"label": "Sensitivity"}, ax=ax4)
        ax4.set_title("(D) Per-Patient Sensitivity", fontweight="bold")
    fig.suptitle("Baseline Comparison of Classical ML Models", fontsize=17, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.show()


Block 14B skipped because the multi-model benchmark is disabled in this v2 workflow.


### Block 14B 输出解读补充
- 图形适合快速汇报，但做严谨对比时仍然建议同时看数值表。


## Block 15：RF 全特征 vs 最终 Top-k 对比

### 这个代码块在做什么
- 比较“全特征 RF”和“最终 top-k RF”的事件级表现。
- 目的是看特征压缩之后是否仍然保留足够的检测性能。

### 输出怎么解读
- `Sensitivity` 如果接近，说明 top-k 压缩没有明显损害召回。
- `FAR_per_Hour` 如果变低，说明精简特征反而更稳。
- 这是后面部署压缩是否合理的重要依据。


In [17]:
# 中文导读：这里比较 RF 全特征版和最终 top-k 版，回答“压缩后是否还值得用”。
# =============================================================================
# Block 15. RF full feature vs final top-k event comparison
# =============================================================================
if not RUN_RF_FULL_COMPARE:
    print("⏭ Block 15: RF Full vs Top-k skipped (RUN_RF_FULL_COMPARE = False)")
else:
    deploy_cfg, deploy_caches, final_top_k, deploy_cfg_source = resolve_final_rf_context()
    PIPELINE_RESULTS["deploy_cfg"] = deploy_cfg
    PIPELINE_RESULTS["deploy_caches"] = deploy_caches
    PIPELINE_RESULTS["final_top_k"] = final_top_k
    # 优先复用 Block 14 已经跑过的随机森林结果，避免重复计算。
    rf_full_results = PIPELINE_RESULTS.get("multi_model_runs", {}).get("random_forest")
    if rf_full_results is None:
        rf_full_results = evaluate_many_patients(caches=deploy_caches, cfg=deploy_cfg, model_name="random_forest", top_k=None)
    rf_topk_results = evaluate_many_patients(caches=deploy_caches, cfg=deploy_cfg, model_name="random_forest", top_k=final_top_k)
    rf_full_matrix_df = build_event_detection_matrix(rf_full_results, deploy_cfg, label="rf_full_feature")
    rf_topk_matrix_df = build_event_detection_matrix(rf_topk_results, deploy_cfg, label=f"rf_top{final_top_k}")
    rf_matrix_compare_df = pd.concat([rf_full_matrix_df, rf_topk_matrix_df], ignore_index=True)
    PIPELINE_RESULTS["rf_full_results"] = rf_full_results
    PIPELINE_RESULTS["rf_topk_results"] = rf_topk_results
    PIPELINE_RESULTS["rf_matrix_compare_df"] = rf_matrix_compare_df
    display(rf_matrix_compare_df)


⏭ Block 15: RF Full vs Top-k skipped (RUN_RF_FULL_COMPARE = False)


### Block 15 输出解读补充
- 这是判断“压缩是否值得”的关键表。
- 如果 top-k 的 FAR 明显更低，通常说明降维帮模型减少了噪声。


## Block 16：最终 10 位病人的 RF LOSO 验证

### 这个代码块在做什么
- 用最终选定的 RF 配置，在 `chb01` 到 `chb10` 上做正式 LOSO 验证。
- 同时为每位病人生成 `threshold_map`，供后面做病人级最终模型。

### 输出怎么解读
- 第一张表是每位病人的最终结果。
- 第二张表是宏观汇总，通常也是你论文/报告里最重要的一张最终 RF 表。
- `threshold_map` 的大小应该和成功验证的病人数一致。


In [18]:
# 中文导读：这里是最终 RF 的正式 LOSO 验证，同时生成病人级阈值映射。
# =============================================================================
# Block 16. Final 10-patient RF LOSO validation
# =============================================================================
deploy_cfg, deploy_caches, final_top_k, deploy_cfg_source = resolve_final_rf_context()
PIPELINE_RESULTS["deploy_cfg"] = deploy_cfg
PIPELINE_RESULTS["deploy_caches"] = deploy_caches
PIPELINE_RESULTS["final_top_k"] = final_top_k
rf10_loso_results = evaluate_many_patients(caches=deploy_caches, cfg=deploy_cfg, model_name="random_forest", top_k=final_top_k)
rf10_loso_df = rf10_loso_results.get("summary_df", pd.DataFrame()).copy()
display(rf10_loso_df)
rf10_macro_df = pd.DataFrame([{"Patient": "MACRO", "Model": "random_forest", "TopK": int(final_top_k), "Hours": rf10_loso_df["Hours"].sum() if not rf10_loso_df.empty else np.nan, "True_Seizures": rf10_loso_df["True_Seizures"].sum() if not rf10_loso_df.empty else np.nan, "Sensitivity": rf10_loso_df["Sensitivity"].mean() if not rf10_loso_df.empty else np.nan, "FAR_per_Hour": rf10_loso_df["FAR_per_Hour"].median() if not rf10_loso_df.empty else np.nan, "Mean_Delay_s": rf10_loso_df["Mean_Delay_s"].mean() if not rf10_loso_df.empty else np.nan, "Median_Delay_s": rf10_loso_df["Median_Delay_s"].median() if not rf10_loso_df.empty else np.nan, "Median_Threshold": rf10_loso_df["Median_Threshold"].median() if not rf10_loso_df.empty else np.nan}])
display(rf10_macro_df)
# threshold_map 的作用：把每位病人最终验证得到的中位阈值保留下来，供后面病人级最终模型直接复用。
threshold_map = {}
if not rf10_loso_df.empty:
    for _, row in rf10_loso_df.iterrows():
        pid = str(row["Patient"])
        thr = row.get("Median_Threshold", np.nan)
        if not pd.isna(thr):
            threshold_map[pid] = float(thr)
PIPELINE_RESULTS["rf10_loso_results"] = rf10_loso_results
PIPELINE_RESULTS["rf10_loso_df"] = rf10_loso_df
PIPELINE_RESULTS["rf10_macro_df"] = rf10_macro_df
PIPELINE_RESULTS["threshold_map"] = threshold_map


[Cache] Ready for TUNED_RF_CFG final RF context: 10 patients


[chb01] random_forest | Sens=100.00% | FAR/hr=0.6658 | MedianThr=0.499


[chb02] random_forest | Sens=100.00% | FAR/hr=0.2552 | MedianThr=0.308


[chb03] random_forest | Sens=100.00% | FAR/hr=0.5263 | MedianThr=0.672


[chb04] random_forest | Sens=100.00% | FAR/hr=0.5919 | MedianThr=0.551


...

[chb08] random_forest | Sens=100.00% | FAR/hr=0.7498 | MedianThr=0.464


[chb09] random_forest | Sens=100.00% | FAR/hr=0.2171 | MedianThr=0.811


[chb10] random_forest | Sens=100.00% | FAR/hr=0.1199 | MedianThr=0.811


,Patient,Model,TopK,Hours,True_Seizures,Sensitivity,FAR_per_Hour,Mean_Delay_s,Median_Delay_s,Median_Threshold
0,chb01,random_forest,30,40.551667,7.0,1.000000,0.665817,4.285714,2.0,0.499
1,chb02,random_forest,30,35.266111,3.0,1.000000,0.255203,3.333333,4.0,0.308
2,chb03,random_forest,30,38.001667,7.0,1.000000,0.526293,4.000000,2.0,0.672
3,chb04,random_forest,30,152.056111,4.0,1.000000,0.591887,35.500000,15.0,0.551
4,chb05,random_forest,30,39.002778,5.0,1.000000,0.179474,12.400000,6.0,0.482
5,chb06,random_forest,30,62.734444,9.0,0.777778,0.797010,2.857143,4.0,0.204
6,chb07,random_forest,30,67.051667,3.0,1.000000,0.089483,15.333333,18.0,0.690
7,chb08,random_forest,30,20.006111,5.0,1.000000,0.749771,11.200000,12.0,0.464
8,chb09,random_forest,30,59.870000,4.0,1.000000,0.217137,7.500000,6.0,0.811
9,chb10,random_forest,30,50.022778,7.0,1.000000,0.119945,3.142857,4.0,0.811


,Patient,Model,TopK,Hours,True_Seizures,Sensitivity,FAR_per_Hour,Mean_Delay_s,Median_Delay_s,Median_Threshold
0,MACRO,random_forest,30,564.563333,54.0,0.977778,0.390748,9.955238,5.0,0.525


### Block 16 输出解读补充
- 这一块的宏观表是整份传统 ML 流程里最重要的最终结果之一。
- 如果你要写总结，通常要引用这里的数值。


## Block 17：训练每位病人的最终部署 RF

### 这个代码块在做什么
- 基于 Block 16 的阈值和 top-k，给每位病人训练一个最终部署模型。
- 同时选一个 demo patient，导出后续展示和压缩要用的 `deploy_*` 变量。

### 输出怎么解读
- `patient_final_model_index_df`：看每位病人的最终模型是否训练成功。
- `deploy_importance_df`：看 demo patient 的 top-k 特征重要性排序。
- 如果某位病人状态不是 `ok`，说明部署链还不完整。


In [19]:
# Block 17. Patient-specific final model training
print("Block 17 skipped in clinical error analysis v2 (patient-specific deployment models are not required for error analysis).")
PIPELINE_RESULTS["patient_final_models"] = {}
PIPELINE_RESULTS["patient_final_model_index_df"] = pd.DataFrame()
PIPELINE_RESULTS["deploy_importance_df"] = pd.DataFrame()
gc.collect()


Block 17 skipped in clinical error analysis v2 (patient-specific deployment models are not required for error analysis).


360

### Block 17 输出解读补充
- 这一步完成后，整条“训练 -> 验证 -> 病人级最终模型 -> 部署变量”链路才算真正闭环。


## Block 18：把结果整理成更适合阅读的表格

### 这个代码块在做什么
- 不重新训练模型，只把已有结果重新排版展示。
- 方便你在 notebook 里快速检查关键输出是否齐全。

### 输出怎么解读
- `Variable Check` 先看关键变量是否都存在。
- 后面各表分别对应 baseline、RF 对比、最终验证、病人模型索引、特征重要性等结果组。
- 这是最适合人工审阅的一站式汇总页。


In [20]:
# 中文导读：这里不会重跑实验，只是把已有结果分组排版，方便人工检查。
# =============================================================================
# Block 18. Organized output display
# =============================================================================
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def _md(text: str):
    display(Markdown(text))


def _safe_copy_df(name: str) -> pd.DataFrame:
    obj = PIPELINE_RESULTS.get(name, None)
    return obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame()


def _show_df(title: str, df: pd.DataFrame, note: str = "", round_cols: dict = None):
    _md(f"## {title}")
    if note:
        _md(note)
    if df is None or df.empty:
        print("[Empty table / variable not found]")
        return
    out = df.copy()
    if round_cols:
        for col, digits in round_cols.items():
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce").round(digits)
    display(out)
    print(f"shape = {out.shape}")


def _reorder_cols(df: pd.DataFrame, preferred_cols: list) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    cols_exist = [c for c in preferred_cols if c in df.columns]
    cols_rest = [c for c in df.columns if c not in cols_exist]
    return df[cols_exist + cols_rest].copy()


def _add_metric_rank(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    out = df.copy()
    if {"Sensitivity", "FAR_per_Hour", "Mean_Delay_s"}.issubset(out.columns):
        out = out.sort_values(["Sensitivity", "FAR_per_Hour", "Mean_Delay_s"], ascending=[False, True, True]).reset_index(drop=True)
        out.insert(0, "Rank", np.arange(1, len(out) + 1))
    return out

# 先检查关键结果变量是否存在，确认前面的训练/验证链路已经完整跑通。
vars_to_check = ["final_top_k", "multi_model_macro_df", "multi_model_per_patient_df", "rf_matrix_compare_df", "rf10_loso_df", "rf10_macro_df", "patient_final_model_index_df", "deploy_importance_df", "demo_patient_id", "deploy_threshold"]
var_status_df = pd.DataFrame([{"Variable": v, "Exists": v in PIPELINE_RESULTS, "Type": type(PIPELINE_RESULTS.get(v)).__name__ if v in PIPELINE_RESULTS else "-"} for v in vars_to_check])
_show_df("0. Variable Check", var_status_df, note="Confirm the key outputs from Block 14-17 are present.")

multi_model_macro_df_clean = _add_metric_rank(_reorder_cols(_safe_copy_df("multi_model_macro_df"), ["Model", "TopK", "Patients", "Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "Median_Threshold"]))
_show_df("1. Multi-Model Benchmark - Macro Ranking", multi_model_macro_df_clean)
multi_model_per_patient_df_clean = _reorder_cols(_safe_copy_df("multi_model_per_patient_df"), ["Patient", "Model", "TopK", "Hours", "True_Seizures", "Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "Median_Threshold"])
_show_df("2. Multi-Model Benchmark - Per-Patient Results", multi_model_per_patient_df_clean)
rf_matrix_compare_df_clean = _reorder_cols(_safe_copy_df("rf_matrix_compare_df"), ["Variant", "TP_events", "FN_events", "FP_events", "Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Hours", "Patients"])
_show_df("3. RF Full Feature vs RF Final Top-k", rf_matrix_compare_df_clean)
rf10_loso_df_clean = _reorder_cols(_safe_copy_df("rf10_loso_df"), ["Patient", "Hours", "True_Seizures", "Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "Median_Threshold"])
_show_df("4A. Final 10-Patient RF Validation - Per-Patient", rf10_loso_df_clean)
rf10_macro_df_clean = _reorder_cols(_safe_copy_df("rf10_macro_df"), ["Patient", "Model", "TopK", "Hours", "True_Seizures", "Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "Median_Threshold"])
_show_df("4B. Final 10-Patient RF Validation - Macro Summary", rf10_macro_df_clean)
patient_final_model_index_df_clean = _reorder_cols(_safe_copy_df("patient_final_model_index_df"), ["Patient", "Status", "Reason", "Threshold", "InputDim", "TrainRows", "TrainPos", "TrainNeg"])
_show_df("5. Final Patient-Specific RF Model Index", patient_final_model_index_df_clean)
deploy_importance_df_clean = _reorder_cols(_safe_copy_df("deploy_importance_df"), ["Global_Index", "Feature_Name", "Importance"])
_show_df("6. Demo Patient Top-20 Feature Importance", deploy_importance_df_clean.head(20))


## 0. Variable Check

Confirm the key outputs from Block 14-17 are present.

,Variable,Exists,Type
0,final_top_k,True,int
1,multi_model_macro_df,True,DataFrame
2,multi_model_per_patient_df,True,DataFrame
3,rf_matrix_compare_df,False,-
4,rf10_loso_df,True,DataFrame
5,rf10_macro_df,True,DataFrame
6,patient_final_model_index_df,True,DataFrame
7,deploy_importance_df,True,DataFrame
8,demo_patient_id,False,-
9,deploy_threshold,False,-


shape = (10, 3)


## 1. Multi-Model Benchmark - Macro Ranking

...

shape = (1, 10)


## 5. Final Patient-Specific RF Model Index

[Empty table / variable not found]


## 6. Demo Patient Top-20 Feature Importance

[Empty table / variable not found]


### Block 18 输出解读补充
- 如果你只想快速检查 notebook 是否完整跑通，优先看这个 block 就够了。


## Block 19：特征重要性可视化

### 这个代码块在做什么
- 用 demo patient 的最终 RF 模型，把重要特征画出来。
- 帮助理解模型到底依赖哪些频段、哪些通道、哪些时间滞后。

### 输出怎么解读
- 横轴越大，说明该特征对当前 demo patient 的模型越重要。
- 如果 top 特征集中在某些通道/频带，后续可以据此做特征分析或模型裁剪。


In [21]:
# Block 19. Feature Importance Visualization
print("Block 19 skipped in clinical error analysis v2 (feature-importance plotting is not required for the error-analysis validation path).")


Block 19 skipped in clinical error analysis v2 (feature-importance plotting is not required for the error-analysis validation path).


### Block 19 输出解读补充
- 重要性高不等于因果关系强，但它能帮助你理解模型主要依赖的信息来源。


## Block 20：推理延迟与资源画像

### 这个代码块在做什么
- 估计最终部署模型一次推理的耗时、内存占用等资源信息。
- 这是工程可部署性检查，不是检测性能比较本身。

### 输出怎么解读
- 延迟越低越适合实时部署。
- 如果这里很慢，不一定说明模型效果差，但说明部署代价偏高。


In [22]:
# 中文导读：这里评估的是部署代价，例如推理耗时和资源占用，不是检测准确率本身。
# =============================================================================
# Block 20. Inference Latency and Resource Profiling (Demo Patient Model) - Academic Table Output
# =============================================================================
if not RUN_LATENCY_PROFILING:
    print("⏭ Block 20: Latency Profiling skipped (RUN_LATENCY_PROFILING = False)")
else:
    def benchmark_single_epoch_latency(model, X_stream: np.ndarray, n_steps: int = 1000) -> Dict[str, float]:
        n_steps = min(n_steps, len(X_stream))
        timings = []
        for i in range(n_steps):
            sample = X_stream[i : i + 1]
            t0 = time.perf_counter()
            _ = model.predict_proba(sample)
            t1 = time.perf_counter()
            timings.append(t1 - t0)
        timings = np.asarray(timings)
        return {
            "n_steps": int(n_steps),
            "mean_ms": float(np.mean(timings) * 1000.0),
            "median_ms": float(np.median(timings) * 1000.0),
            "p95_ms": float(np.percentile(timings, 95) * 1000.0),
            "fps_estimate": float(1.0 / np.mean(timings)) if np.mean(timings) > 0 else np.nan,
        }

    def profile_rf_model(model, input_dim: int) -> Dict[str, float]:
        buffer = io.BytesIO()
        joblib.dump(model, buffer)
        raw_bytes = buffer.getvalue()
        estimators = list(getattr(model, "estimators_", []))
        total_nodes = int(sum(est.tree_.node_count for est in estimators)) if estimators else 0
        total_leaves = int(sum(np.sum(est.tree_.children_left == -1) for est in estimators)) if estimators else 0
        max_depth = int(max(est.tree_.max_depth for est in estimators)) if estimators else 0
        mean_depth = float(np.mean([est.tree_.max_depth for est in estimators])) if estimators else 0.0

        return {
            "input_dim": int(input_dim),
            "memory_mb": float(len(raw_bytes) / (1024.0 * 1024.0)), # Changed to MB for better readability
            "n_trees": int(len(estimators)),
            "total_nodes": int(total_nodes),
            "total_leaves": int(total_leaves),
            "decision_nodes": int(total_nodes - total_leaves),
            "max_tree_depth": int(max_depth),
            "mean_tree_depth": float(mean_depth),
        }

    # --- Execution Logic ---
    if "demo_patient_id" not in PIPELINE_RESULTS or "patient_final_models" not in PIPELINE_RESULTS:
        raise ValueError("Please run Block 17 first to generate patient_final_models.")

    demo_bundle = patient_final_models[demo_patient_id]
    demo_payload = deploy_caches[demo_patient_id] # Ensure deploy_caches is available in global variables
    X_demo_full, y_demo_full = collect_rows_from_files(demo_payload, list(demo_payload["files"].keys()), None)
    X_demo_light = X_demo_full[:, demo_bundle["top_k_indices"]]

    X_latency_stream = X_demo_light[: min(1500, len(X_demo_light))]
    lat = benchmark_single_epoch_latency(demo_bundle["model"], X_latency_stream, n_steps=1000)
    hw = profile_rf_model(demo_bundle["model"], input_dim=int(demo_bundle["input_dim"]))

    # --- Build formatted Markdown tables ---
    markdown_output = f"""
    ### Table 1: Real-time Inference Latency Evaluation for Wearable Devices (Demo Patient: {demo_patient_id})

    | Metrics | Value | Description |
    | :--- | :--- | :--- |
    | **Test Epochs** | {lat['n_steps']} | Baseline for continuous inference stability test |
    | **Mean Latency** | **{lat['mean_ms']:.2f} ms** | Average processing time per data epoch |
    | **Median Latency** | {lat['median_ms']:.2f} ms | Central tendency of the data distribution |
    | **P95 Latency** | **{lat['p95_ms']:.2f} ms** | Worst-case response guarantee of the system |
    | **Throughput** | {lat['fps_estimate']:.1f} FPS | Number of EEG epochs processed per second |

    ### Table 2: Hardware and Memory Overhead Profiling of Random Forest Model

    | Model Architecture | Value | Resource Overhead | Value |
    | :--- | :--- | :--- | :--- |
    | **Input Dim** | {hw['input_dim']} | **Memory Footprint**| **{hw['memory_mb']:.2f} MB** |
    | **Number of Trees** | {hw['n_trees']} | **Total Nodes** | {hw['total_nodes']:,} |
    | **Max Tree Depth** | {hw['max_tree_depth']} | **Decision Nodes** | {hw['decision_nodes']:,} |
    | **Mean Tree Depth** | {hw['mean_tree_depth']:.1f} | **Leaf Nodes** | {hw['total_leaves']:,} |
    """

    display(Markdown(markdown_output))
    print("✅ Block 20 Complete: Resource profiling for demo patient model generated.")

⏭ Block 20: Latency Profiling skipped (RUN_LATENCY_PROFILING = False)


### Block 20 输出解读补充
- 这里更多回答“能不能部署”，而不是“准不准”。


## Block 21：把 RF 模型导出成 C Header

### 这个代码块在做什么
- 把最终 RF 结构压缩导出成 C 侧更容易使用的格式。
- 主要面向嵌入式/轻量部署场景。

### 输出怎么解读
- 关注是否成功生成导出文件、导出特征维度是否正确。
- 如果这里失败，通常和模型结构、输出路径或文件权限有关。


In [23]:
# 中文导读：这里把最终 RF 导出成更适合嵌入式使用的 C Header 表达形式。
# =============================================================================
# Block 21. Extreme Model Compression: Exporting RF to C Header (Optimized)
# =============================================================================
# Ensure emlearn is installed for embedded C generation
if not RUN_C_EXPORT:
    print("⏭ Block 21: C Header Export skipped (RUN_C_EXPORT = False)")
else:
    try:
        import emlearn
    except ImportError:
        print("emlearn library not found. Installing it now...")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "emlearn"])
        import emlearn

    print("---------------------------------------------------------")
    print("Block 21: Extreme Model Compression (RF -> Optimized C Header)")
    print("---------------------------------------------------------")

    if "deploy_model" not in PIPELINE_RESULTS:
        raise ValueError("deploy_model not found. Please ensure Block 17 has been run!")

    # Identify the number of trees in the Random Forest
    tree_count = len(getattr(deploy_model, "estimators_", []))
    print(f"Converting the model with {tree_count} trees to an optimized C header file...")
    print("Using emlearn to flatten the tree structure into 1D arrays for MCU deployment...\n")

    try:
        # Convert the scikit-learn model to embedded C using the 'inline' method
        # This avoids deep recursion and uses efficient array lookups instead.
        cmodel = emlearn.convert(deploy_model, method='inline')

        # Save the generated C code to a local header file
        c_file_path = "eeg_seizure_detector_rf.h"
        cmodel.save(file=c_file_path, name="eeg_rf")

        # Measure the actual file size of the generated header
        file_size_kb = os.path.getsize(c_file_path) / 1024.0

        print(f"✅ Conversion successful! Optimized C header saved to: {c_file_path}")
        print(f"📄 File Size (Memory Footprint): {file_size_kb:.2f} KB")
        print("\n💡 Engineering Note:")
        print("This file contains flat arrays (nodes, leaves, thresholds) rather than nested if-else statements.")
        print("It eliminates stack overflow risks and is fully compatible with MCU Flash memory (PROGMEM).")

    except Exception as e:
        print(f"❌ An error occurred during conversion: {e}")

⏭ Block 21: C Header Export skipped (RUN_C_EXPORT = False)


### Block 21 输出解读补充
- 这一步更偏工程导出，只有当你真的需要 C/嵌入式部署时才是重点。


## Block 22：病人级最终模型汇总

### 这个代码块在做什么
- 把每位病人的最终模型状态、阈值、输入维度等做成总结表。
- 方便你快速查看部署对象是否完整。

### 输出怎么解读
- `ok` 表示该病人的最终模型已经准备好。
- `Threshold / InputDim / TrainRows` 可以帮助你理解不同病人的训练规模差异。


In [24]:
# Block 22. Patient-Level Final Model Summary
print("Block 22 skipped in clinical error analysis v2 (patient-level deployment summary is not required for the error-analysis validation path).")
PIPELINE_RESULTS["patient_model_summary_df"] = pd.DataFrame()


Block 22 skipped in clinical error analysis v2 (patient-level deployment summary is not required for the error-analysis validation path).


### Block 22 输出解读补充
- 这是面向部署管理的汇总表，适合快速核查每位病人的最终状态。


## Block 23：导出每位病人的最终模型与 metadata

### 这个代码块在做什么
- 按病人分别导出模型文件和元数据文件。
- 给后续接口调用、部署脚本、离线验证使用。

### 输出怎么解读
- 输出表里会给出每位病人的模型路径、metadata 路径、阈值和输入维度。
- 如果某位病人没导出成功，要先回头看 Block 17 是否训练成功。


In [25]:
# 中文导读：这里把每位病人的最终模型和 metadata 单独导出到磁盘。
# =============================================================================
# Block 23. 导出 10 位病人的最终模型与 metadata
# =============================================================================
if not RUN_PATIENT_EXPORT:
    print("⏭ Block 23: Patient model export skipped (RUN_PATIENT_EXPORT = False)")
else:
    if "patient_final_models" not in PIPELINE_RESULTS or len(patient_final_models) == 0:
        raise ValueError("请先运行 Block 17，生成 patient_final_models。")
    base_export_paths = get_export_paths(deploy_cfg)
    export_root = base_export_paths["model"].parent
    patient_export_root = export_root / "patient_models"
    ensure_dir(patient_export_root)
    export_rows = []
    for pid, bundle in sorted(patient_final_models.items()):
        patient_dir = patient_export_root / pid
        ensure_dir(patient_dir)
        model_path = patient_dir / "rf_model.joblib"
        metadata_path = patient_dir / "metadata.json"
        joblib.dump(bundle["model"], model_path)
        save_json({"patient_id": pid, "threshold": float(bundle["threshold"]), "input_dim": int(bundle["input_dim"]), "requested_top_k": int(final_top_k)}, metadata_path)
        export_rows.append({"Patient": pid, "ModelPath": str(model_path), "MetadataPath": str(metadata_path), "Threshold": float(bundle["threshold"]), "InputDim": int(bundle["input_dim"]), "TopK": int(len(bundle["top_k_indices"]))})
    patient_export_index_df = pd.DataFrame(export_rows).sort_values("Patient").reset_index(drop=True)
    display(patient_export_index_df)
    PIPELINE_RESULTS["patient_export_index_df"] = patient_export_index_df


⏭ Block 23: Patient model export skipped (RUN_PATIENT_EXPORT = False)


### Block 23 输出解读补充
- 导出成功后，后面的外部服务或脚本就不必再依赖 notebook 内存变量。


## Block 26：临床视角可视化

### 这个代码块在做什么
- 选一个案例，把概率曲线、真实标签和滤波后的 EEG 一起画出来。
- 这是最贴近“临床观感”的结果展示方式。

### 输出怎么解读
- 看概率峰值是否和真实发作区间对齐。
- 看平滑后是否能减少零碎误报。
- 如果概率曲线乱跳但真实标签附近没有明显提升，说明模型可能还不够稳。


In [26]:
# 中文导读：这里做病例级可视化，用概率曲线和真实标签一起看模型行为。
# =============================================================================
# Block 26. Clinical single-case view: probability + true state + filtered EEG
# =============================================================================
if not RUN_STREAM_DEMO:
    print("⏭ Block 26: Clinical Visualization skipped (RUN_STREAM_DEMO = False)")
else:
    def align_channels_window(
        raw: mne.io.BaseRaw,
        target_channels: Sequence[str],
        start_idx: int,
        stop_idx: int,
        policy: str = "strict",
    ) -> Tuple[np.ndarray, Dict[str, Any]]:
        """Window-based channel alignment with pre-allocated output (optimization #19).
        NOTE: Logic mirrors align_channels(); consider merging in future refactor.
        """
        normalized_targets = [normalize_channel_name(ch) for ch in target_channels]
        existing = {normalize_channel_name(ch): ch for ch in raw.ch_names}

        n_ch = len(normalized_targets)
        n_times = stop_idx - start_idx
        data = np.zeros((n_ch, n_times), dtype=float)
        missing_channels = []
        reversed_channels = []

        for i, target in enumerate(normalized_targets):
            if target in existing:
                data[i] = raw.get_data(picks=[existing[target]], start=start_idx, stop=stop_idx)[0]
                continue

            if "-" in target:
                reverse_target = "-".join(target.split("-")[::-1])
                if reverse_target in existing:
                    data[i] = -raw.get_data(picks=[existing[reverse_target]], start=start_idx, stop=stop_idx)[0]
                    reversed_channels.append(target)
                    continue

            if policy == "zero_fill":
                # data[i] already zeros
                missing_channels.append(target)
            else:
                raise ValueError(f"Missing target channel: {target}")

        info = {
            "missing_channels": missing_channels,
            "missing_count": len(missing_channels),
            "reversed_channels": reversed_channels,
        }
        return data, info


    def build_clinical_case_dataframe(
        patient_id: str,
        patient_payload: Dict[str, Any],
        model,
        feature_indices: np.ndarray,
        threshold: float,
        pre_seconds: int = 12,
        post_seconds: int = 14,
    ) -> Tuple[str, pd.DataFrame, Dict[str, Any]]:
        seizure_files = [
            f for f, d in patient_payload["files"].items()
            if d["has_seizure"] and d.get("has_positive_epoch", False)
        ]
        if not seizure_files:
            seizure_files = [f for f, d in patient_payload["files"].items() if d["has_seizure"]]
        if not seizure_files:
            raise ValueError(f"No seizure file found for {patient_id}.")

        file_name = sorted(seizure_files)[0]
        item = patient_payload["files"][file_name]

        X = item["X"][:, feature_indices]
        y = item["y"].astype(np.int8)

        seizure_positions = np.flatnonzero(y == 1)
        if len(seizure_positions) == 0:
            raise ValueError(f"No positive epochs in {patient_id}/{file_name}.")

        onset_epoch = int(seizure_positions[0])
        _cfg_case = deploy_cfg if "deploy_cfg" in PIPELINE_RESULTS else CFG
        epoch_len_s = _cfg_case.feature.epoch_len_s
        onset_time_s = onset_epoch * epoch_len_s

        pre_epochs = max(1, int(np.ceil(pre_seconds / epoch_len_s)))
        post_epochs = max(1, int(np.ceil(post_seconds / epoch_len_s)))

        start_epoch = max(0, onset_epoch - pre_epochs)
        end_epoch = min(len(y), onset_epoch + post_epochs + 1)

        epoch_indices = np.arange(start_epoch, end_epoch)
        probs = model.predict_proba(X[start_epoch:end_epoch])[:, 1]
        alarms = probs >= threshold

        case_df = pd.DataFrame(
            {
                "epoch_index": epoch_indices,
                "time_s": epoch_indices * epoch_len_s,
                "time_rel_s": (epoch_indices - onset_epoch) * epoch_len_s,
                "y_true": y[start_epoch:end_epoch],
                "prob": probs,
                "alarm": alarms.astype(bool),
            }
        )

        hit_in_seizure = bool(np.any((case_df["y_true"] == 1) & (case_df["alarm"])))
        false_alarm_before = bool(np.any((case_df["time_rel_s"] < 0) & (case_df["alarm"])))
        first_alarm_rel_s = float(case_df.loc[case_df["alarm"], "time_rel_s"].iloc[0]) if np.any(case_df["alarm"]) else np.nan

        metrics = {
            "patient_id": patient_id,
            "file_name": file_name,
            "onset_epoch": onset_epoch,
            "onset_time_s": onset_time_s,
            "threshold": float(threshold),
            "hit_in_seizure": hit_in_seizure,
            "false_alarm_before": false_alarm_before,
            "first_alarm_rel_s": first_alarm_rel_s,
        }
        return file_name, case_df, metrics


    def load_filtered_eeg_segment(
        cfg: ExperimentConfig,
        patient_id: str,
        file_name: str,
        start_time_s: float,
        end_time_s: float,
        onset_time_s: float,
        plot_channels: Sequence[str],
    ) -> Tuple[np.ndarray, np.ndarray, List[str], Dict[str, Any]]:
        edf_path = Path(cfg.data_root) / patient_id / file_name
        if not edf_path.exists():
            raise FileNotFoundError(f"EDF not found: {edf_path}")

        raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
        raw = deduplicate_and_normalize_raw(raw)
        fs = int(raw.info["sfreq"])

        start_idx = max(0, int(start_time_s * fs))
        stop_idx = min(raw.n_times, int(end_time_s * fs))
        if stop_idx <= start_idx:
            raw.close()
            raise ValueError("Invalid EEG window: stop <= start")

        data_window, align_info = align_channels_window(
            raw=raw,
            target_channels=cfg.channels,
            start_idx=start_idx,
            stop_idx=stop_idx,
            policy=cfg.feature.channel_missing_policy,
        )
        raw.close()

        if cfg.feature.scale_to_uV:
            data_window = data_window * 1e6

        data_bp = bandpass_filter_multich(
            data_window,
            fs=fs,
            lowcut=cfg.feature.bandpass_low_hz,
            highcut=cfg.feature.bandpass_high_hz,
            method=cfg.feature.bandpass_method,
            butter_order=cfg.feature.butter_order,
        )

        target_norm = [normalize_channel_name(ch) for ch in cfg.channels]
        plot_indices = []
        plot_names = []
        for ch in plot_channels:
            ch_norm = normalize_channel_name(ch)
            if ch_norm in target_norm:
                plot_indices.append(target_norm.index(ch_norm))
                plot_names.append(ch)

        if len(plot_indices) == 0:
            plot_indices = list(range(min(4, data_bp.shape[0])))
            plot_names = [cfg.channels[idx] for idx in plot_indices]

        eeg_plot = data_bp[plot_indices]
        t_abs = np.arange(eeg_plot.shape[1]) / fs + (start_idx / fs)
        t_rel = t_abs - onset_time_s

        return eeg_plot, t_rel, plot_names, align_info


    clinical_patient_id = demo_patient_id if "demo_patient_id" in PIPELINE_RESULTS else next(iter(caches.keys()))
    clinical_file, clinical_df, clinical_metrics = build_clinical_case_dataframe(
        patient_id=clinical_patient_id,
        patient_payload=(deploy_caches if "deploy_caches" in PIPELINE_RESULTS else caches)[clinical_patient_id],
        model=deploy_model,
        feature_indices=deploy_topk_indices,
        threshold=deploy_threshold,
        pre_seconds=15,
        post_seconds=15,
    )

    window_start_s = float(clinical_metrics["onset_time_s"] - 15)
    window_end_s = float(clinical_metrics["onset_time_s"] + 15)

    eeg_plot, eeg_t_rel, eeg_names, eeg_align_info = load_filtered_eeg_segment(
        cfg=deploy_cfg if "deploy_cfg" in PIPELINE_RESULTS else CFG,
        patient_id=clinical_patient_id,
        file_name=clinical_file,
        start_time_s=max(0.0, window_start_s),
        end_time_s=window_end_s,
        onset_time_s=float(clinical_metrics["onset_time_s"]),
        plot_channels=["FP1-F7", "F7-T7", "T7-P7", "P7-O1"],
    )

    status_row = pd.DataFrame(
        [
            {
                "Patient": clinical_patient_id,
                "File": clinical_file,
                "Threshold": clinical_metrics["threshold"],
                "Hit_In_Seizure": clinical_metrics["hit_in_seizure"],
                "False_Alarm_Before_Onset": clinical_metrics["false_alarm_before"],
                "First_Alarm_Relative_s": clinical_metrics["first_alarm_rel_s"],
                "Missing_Channels": eeg_align_info["missing_count"],
            }
        ]
    )
    display(status_row)
    display(clinical_df)

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [1.0, 1.6]})

    # Panel 1: probability + threshold + true seizure state in the same panel
    ax_prob = axes[0]
    ax_true = ax_prob.twinx()

    ax_prob.plot(clinical_df["time_rel_s"], clinical_df["prob"], marker="o", lw=1.8, label="Predicted probability")
    ax_prob.axhline(clinical_metrics["threshold"], color="tab:red", linestyle="--", label=f"Threshold={clinical_metrics['threshold']:.2f}")
    ax_prob.axvline(0, color="black", linestyle="--", alpha=0.85, label="Seizure onset")

    ax_true.step(
        clinical_df["time_rel_s"],
        clinical_df["y_true"],
        where="mid",
        color="tab:green",
        lw=1.8,
        label="True seizure state (0/1)",
    )
    ax_true.fill_between(
        clinical_df["time_rel_s"],
        0,
        clinical_df["y_true"],
        step="mid",
        color="tab:green",
        alpha=0.15,
    )

    ax_prob.set_ylabel("Predicted probability")
    ax_prob.set_ylim(0, 1.05)
    ax_true.set_ylabel("True seizure state")
    ax_true.set_ylim(-0.05, 1.05)
    ax_true.set_yticks([0, 1])
    ax_prob.set_title(f"Clinical comparison around onset ({clinical_patient_id} / {clinical_file})")
    ax_prob.grid(True, linestyle="--", alpha=0.35)

    h1, l1 = ax_prob.get_legend_handles_labels()
    h2, l2 = ax_true.get_legend_handles_labels()
    ax_prob.legend(h1 + h2, l1 + l2, loc="best")

    # Panel 2: filtered raw EEG for clinical review
    signal_scale = float(np.nanpercentile(np.abs(eeg_plot), 95))
    offset = max(signal_scale * 2.5, 5.0)
    for i, (name, sig) in enumerate(zip(eeg_names, eeg_plot)):
        axes[1].plot(eeg_t_rel, sig + i * offset, lw=0.8)

    axes[1].set_yticks([i * offset for i in range(len(eeg_names))])
    axes[1].set_yticklabels(eeg_names)
    axes[1].axvline(0, color="black", linestyle="--", alpha=0.85)
    axes[1].set_xlabel("Relative time to onset (s)")
    axes[1].set_ylabel("Filtered raw EEG (uV, stacked)")
    _cfg_vis = deploy_cfg if "deploy_cfg" in PIPELINE_RESULTS else CFG
    axes[1].set_title(
        f"Bandpass-filtered raw EEG in same window (method={_cfg_vis.feature.bandpass_method}, order={_cfg_vis.feature.butter_order})"
    )
    axes[1].grid(True, linestyle="--", alpha=0.25)

    plt.tight_layout()
    plt.show()

    print(
        f"Clinical check | Hit in seizure: {clinical_metrics['hit_in_seizure']} | "
        f"False alarm before onset: {clinical_metrics['false_alarm_before']} | "
        f"First alarm relative time (s): {clinical_metrics['first_alarm_rel_s']}"
    )
    print("Block 26 done: probability-vs-true-state and filtered raw EEG are both shown.")




⏭ Block 26: Clinical Visualization skipped (RUN_STREAM_DEMO = False)


### Block 26 输出解读补充
- 这一页最适合做案例展示，尤其适合报告里解释“模型到底是怎么报出来的”。


## Block 27：临床误差分析配置

### 这个代码块在做什么
这一块把临床误差分析需要的配置集中起来，包括重点病人、延迟判定阈值、转移态容忍窗口、图片导出目录，以及是否执行局部变体实验和全体 10 位病人的二次验证。

### 为什么要单独做这一层
前面的 notebook 更偏“模型有没有跑通”；这里开始转向“模型为什么会成功或失败”。把分析配置单独收拢出来，可以让后面的病例研究、误报分类和变体实验都复用同一套规则。

### 输出怎么解读
运行后会打印重点病人、病例窗口长度和导出目录。如果这里显示的病人不是 `chb04 / chb08`，或者导出目录不是新的 `clinical_error_analysis_exports`，说明分析上下文还没有切到这次任务需要的版本。


In [27]:
# Block 27. Clinical Error Analysis v5 Config
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Tuple


@dataclass
class ErrorAnalysisConfig:
    focus_patients: Tuple[str, ...] = ("chb04", "chb08")
    delay_alert_threshold_s: int = 10
    transition_margin_s: int = 30
    case_pre_s: int = 20
    case_post_s: int = 40
    export_subdir: str = "clinical_error_analysis_exports_v5"
    report_file_name: str = "EE6019_Clinical_Error_Analysis_v5.md"
    reference_export_subdir: str = "clinical_error_analysis_exports_v4"
    run_focus_analysis: bool = True
    run_variant_ablation: bool = True
    run_global_variant_validation: bool = True
    max_macro_sensitivity_drop: float = 0.02
    max_focus_delay_increase_s: float = 0.0
    reference_variant_names: Tuple[str, ...] = (
        "baseline_rf_top30",
        "proxy_hybrid_plus_count_rf",
    )
    new_variant_names: Tuple[str, ...] = (
        "proxy_hybrid_plus_count_delayfirst_rf",
        "proxy_hybrid_plus_count_delayfirst_d2_rf",
    )
    proxy_feature_recipes: Dict[str, Tuple[str, ...]] = field(
        default_factory=lambda: {
            "proxy_hybrid_plus_count_rf": (
                "epoch_ptp_max",
                "high_amp_channel_count",
                "line_length_median",
                "broadband_lowfreq_ratio",
            ),
            "proxy_hybrid_plus_count_delayfirst_rf": (
                "epoch_ptp_max",
                "high_amp_channel_count",
                "line_length_median",
                "broadband_lowfreq_ratio",
            ),
            "proxy_hybrid_plus_count_delayfirst_d2_rf": (
                "epoch_ptp_max",
                "high_amp_channel_count",
                "line_length_median",
                "broadband_lowfreq_ratio",
            ),
        }
    )


ERROR_ANALYSIS_CFG = ErrorAnalysisConfig()
NOTEBOOK_WORK_ROOT = next((Path.home() / "Downloads").rglob("EE6019 Final Code-Baseline.ipynb")).parent
LEGACY_BASELINE_NOTEBOOK_PATH = NOTEBOOK_WORK_ROOT / "EE6019 Final Code-Baseline.ipynb"
LEGACY_CLINICAL_NOTEBOOK_PATH = NOTEBOOK_WORK_ROOT / "EE6019 Final Code- Clinical Error Analysis.ipynb"
V4_NOTEBOOK_PATH = NOTEBOOK_WORK_ROOT / "EE6019 Final Code- Clinical Error Analysis v4.ipynb"
V5_NOTEBOOK_PATH = NOTEBOOK_WORK_ROOT / "EE6019 Final Code- Clinical Error Analysis v5.ipynb"
V4_EXPORT_ROOT = NOTEBOOK_WORK_ROOT / ERROR_ANALYSIS_CFG.reference_export_subdir
CLINICAL_ERROR_EXPORT_ROOT = ensure_dir(NOTEBOOK_WORK_ROOT / ERROR_ANALYSIS_CFG.export_subdir)
CLINICAL_ERROR_REPORT_PATH = NOTEBOOK_WORK_ROOT / ERROR_ANALYSIS_CFG.report_file_name

PIPELINE_RESULTS["ERROR_ANALYSIS_CFG"] = ERROR_ANALYSIS_CFG
PIPELINE_RESULTS["NOTEBOOK_WORK_ROOT"] = str(NOTEBOOK_WORK_ROOT)
PIPELINE_RESULTS["LEGACY_BASELINE_NOTEBOOK_PATH"] = str(LEGACY_BASELINE_NOTEBOOK_PATH)
PIPELINE_RESULTS["LEGACY_CLINICAL_NOTEBOOK_PATH"] = str(LEGACY_CLINICAL_NOTEBOOK_PATH)
PIPELINE_RESULTS["V4_NOTEBOOK_PATH"] = str(V4_NOTEBOOK_PATH)
PIPELINE_RESULTS["V5_NOTEBOOK_PATH"] = str(V5_NOTEBOOK_PATH)
PIPELINE_RESULTS["V4_EXPORT_ROOT"] = str(V4_EXPORT_ROOT)
PIPELINE_RESULTS["clinical_error_export_root"] = str(CLINICAL_ERROR_EXPORT_ROOT)
PIPELINE_RESULTS["clinical_error_report_path"] = str(CLINICAL_ERROR_REPORT_PATH)

print("Block 27 v5 ready.")
print(f"  focus_patients          = {ERROR_ANALYSIS_CFG.focus_patients}")
print(f"  delay_alert_threshold_s = {ERROR_ANALYSIS_CFG.delay_alert_threshold_s}")
print(f"  reference_variants      = {ERROR_ANALYSIS_CFG.reference_variant_names}")
print(f"  new_variant_names       = {ERROR_ANALYSIS_CFG.new_variant_names}")
print(f"  v4_export_root          = {V4_EXPORT_ROOT}")
print(f"  export_root             = {CLINICAL_ERROR_EXPORT_ROOT}")
print(f"  proxy_feature_recipes   = {ERROR_ANALYSIS_CFG.proxy_feature_recipes}")


Block 27 v5 ready.


  focus_patients          = ('chb04', 'chb08')
  delay_alert_threshold_s = 10
  reference_variants      = ('baseline_rf_top30', 'proxy_hybrid_plus_count_rf')
  new_variant_names       = ('proxy_hybrid_plus_count_delayfirst_rf', 'proxy_hybrid_plus_count_delayfirst_d2_rf')
  v4_export_root          = <LOCAL_DATA_ROOT>\EE6019 新代码修改\clinical_error_analysis_exports_v4
  export_root             = <LOCAL_DATA_ROOT>\EE6019 新代码修改\clinical_error_analysis_exports_v5
  proxy_feature_recipes   = {'proxy_hybrid_plus_count_rf': ('epoch_ptp_max', 'high_amp_channel_count', 'line_length_median', 'broadband_lowfreq_ratio'), 'proxy_hybrid_plus_count_delayfirst_rf': ('epoch_ptp_max', 'high_amp_channel_count', 'line_length_median', 'broadband_lowfreq_ratio'), 'proxy_hybrid_plus_count_delayfirst_d2_rf': ('epoch_ptp_max', 'high_amp_channel_count', 'line_length_median', 'broadband_lowfreq_ratio')}


### Block 27 结果在论文里怎么用
这一块本身不是结果，而是“分析协议说明”。在论文里可以把它理解成误差分析的实验设置：重点病人是谁、延迟如何定义、什么叫转移态附近误报、导出图是从多大的时间窗截取的。


## Block 28：逐文件预测轨迹与事件级匹配

### 这个代码块在做什么
这里重新执行最终 tuned RF 的 LOSO 评估，但不只是保留病人级 summary，而是把每个测试文件里每个 epoch 的真实标签、概率、平滑后二值输出、持续时间约束后的最终报警都保留下来。随后，再把连续真实发作段和连续报警段做事件级匹配，标记为 TP / FP / FN / TN。

### 为什么这一步很关键
论文级误差分析不能只看 `Sensitivity` 和 `FAR/hr` 三个总指标。要解释 chb04 为什么延迟、chb08 为什么误报，必须回到“哪个文件、哪一段时间、概率是怎么爬升的、报警到底从哪里开始”的粒度。

### 输出怎么解读
运行后会生成三类核心结果：
1. `baseline_prediction_traces`：逐 epoch 轨迹表；
2. `baseline_pred_events_df`：预测事件表，重点看 `FP`；
3. `baseline_true_events_df`：真实事件表，重点看 `TP / FN / delay_s`。
另外还会和 Block 16 的 `rf10_loso_df` 做一致性核对，确认新的轨迹没有把原始最终 RF 的定义跑偏。


In [28]:
# 中文导读：这一块把“病人级 summary”展开成“逐文件、逐 epoch、逐事件”的完整轨迹。
# 这样后面才能做延迟分析、误报归因和病例图。
# =============================================================================
# Block 28. Prediction Trace Builder + Event Matching
# =============================================================================
from collections import defaultdict


def _split_vector_by_lengths(vec: np.ndarray, lengths: Sequence[int]) -> List[np.ndarray]:
    parts = []
    start = 0
    for length in lengths:
        stop = start + int(length)
        parts.append(vec[start:stop])
        start = stop
    return parts


def _overlap_interval(a_start: int, a_end: int, b_start: int, b_end: int) -> bool:
    return (a_start < b_end) and (b_start < a_end)


def _build_event_tables_from_trace(
    trace_df: pd.DataFrame,
    epoch_len_s: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if trace_df.empty:
        empty_trace = trace_df.copy()
        return empty_trace, pd.DataFrame(), pd.DataFrame()

    annotated_frames = []
    pred_rows = []
    true_rows = []

    for (patient_id, file_name), group in trace_df.groupby(["patient_id", "file_name"], sort=True):
        group = group.sort_values("epoch_index").reset_index(drop=True).copy()
        y_true = group["y_true"].to_numpy(dtype=np.int8)
        y_alarm = group["y_alarm"].to_numpy(dtype=np.int8)
        y_score = group["y_score"].to_numpy(dtype=float)

        true_starts, true_ends = extract_binary_runs(y_true)
        pred_starts, pred_ends = extract_binary_runs(y_alarm)

        true_events = []
        pred_events = []

        for idx, (start_epoch, end_epoch) in enumerate(zip(true_starts, true_ends), start=1):
            event_id = f"{patient_id}|{file_name}|T{idx:02d}"
            true_events.append(
                {
                    "patient_id": patient_id,
                    "file_name": file_name,
                    "true_event_id": event_id,
                    "true_start_epoch": int(start_epoch),
                    "true_end_epoch": int(end_epoch),
                    "true_onset_time_s": float(start_epoch * epoch_len_s),
                    "true_offset_time_s": float(end_epoch * epoch_len_s),
                    "duration_s": float((end_epoch - start_epoch) * epoch_len_s),
                }
            )

        for idx, (start_epoch, end_epoch) in enumerate(zip(pred_starts, pred_ends), start=1):
            event_id = f"{patient_id}|{file_name}|P{idx:02d}"
            score_slice = y_score[start_epoch:end_epoch]
            peak_prob = float(np.max(score_slice)) if len(score_slice) else np.nan
            pred_events.append(
                {
                    "patient_id": patient_id,
                    "file_name": file_name,
                    "pred_event_id": event_id,
                    "pred_start_epoch": int(start_epoch),
                    "pred_end_epoch": int(end_epoch),
                    "onset_time_s": float(start_epoch * epoch_len_s),
                    "offset_time_s": float(end_epoch * epoch_len_s),
                    "duration_s": float((end_epoch - start_epoch) * epoch_len_s),
                    "peak_prob": peak_prob,
                }
            )

        for pred in pred_events:
            matched_true_ids = []
            for true in true_events:
                if _overlap_interval(
                    pred["pred_start_epoch"],
                    pred["pred_end_epoch"],
                    true["true_start_epoch"],
                    true["true_end_epoch"],
                ):
                    matched_true_ids.append(true["true_event_id"])
            pred["matched_true_event_ids"] = matched_true_ids
            pred["event_role"] = "TP" if matched_true_ids else "FP"

        for true in true_events:
            matched_pred_ids = []
            delay_s = np.nan
            for pred in pred_events:
                if _overlap_interval(
                    pred["pred_start_epoch"],
                    pred["pred_end_epoch"],
                    true["true_start_epoch"],
                    true["true_end_epoch"],
                ):
                    matched_pred_ids.append(pred["pred_event_id"])
            if matched_pred_ids:
                true_slice = y_alarm[true["true_start_epoch"] : true["true_end_epoch"]]
                alarm_positions = np.flatnonzero(true_slice == 1)
                if alarm_positions.size > 0:
                    delay_s = float(int(alarm_positions[0]) * epoch_len_s)
            true["matched_pred_event_ids"] = matched_pred_ids
            true["event_role"] = "TP" if matched_pred_ids else "FN"
            true["delay_s"] = delay_s

        group["true_event_id"] = ""
        group["pred_event_id"] = ""
        group["event_role"] = "TN"

        for true in true_events:
            start = int(true["true_start_epoch"])
            end = int(true["true_end_epoch"])
            group.loc[start:end - 1, "true_event_id"] = true["true_event_id"]
            if true["event_role"] == "FN":
                group.loc[start:end - 1, "event_role"] = "FN"

        for pred in pred_events:
            start = int(pred["pred_start_epoch"])
            end = int(pred["pred_end_epoch"])
            group.loc[start:end - 1, "pred_event_id"] = pred["pred_event_id"]
            if pred["event_role"] == "FP":
                group.loc[start:end - 1, "event_role"] = "FP"

        for true in true_events:
            if true["event_role"] != "TP":
                continue
            start = int(true["true_start_epoch"])
            end = int(true["true_end_epoch"])
            group.loc[start:end - 1, "event_role"] = "TP"

        for pred in pred_events:
            if pred["event_role"] != "TP":
                continue
            start = int(pred["pred_start_epoch"])
            end = int(pred["pred_end_epoch"])
            group.loc[start:end - 1, "event_role"] = "TP"

        annotated_frames.append(group)
        pred_rows.extend(pred_events)
        true_rows.extend(true_events)

    trace_out = pd.concat(annotated_frames, ignore_index=True)
    pred_events_df = pd.DataFrame(pred_rows).sort_values(
        ["patient_id", "file_name", "onset_time_s"]
    ).reset_index(drop=True)
    true_events_df = pd.DataFrame(true_rows).sort_values(
        ["patient_id", "file_name", "true_onset_time_s"]
    ).reset_index(drop=True)
    return trace_out, pred_events_df, true_events_df


def evaluate_patient_loso_with_traces(
    patient_payload: Dict[str, Any],
    cfg: ExperimentConfig,
    model_name: str = "random_forest",
    top_k: Optional[int] = None,
    patient_seed_offset: int = 0,
    fold_topk_cache: Optional[Dict[Tuple[Any, ...], np.ndarray]] = None,
    alarm_postprocess_fn=None,
) -> Dict[str, Any]:
    patient_id = patient_payload["meta"]["patient_id"]
    file_payload = patient_payload["files"]
    seizure_files = sorted([f for f, d in file_payload.items() if d["has_seizure"]])
    bg_files = sorted([f for f, d in file_payload.items() if not d["has_seizure"]])

    if len(seizure_files) < 2:
        raise ValueError("Seizure file count < 2; cannot run LOSO.")

    X_all, y_all, file_row_spans = build_patient_matrix_index(patient_payload)
    feature_cfg_hash = topk_feature_cache_hash(cfg)

    rng = np.random.default_rng(cfg.eval.random_state + patient_seed_offset)
    shuffled_bg = bg_files.copy()
    rng.shuffle(shuffled_bg)
    bg_chunks = np.array_split(shuffled_bg, len(seizure_files))

    fold_rows = []
    all_y_true = []
    all_y_pred = []
    all_y_score = []
    all_selected_features = []
    trace_frames = []
    topk_cache_hits = 0
    topk_cache_misses = 0

    for outer_idx, test_seizure_file in enumerate(seizure_files):
        outer_train_seizure_files = seizure_files[:outer_idx] + seizure_files[outer_idx + 1 :]
        outer_test_bg_files = list(bg_chunks[outer_idx])
        outer_test_bg_set = set(outer_test_bg_files)
        outer_train_bg_files = [f for f in bg_files if f not in outer_test_bg_set]

        inner_train_seizure_files, inner_train_bg_files, val_seizure_files, val_bg_files = split_inner_validation_files(
            outer_train_seizure_files,
            outer_train_bg_files,
            seed=cfg.eval.random_state + patient_seed_offset + outer_idx,
        )

        inner_train_files = inner_train_seizure_files + inner_train_bg_files
        if len(inner_train_files) == 0:
            raise ValueError("No inner-train files available in this fold.")

        feature_indices = None
        if top_k is not None:
            selector_seed = cfg.eval.random_state + 1000 + outer_idx
            cache_key = (
                patient_id,
                feature_cfg_hash,
                int(top_k),
                int(outer_idx),
                tuple(inner_train_files),
                int(selector_seed),
            )

            if fold_topk_cache is not None and cache_key in fold_topk_cache:
                feature_indices = fold_topk_cache[cache_key]
                topk_cache_hits += 1
            else:
                selector_rng = np.random.default_rng(selector_seed)
                X_selector, y_selector = sample_rows_from_matrix_index(
                    X_all=X_all,
                    y_all=y_all,
                    file_row_spans=file_row_spans,
                    file_names=inner_train_files,
                    cfg=cfg,
                    rng=selector_rng,
                    feature_indices=None,
                )
                if len(y_selector) == 0:
                    raise ValueError("No inner-train samples available in this fold.")
                selector_model_name = model_name if model_name in {"random_forest", "xgboost"} else "random_forest"
                feature_indices = select_top_k_features_tree(
                    X_selector,
                    y_selector,
                    top_k=top_k,
                    cfg=cfg,
                    selector_model_name=selector_model_name,
                )
                if fold_topk_cache is not None:
                    fold_topk_cache[cache_key] = feature_indices
                topk_cache_misses += 1

            all_selected_features.append(feature_indices)

        use_fixed_threshold = bool(getattr(cfg.eval, "fixed_threshold_mode", False))
        if use_fixed_threshold:
            threshold = float(getattr(cfg.eval, "fixed_threshold_value", cfg.eval.default_threshold))
        else:
            threshold = cfg.eval.default_threshold
            if len(val_seizure_files) > 0:
                train_rng = np.random.default_rng(cfg.eval.random_state + 2000 + outer_idx)
                X_train_inner, y_train_inner = sample_rows_from_matrix_index(
                    X_all=X_all,
                    y_all=y_all,
                    file_row_spans=file_row_spans,
                    file_names=inner_train_files,
                    cfg=cfg,
                    rng=train_rng,
                    feature_indices=feature_indices,
                )
                inner_model = make_model(
                    model_name,
                    cfg,
                    scale_pos_weight=compute_scale_pos_weight(y_train_inner),
                )
                inner_model.fit(X_train_inner, y_train_inner)
                X_val, y_val = collect_rows_from_matrix_index(
                    X_all=X_all,
                    y_all=y_all,
                    file_row_spans=file_row_spans,
                    file_names=val_seizure_files + val_bg_files,
                    feature_indices=feature_indices,
                )
                val_scores = inner_model.predict_proba(X_val)[:, 1]
                threshold = choose_threshold_from_validation(y_val, val_scores, cfg)["threshold"]

        outer_rng = np.random.default_rng(cfg.eval.random_state + 3000 + outer_idx)
        X_train_outer, y_train_outer = sample_rows_from_matrix_index(
            X_all=X_all,
            y_all=y_all,
            file_row_spans=file_row_spans,
            file_names=outer_train_seizure_files + outer_train_bg_files,
            cfg=cfg,
            rng=outer_rng,
            feature_indices=feature_indices,
        )

        model = make_model(
            model_name,
            cfg,
            scale_pos_weight=compute_scale_pos_weight(y_train_outer),
        )
        model.fit(X_train_outer, y_train_outer)

        test_files = sorted([test_seizure_file] + outer_test_bg_files)
        X_test, y_test = collect_rows_from_matrix_index(
            X_all=X_all,
            y_all=y_all,
            file_row_spans=file_row_spans,
            file_names=test_files,
            feature_indices=feature_indices,
        )
        test_lengths = [int(len(file_payload[file_name]["y"])) for file_name in test_files]

        y_score_raw = model.predict_proba(X_test)[:, 1]
        y_score_smoothed = medfilt(y_score_raw, kernel_size=5)
        y_pred_raw = (y_score_raw >= threshold).astype(np.int8)
        y_pred_smooth = (y_score_smoothed >= threshold).astype(np.int8)
        y_alarm = apply_duration_constraint(y_pred_smooth, cfg.eval.min_duration_epochs)

        if alarm_postprocess_fn is not None:
            y_alarm = alarm_postprocess_fn(
                patient_id=patient_id,
                patient_payload=patient_payload,
                cfg=cfg,
                outer_idx=outer_idx,
                threshold=float(threshold),
                test_files=test_files,
                test_lengths=test_lengths,
                outer_train_files=outer_train_seizure_files + outer_train_bg_files,
                y_true=y_test,
                y_score_raw=y_score_raw,
                y_score_smoothed=y_score_smoothed,
                y_pred_raw=y_pred_raw,
                y_pred_smooth=y_pred_smooth,
                y_alarm=y_alarm,
            )

        fold_metrics = compute_event_metrics(y_test, y_alarm, cfg.feature.epoch_len_s)
        fold_rows.append(
            {
                "fold": outer_idx,
                "threshold": float(threshold),
                "test_seizure_file": test_seizure_file,
                "n_test_bg_files": len(outer_test_bg_files),
                "sensitivity": fold_metrics["sensitivity"],
                "far_per_hour": fold_metrics["far_per_hour"],
                "median_delay_s": fold_metrics["median_delay_s"],
            }
        )

        y_true_parts = _split_vector_by_lengths(y_test, test_lengths)
        y_score_raw_parts = _split_vector_by_lengths(y_score_raw, test_lengths)
        y_score_parts = _split_vector_by_lengths(y_score_smoothed, test_lengths)
        y_pred_raw_parts = _split_vector_by_lengths(y_pred_raw, test_lengths)
        y_pred_smooth_parts = _split_vector_by_lengths(y_pred_smooth, test_lengths)
        y_alarm_parts = _split_vector_by_lengths(y_alarm, test_lengths)

        for file_name, part_true, part_score_raw, part_score, part_pred_raw, part_pred_smooth, part_alarm in zip(
            test_files,
            y_true_parts,
            y_score_raw_parts,
            y_score_parts,
            y_pred_raw_parts,
            y_pred_smooth_parts,
            y_alarm_parts,
        ):
            n_rows = len(part_true)
            trace_frames.append(
                pd.DataFrame(
                    {
                        "patient_id": patient_id,
                        "file_name": file_name,
                        "fold": int(outer_idx),
                        "epoch_index": np.arange(n_rows, dtype=int),
                        "time_s": np.arange(n_rows, dtype=float) * cfg.feature.epoch_len_s,
                        "y_true": part_true.astype(np.int8),
                        "y_score_raw": part_score_raw.astype(float),
                        "y_score": part_score.astype(float),
                        "y_pred_raw": part_pred_raw.astype(np.int8),
                        "y_pred_smooth": part_pred_smooth.astype(np.int8),
                        "y_alarm": part_alarm.astype(np.int8),
                        "threshold": float(threshold),
                        "model_name": model_name,
                        "top_k": -1 if top_k is None else int(top_k),
                    }
                )
            )

        all_y_true.append(y_test)
        all_y_pred.append(y_alarm)
        all_y_score.append(y_score_smoothed)

    y_true_cat = np.concatenate(all_y_true)
    y_pred_cat = np.concatenate(all_y_pred)
    y_score_cat = np.concatenate(all_y_score)
    patient_metrics = compute_event_metrics(y_true_cat, y_pred_cat, cfg.feature.epoch_len_s)

    trace_df = pd.concat(trace_frames, ignore_index=True)
    trace_df, pred_events_df, true_events_df = _build_event_tables_from_trace(
        trace_df=trace_df,
        epoch_len_s=cfg.feature.epoch_len_s,
    )

    summary = {
        "Patient": patient_id,
        "Model": model_name,
        "TopK": top_k if top_k is not None else -1,
        "Hours": patient_metrics["hours"],
        "True_Seizures": patient_metrics["events"],
        "Sensitivity": patient_metrics["sensitivity"],
        "FAR_per_Hour": patient_metrics["far_per_hour"],
        "Mean_Delay_s": patient_metrics["mean_delay_s"],
        "Median_Delay_s": patient_metrics["median_delay_s"],
        "Median_Threshold": float(np.median([row["threshold"] for row in fold_rows])),
    }
    return {
        "summary": summary,
        "folds": pd.DataFrame(fold_rows),
        "y_true": y_true_cat,
        "y_pred": y_pred_cat,
        "y_score": y_score_cat,
        "selected_features": all_selected_features,
        "trace_df": trace_df,
        "pred_events_df": pred_events_df,
        "true_events_df": true_events_df,
        "topk_cache_hits": int(topk_cache_hits),
        "topk_cache_misses": int(topk_cache_misses),
    }


def evaluate_many_patients_with_traces(
    caches: Dict[str, Dict[str, Any]],
    cfg: ExperimentConfig,
    model_name: str = "random_forest",
    top_k: Optional[int] = None,
    patient_ids: Optional[Sequence[str]] = None,
    fold_topk_cache: Optional[Dict[Tuple[Any, ...], np.ndarray]] = None,
    alarm_postprocess_fn=None,
) -> Dict[str, Any]:
    patient_outputs = {}
    summary_rows = []
    trace_frames = []
    pred_frames = []
    true_frames = []
    total_cache_hits = 0
    total_cache_misses = 0

    patient_order = tuple(patient_ids) if patient_ids is not None else tuple(cfg.eval.patient_ids)
    for patient_id in patient_order:
        if patient_id not in caches:
            continue
        try:
            patient_numeric = int(str(patient_id).replace("chb", ""))
            result = evaluate_patient_loso_with_traces(
                patient_payload=caches[patient_id],
                cfg=cfg,
                model_name=model_name,
                top_k=top_k,
                patient_seed_offset=max(0, patient_numeric - 1) * 100,
                fold_topk_cache=fold_topk_cache,
                alarm_postprocess_fn=alarm_postprocess_fn,
            )
            patient_outputs[patient_id] = result
            summary_rows.append(result["summary"])
            trace_frames.append(result["trace_df"])
            pred_frames.append(result["pred_events_df"])
            true_frames.append(result["true_events_df"])
            total_cache_hits += int(result.get("topk_cache_hits", 0))
            total_cache_misses += int(result.get("topk_cache_misses", 0))
            print(
                f"[{patient_id}] trace-ready {model_name} | "
                f"Sens={result['summary']['Sensitivity']:.2%} | "
                f"FAR/hr={result['summary']['FAR_per_Hour']:.4f} | "
                f"MedianThr={result['summary']['Median_Threshold']:.3f}"
            )
        except Exception as exc:
            print(f"[警告] {patient_id} 轨迹评估失败: {exc}")

    summary_df = pd.DataFrame(summary_rows)
    trace_df = pd.concat(trace_frames, ignore_index=True) if trace_frames else pd.DataFrame()
    pred_events_df = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    true_events_df = pd.concat(true_frames, ignore_index=True) if true_frames else pd.DataFrame()
    return {
        "patient_outputs": patient_outputs,
        "summary_df": summary_df,
        "trace_df": trace_df,
        "pred_events_df": pred_events_df,
        "true_events_df": true_events_df,
        "topk_cache_hits": int(total_cache_hits),
        "topk_cache_misses": int(total_cache_misses),
    }


def _recompute_summary_from_trace(trace_df: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    rows = []
    for patient_id, group in trace_df.groupby("patient_id", sort=True):
        metrics = compute_event_metrics(
            y_true=group["y_true"].to_numpy(dtype=np.int8),
            y_pred_binary=group["y_alarm"].to_numpy(dtype=np.int8),
            epoch_len_s=cfg.feature.epoch_len_s,
        )
        rows.append(
            {
                "Patient": patient_id,
                "Sensitivity": metrics["sensitivity"],
                "FAR_per_Hour": metrics["far_per_hour"],
                "Mean_Delay_s": metrics["mean_delay_s"],
                "Median_Delay_s": metrics["median_delay_s"],
                "Hours": metrics["hours"],
                "True_Seizures": metrics["events"],
            }
        )
    return pd.DataFrame(rows).sort_values("Patient").reset_index(drop=True)


if not RUN_CLINICAL_ERROR_ANALYSIS:
    print("⏭ Block 28 skipped (RUN_CLINICAL_ERROR_ANALYSIS = False)")
else:
    baseline_trace_bundle = evaluate_many_patients_with_traces(
        caches=deploy_caches,
        cfg=deploy_cfg,
        model_name="random_forest",
        top_k=final_top_k,
        patient_ids=deploy_cfg.eval.patient_ids,
        fold_topk_cache={},
        alarm_postprocess_fn=None,
    )

    baseline_prediction_traces = baseline_trace_bundle["trace_df"].copy()
    baseline_pred_events_df = baseline_trace_bundle["pred_events_df"].copy()
    baseline_true_events_df = baseline_trace_bundle["true_events_df"].copy()
    baseline_trace_summary_df = baseline_trace_bundle["summary_df"].copy()
    baseline_trace_recomputed_df = _recompute_summary_from_trace(baseline_prediction_traces, deploy_cfg)

    if not rf10_loso_df.empty:
        trace_consistency_df = rf10_loso_df.merge(
            baseline_trace_summary_df[
                ["Patient", "Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "Median_Threshold"]
            ],
            on="Patient",
            suffixes=("_block16", "_trace"),
            how="outer",
        ).sort_values("Patient").reset_index(drop=True)
        for metric_name in ["Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "Median_Threshold"]:
            left = f"{metric_name}_block16"
            right = f"{metric_name}_trace"
            trace_consistency_df[f"{metric_name}_abs_diff"] = (
                pd.to_numeric(trace_consistency_df[left], errors="coerce")
                - pd.to_numeric(trace_consistency_df[right], errors="coerce")
            ).abs()
    else:
        trace_consistency_df = pd.DataFrame()

    PIPELINE_RESULTS["baseline_trace_bundle"] = baseline_trace_bundle
    PIPELINE_RESULTS["baseline_prediction_traces"] = baseline_prediction_traces
    PIPELINE_RESULTS["baseline_pred_events_df"] = baseline_pred_events_df
    PIPELINE_RESULTS["baseline_true_events_df"] = baseline_true_events_df
    PIPELINE_RESULTS["baseline_trace_summary_df"] = baseline_trace_summary_df
    PIPELINE_RESULTS["baseline_trace_recomputed_df"] = baseline_trace_recomputed_df
    PIPELINE_RESULTS["trace_consistency_df"] = trace_consistency_df

    display(baseline_trace_summary_df)
    display(trace_consistency_df)
    print("✅ Block 28 完成：逐文件轨迹和事件级匹配已生成。")


[chb01] trace-ready random_forest | Sens=100.00% | FAR/hr=0.6658 | MedianThr=0.499


[chb02] trace-ready random_forest | Sens=100.00% | FAR/hr=0.2552 | MedianThr=0.308


[chb03] trace-ready random_forest | Sens=100.00% | FAR/hr=0.5263 | MedianThr=0.672


[chb04] trace-ready random_forest | Sens=100.00% | FAR/hr=0.5919 | MedianThr=0.551


[chb05] trace-ready random_forest | Sens=100.00% | FAR/hr=0.1795 | MedianThr=0.482


...

[chb09] trace-ready random_forest | Sens=100.00% | FAR/hr=0.2171 | MedianThr=0.811


[chb10] trace-ready random_forest | Sens=100.00% | FAR/hr=0.1199 | MedianThr=0.811


,Patient,Model,TopK,Hours,True_Seizures,Sensitivity,FAR_per_Hour,Mean_Delay_s,Median_Delay_s,Median_Threshold
0,chb01,random_forest,30,40.551667,7.0,1.000000,0.665817,4.285714,2.0,0.499
1,chb02,random_forest,30,35.266111,3.0,1.000000,0.255203,3.333333,4.0,0.308
2,chb03,random_forest,30,38.001667,7.0,1.000000,0.526293,4.000000,2.0,0.672
3,chb04,random_forest,30,152.056111,4.0,1.000000,0.591887,35.500000,15.0,0.551
4,chb05,random_forest,30,39.002778,5.0,1.000000,0.179474,12.400000,6.0,0.482
5,chb06,random_forest,30,62.734444,9.0,0.777778,0.797010,2.857143,4.0,0.204
6,chb07,random_forest,30,67.051667,3.0,1.000000,0.089483,15.333333,18.0,0.690
7,chb08,random_forest,30,20.006111,5.0,1.000000,0.749771,11.200000,12.0,0.464
8,chb09,random_forest,30,59.870000,4.0,1.000000,0.217137,7.500000,6.0,0.811
9,chb10,random_forest,30,50.022778,7.0,1.000000,0.119945,3.142857,4.0,0.811


,Patient,Model,TopK,Hours,True_Seizures,Sensitivity_block16,FAR_per_Hour_block16,Mean_Delay_s_block16,Median_Delay_s_block16,Median_Threshold_block16,Sensitivity_trace,FAR_per_Hour_trace,Mean_Delay_s_trace,Median_Delay_s_trace,Median_Threshold_trace,Sensitivity_abs_diff,FAR_per_Hour_abs_diff,Mean_Delay_s_abs_diff,Median_Delay_s_abs_diff,Median_Threshold_abs_diff
0,chb01,random_forest,30,40.551667,7.0,1.000000,0.665817,4.285714,2.0,0.499,1.000000,0.665817,4.285714,2.0,0.499,0.0,0.0,0.0,0.0,0.0
1,chb02,random_forest,30,35.266111,3.0,1.000000,0.255203,3.333333,4.0,0.308,1.000000,0.255203,3.333333,4.0,0.308,0.0,0.0,0.0,0.0,0.0
2,chb03,random_forest,30,38.001667,7.0,1.000000,0.526293,4.000000,2.0,0.672,1.000000,0.526293,4.000000,2.0,0.672,0.0,0.0,0.0,0.0,0.0
3,chb04,random_forest,30,152.056111,4.0,1.000000,0.591887,35.500000,15.0,0.551,1.000000,0.591887,35.500000,15.0,0.551,0.0,0.0,0.0,0.0,0.0
4,chb05,random_forest,30,39.002778,5.0,1.000000,0.179474,12.400000,6.0,0.482,1.000000,0.179474,12.400000,6.0,0.482,0.0,0.0,0.0,0.0,0.0
5,chb06,random_forest,30,62.734444,9.0,0.777778,0.797010,2.857143,4.0,0.204,0.777778,0.797010,2.857143,4.0,0.204,0.0,0.0,0.0,0.0,0.0
6,chb07,random_forest,30,67.051667,3.0,1.000000,0.089483,15.333333,18.0,0.690,1.000000,0.089483,15.333333,18.0,0.690,0.0,0.0,0.0,0.0,0.0
7,chb08,random_forest,30,20.006111,5.0,1.000000,0.749771,11.200000,12.0,0.464,1.000000,0.749771,11.200000,12.0,0.464,0.0,0.0,0.0,0.0,0.0
8,chb09,random_forest,30,59.870000,4.0,1.000000,0.217137,7.500000,6.0,0.811,1.000000,0.217137,7.500000,6.0,0.811,0.0,0.0,0.0,0.0,0.0
9,chb10,random_forest,30,50.022778,7.0,1.000000,0.119945,3.142857,4.0,0.811,1.000000,0.119945,3.142857,4.0,0.811,0.0,0.0,0.0,0.0,0.0


✅ Block 28 完成：逐文件轨迹和事件级匹配已生成。


### Block 28 结果在论文里怎么用
这一块是后续所有“批判性反思”的证据底座。后面如果要说“某次报警是过渡态导致”“某位病人的高灵敏度伴随明显延迟”，都应该回到这里生成的轨迹和事件表来举证，而不是只引用宏观指标。


## Block 29：重点病例抽取与病例图导出

### 这个代码块在做什么
这一块专门把 `chb04` 的延迟 TP 和 `chb08` 的 FP 抽出来，做成两个重点表，并自动导出对应病例图。图里会同时画出概率曲线、报警轨迹、真实标签和滤波后的 EEG 波形，方便你直接拿来解释“这次模型为什么晚报 / 为什么误报”。

### 输出怎么解读
- `chb04_delay_events_df`：优先看 `delay_s` 最大的事件，说明最需要讨论的延迟案例。
- `chb08_fp_events_df`：优先看 `peak_prob` 高且持续时间长的误报，说明这类 FP 不是零碎噪声，而是模型真的被某种模式“说服了”。
- 导出的 PNG：适合直接放到论文或汇报的病例分析章节中。


In [29]:
# 中文导读：这里把最关键的两个失败模式变成可人工审阅的病例表和病例图。
# =============================================================================
# Block 29. Focused Case Extraction + Case Plot Export
# =============================================================================
def align_channels_window_for_error_analysis(
    raw: mne.io.BaseRaw,
    target_channels: Sequence[str],
    start_idx: int,
    stop_idx: int,
    policy: str = "strict",
) -> Tuple[np.ndarray, Dict[str, Any]]:
    normalized_targets = [normalize_channel_name(ch) for ch in target_channels]
    existing = {normalize_channel_name(ch): ch for ch in raw.ch_names}
    n_ch = len(normalized_targets)
    n_times = stop_idx - start_idx
    data = np.zeros((n_ch, n_times), dtype=float)
    missing_channels = []
    reversed_channels = []

    for i, target in enumerate(normalized_targets):
        if target in existing:
            data[i] = raw.get_data(picks=[existing[target]], start=start_idx, stop=stop_idx)[0]
            continue
        if "-" in target:
            reverse_target = "-".join(target.split("-")[::-1])
            if reverse_target in existing:
                data[i] = -raw.get_data(picks=[existing[reverse_target]], start=start_idx, stop=stop_idx)[0]
                reversed_channels.append(target)
                continue
        if policy == "zero_fill":
            missing_channels.append(target)
        else:
            raise ValueError(f"Missing target channel: {target}")

    return data, {
        "missing_channels": missing_channels,
        "missing_count": len(missing_channels),
        "reversed_channels": reversed_channels,
    }


def load_filtered_eeg_segment_for_event(
    cfg: ExperimentConfig,
    patient_id: str,
    file_name: str,
    start_time_s: float,
    end_time_s: float,
    anchor_time_s: float,
    plot_channels: Sequence[str],
) -> Tuple[np.ndarray, np.ndarray, List[str], Dict[str, Any]]:
    edf_path = Path(cfg.data_root) / patient_id / file_name
    raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
    raw = deduplicate_and_normalize_raw(raw)
    fs = int(raw.info["sfreq"])
    start_idx = max(0, int(start_time_s * fs))
    stop_idx = min(raw.n_times, int(end_time_s * fs))
    data_window, align_info = align_channels_window_for_error_analysis(
        raw=raw,
        target_channels=cfg.channels,
        start_idx=start_idx,
        stop_idx=stop_idx,
        policy=cfg.feature.channel_missing_policy,
    )
    raw.close()
    if cfg.feature.scale_to_uV:
        data_window = data_window * 1e6
    data_bp = bandpass_filter_multich(
        data_window,
        fs=fs,
        lowcut=cfg.feature.bandpass_low_hz,
        highcut=cfg.feature.bandpass_high_hz,
        method=cfg.feature.bandpass_method,
        butter_order=cfg.feature.butter_order,
    )

    target_norm = [normalize_channel_name(ch) for ch in cfg.channels]
    plot_indices = []
    plot_names = []
    for ch in plot_channels:
        ch_norm = normalize_channel_name(ch)
        if ch_norm in target_norm:
            plot_indices.append(target_norm.index(ch_norm))
            plot_names.append(ch)
    if len(plot_indices) == 0:
        plot_indices = list(range(min(4, data_bp.shape[0])))
        plot_names = [cfg.channels[idx] for idx in plot_indices]

    eeg_plot = data_bp[plot_indices]
    t_abs = np.arange(eeg_plot.shape[1]) / fs + (start_idx / fs)
    t_rel = t_abs - anchor_time_s
    return eeg_plot, t_rel, plot_names, align_info


def plot_case_from_event_row(
    event_row: pd.Series,
    trace_df: pd.DataFrame,
    cfg: ExperimentConfig,
    export_path: Path,
    case_pre_s: int,
    case_post_s: int,
    plot_channels: Sequence[str] = ("FP1-F7", "F7-T7", "T7-P7", "P7-O1"),
) -> None:
    patient_id = str(event_row["patient_id"])
    file_name = str(event_row["file_name"])
    if "true_start_epoch" in event_row.index:
        anchor_epoch = int(event_row["true_start_epoch"])
        end_epoch = int(event_row["true_end_epoch"])
        role = str(event_row.get("event_role", "TP"))
    else:
        anchor_epoch = int(event_row["pred_start_epoch"])
        end_epoch = int(event_row["pred_end_epoch"])
        role = str(event_row.get("event_role", "FP"))

    anchor_time_s = anchor_epoch * cfg.feature.epoch_len_s
    end_time_s = end_epoch * cfg.feature.epoch_len_s
    window_start_s = max(0.0, anchor_time_s - float(case_pre_s))
    window_end_s = end_time_s + float(case_post_s)

    trace_seg = trace_df[
        (trace_df["patient_id"] == patient_id)
        & (trace_df["file_name"] == file_name)
        & (trace_df["time_s"] >= window_start_s)
        & (trace_df["time_s"] <= window_end_s)
    ].copy()
    if trace_seg.empty:
        return
    trace_seg["time_rel_s"] = trace_seg["time_s"] - anchor_time_s

    eeg_plot, eeg_t_rel, eeg_names, _ = load_filtered_eeg_segment_for_event(
        cfg=cfg,
        patient_id=patient_id,
        file_name=file_name,
        start_time_s=window_start_s,
        end_time_s=window_end_s,
        anchor_time_s=anchor_time_s,
        plot_channels=plot_channels,
    )

    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(3, 1, height_ratios=[1.1, 1.0, 2.8], hspace=0.15)
    ax_prob = fig.add_subplot(gs[0, 0])
    ax_prob.plot(trace_seg["time_rel_s"], trace_seg["y_score"], color="#1f77b4", lw=2, label="Probability")
    ax_prob.axhline(float(trace_seg["threshold"].iloc[0]), color="#d62728", ls="--", lw=1.5, label="Threshold")
    ax_prob.axvline(0.0, color="black", ls=":", lw=1.2, label="Anchor")
    ax_prob.set_ylabel("Prob.")
    ax_prob.set_title(f"{patient_id} | {file_name} | {role}")
    ax_prob.grid(alpha=0.2)
    ax_prob.legend(loc="upper right")

    ax_state = fig.add_subplot(gs[1, 0], sharex=ax_prob)
    ax_state.step(trace_seg["time_rel_s"], trace_seg["y_true"], where="post", lw=2, label="y_true", color="#2ca02c")
    ax_state.step(trace_seg["time_rel_s"], trace_seg["y_pred_smooth"], where="post", lw=1.5, label="y_pred_smooth", color="#ff7f0e")
    ax_state.step(trace_seg["time_rel_s"], trace_seg["y_alarm"], where="post", lw=2, label="y_alarm", color="#d62728")
    ax_state.axvline(0.0, color="black", ls=":", lw=1.2)
    ax_state.set_ylim(-0.1, 1.2)
    ax_state.set_ylabel("State")
    ax_state.grid(alpha=0.2)
    ax_state.legend(loc="upper right", ncol=3)

    ax_eeg = fig.add_subplot(gs[2, 0], sharex=ax_prob)
    if eeg_plot.size > 0:
        spacing = max(50.0, float(np.nanpercentile(np.abs(eeg_plot), 95)) * 1.5)
        offsets = np.arange(eeg_plot.shape[0])[::-1] * spacing
        for name, signal, offset in zip(eeg_names, eeg_plot, offsets):
            ax_eeg.plot(eeg_t_rel, signal + offset, lw=1.0, label=name)
        ax_eeg.set_yticks(offsets)
        ax_eeg.set_yticklabels(eeg_names)
    ax_eeg.axvline(0.0, color="black", ls=":", lw=1.2)
    ax_eeg.set_xlabel("Time relative to event anchor (s)")
    ax_eeg.set_ylabel("Filtered EEG")
    ax_eeg.grid(alpha=0.15)

    ensure_dir(export_path.parent)
    fig.savefig(export_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


focus_patients = tuple(ERROR_ANALYSIS_CFG.focus_patients)
baseline_focus_trace_df = baseline_prediction_traces[
    baseline_prediction_traces["patient_id"].isin(focus_patients)
].copy()
baseline_focus_pred_events_df = baseline_pred_events_df[
    baseline_pred_events_df["patient_id"].isin(focus_patients)
].copy()
baseline_focus_true_events_df = baseline_true_events_df[
    baseline_true_events_df["patient_id"].isin(focus_patients)
].copy()

chb04_delay_events_df = (
    baseline_focus_true_events_df[
        (baseline_focus_true_events_df["patient_id"] == "chb04")
        & (baseline_focus_true_events_df["event_role"] == "TP")
        & (baseline_focus_true_events_df["delay_s"] > float(ERROR_ANALYSIS_CFG.delay_alert_threshold_s))
    ]
    .sort_values(["delay_s", "true_onset_time_s"], ascending=[False, True])
    .reset_index(drop=True)
)

chb08_fp_events_df = (
    baseline_focus_pred_events_df[
        (baseline_focus_pred_events_df["patient_id"] == "chb08")
        & (baseline_focus_pred_events_df["event_role"] == "FP")
    ]
    .sort_values(["peak_prob", "duration_s", "onset_time_s"], ascending=[False, False, True])
    .reset_index(drop=True)
)

case_export_rows = []
max_cases_per_type = 2
for idx, (_, row) in enumerate(chb04_delay_events_df.head(max_cases_per_type).iterrows(), start=1):
    out_path = CLINICAL_ERROR_EXPORT_ROOT / f"chb04_delay_case_{idx:02d}.png"
    plot_case_from_event_row(
        event_row=row,
        trace_df=baseline_prediction_traces,
        cfg=deploy_cfg,
        export_path=out_path,
        case_pre_s=ERROR_ANALYSIS_CFG.case_pre_s,
        case_post_s=ERROR_ANALYSIS_CFG.case_post_s,
    )
    case_export_rows.append(
        {
            "CaseType": "chb04_delay",
            "Patient": "chb04",
            "File": row["file_name"],
            "ImagePath": str(out_path),
            "Delay_s": row.get("delay_s", np.nan),
        }
    )

for idx, (_, row) in enumerate(chb08_fp_events_df.head(max_cases_per_type).iterrows(), start=1):
    out_path = CLINICAL_ERROR_EXPORT_ROOT / f"chb08_fp_case_{idx:02d}.png"
    plot_case_from_event_row(
        event_row=row,
        trace_df=baseline_prediction_traces,
        cfg=deploy_cfg,
        export_path=out_path,
        case_pre_s=ERROR_ANALYSIS_CFG.case_pre_s,
        case_post_s=ERROR_ANALYSIS_CFG.case_post_s,
    )
    case_export_rows.append(
        {
            "CaseType": "chb08_fp",
            "Patient": "chb08",
            "File": row["file_name"],
            "ImagePath": str(out_path),
            "PeakProb": row.get("peak_prob", np.nan),
            "Duration_s": row.get("duration_s", np.nan),
        }
    )

clinical_case_export_df = pd.DataFrame(case_export_rows)

PIPELINE_RESULTS["chb04_delay_events_df"] = chb04_delay_events_df
PIPELINE_RESULTS["chb08_fp_events_df"] = chb08_fp_events_df
PIPELINE_RESULTS["clinical_case_export_df"] = clinical_case_export_df

display(chb04_delay_events_df)
display(chb08_fp_events_df)
display(clinical_case_export_df)
print("✅ Block 29 完成：重点病例表和病例图已导出。")


,patient_id,file_name,true_event_id,true_start_epoch,true_end_epoch,true_onset_time_s,true_offset_time_s,duration_s,matched_pred_event_ids,event_role,delay_s
0,chb04,chb04_28.edf,chb04|chb04_28.edf|T02,1891,1949,3782.0,3898.0,116.0,[chb04|chb04_28.edf|P03],TP,104.0
1,chb04,chb04_28.edf,chb04|chb04_28.edf|T01,839,891,1678.0,1782.0,104.0,"[chb04|chb04_28.edf|P01, chb04|chb04_28.edf|P02]",TP,20.0


,patient_id,file_name,pred_event_id,pred_start_epoch,pred_end_epoch,onset_time_s,offset_time_s,duration_s,peak_prob,matched_true_event_ids,event_role
0,chb08,chb08_15.edf,chb08|chb08_15.edf|P08,568,581,1136.0,1162.0,26.0,0.851369,[],FP
1,chb08,chb08_11.edf,chb08|chb08_11.edf|P02,1570,1576,3140.0,3152.0,12.0,0.742955,[],FP
2,chb08,chb08_15.edf,chb08|chb08_15.edf|P01,49,53,98.0,106.0,8.0,0.735025,[],FP
3,chb08,chb08_02.edf,chb08|chb08_02.edf|P02,1449,1453,2898.0,2906.0,8.0,0.706177,[],FP
4,chb08,chb08_15.edf,chb08|chb08_15.edf|P02,109,112,218.0,224.0,6.0,0.643152,[],FP
5,chb08,chb08_15.edf,chb08|chb08_15.edf|P03,317,321,634.0,642.0,8.0,0.629757,[],FP
6,chb08,chb08_15.edf,chb08|chb08_15.edf|P05,477,481,954.0,962.0,8.0,0.623803,[],FP
7,chb08,chb08_15.edf,chb08|chb08_15.edf|P06,515,518,1030.0,1036.0,6.0,0.616274,[],FP
8,chb08,chb08_15.edf,chb08|chb08_15.edf|P07,546,549,1092.0,1098.0,6.0,0.613065,[],FP
9,chb08,chb08_02.edf,chb08|chb08_02.edf|P03,1469,1474,2938.0,2948.0,10.0,0.588635,[],FP


,CaseType,Patient,File,ImagePath,Delay_s,PeakProb,Duration_s
0,chb04_delay,chb04,chb04_28.edf,\EE6019 新代码修改\clinical...,104.0,NaN,NaN
1,chb04_delay,chb04,chb04_28.edf,\EE6019 新代码修改\clinical...,20.0,NaN,NaN
2,chb08_fp,chb08,chb08_15.edf,\EE6019 新代码修改\clinical...,NaN,0.851369,26.0
3,chb08_fp,chb08,chb08_11.edf,\EE6019 新代码修改\clinical...,NaN,0.742955,12.0


✅ Block 29 完成：重点病例表和病例图已导出。


### Block 29 结果在论文里怎么用
`chb04` 的延迟案例和 `chb08` 的误报案例，是“为什么仅靠总体灵敏度还不够”的最直接证据。论文里可以配合这两类图指出：同样达到高灵敏度，不同病人的失败模式可能完全不同，因此需要单独做病例级解释和定向改进。


## Block 30：误报原因分类与 TP / FP / FN 特征比较

### 这个代码块在做什么
这里先为相关文件提取一组“解释型特征”，包括频带功率汇总、同步特征汇总和工件代理特征。随后，对 `chb08` 的 FP 做启发式分类，再把 TP / FP / FN 三类事件的特征分布做成摘要表和效应量排序。

### 输出怎么解读
- `fp_reason_summary_df`：回答“chb08 的误报更像哪一类问题”。
- `event_type_feature_summary_df`：回答“TP / FP / FN 的典型特征差别是什么”。
- `event_type_effect_size_df`：回答“最能区分这些事件类型的因素是什么”，适合直接写进讨论部分。


In [30]:
# 中文导读：这一块的目标不是再训练模型，而是把失败模式解释得更像论文里的误差分析。
# =============================================================================
# Block 30. FP Reason Labeling + TP/FP/FN Feature Distribution
# =============================================================================
_ANALYSIS_FILE_FEATURE_CACHE = {}


def _standard_band_masks(freqs: np.ndarray) -> Dict[str, np.ndarray]:
    return {
        "delta": (freqs >= 0.5) & (freqs < 4.0),
        "theta": (freqs >= 4.0) & (freqs < 8.0),
        "alpha": (freqs >= 8.0) & (freqs < 13.0),
        "beta": (freqs >= 13.0) & (freqs < 30.0),
        "gamma": (freqs >= 30.0) & (freqs <= 40.0),
        "lowfreq": (freqs >= 0.5) & (freqs < 4.0),
        "broadband": (freqs >= 20.0) & (freqs <= 40.0),
    }


def build_analysis_epoch_features_for_file(
    cfg: ExperimentConfig,
    patient_id: str,
    file_name: str,
) -> pd.DataFrame:
    cache_key = (config_to_hash(cfg), patient_id, file_name)
    cached = _ANALYSIS_FILE_FEATURE_CACHE.get(cache_key)
    if cached is not None:
        return cached.copy()

    edf_path = Path(cfg.data_root) / patient_id / file_name
    raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
    raw = deduplicate_and_normalize_raw(raw)
    aligned_data, _ = align_channels(raw=raw, target_channels=cfg.channels, policy=cfg.feature.channel_missing_policy)
    fs = int(raw.info["sfreq"])
    raw.close()

    if cfg.feature.scale_to_uV:
        aligned_data = aligned_data * 1e6
    data_bp = bandpass_filter_multich(
        aligned_data,
        fs=fs,
        lowcut=cfg.feature.bandpass_low_hz,
        highcut=cfg.feature.bandpass_high_hz,
        method=cfg.feature.bandpass_method,
        butter_order=cfg.feature.butter_order,
    )

    epoch_samples = cfg.feature.epoch_len_s * fs
    n_epochs = data_bp.shape[1] // epoch_samples
    if n_epochs == 0:
        return pd.DataFrame()

    usable = data_bp[:, : n_epochs * epoch_samples]
    epochs = usable.reshape(data_bp.shape[0], n_epochs, epoch_samples).transpose(1, 0, 2)
    epochs_centered = epochs - epochs.mean(axis=2, keepdims=True)
    ptp = np.ptp(epochs, axis=2)
    line_length = np.sum(np.abs(np.diff(epochs, axis=2)), axis=2) / max(1, epochs.shape[2] - 1)

    fft_vals = np.fft.rfft(epochs_centered, axis=-1)
    power = fft_vals.real ** 2 + fft_vals.imag ** 2
    freqs = np.fft.rfftfreq(epoch_samples, d=1.0 / fs)
    masks = _standard_band_masks(freqs)

    band_agg = {}
    for band_name in ["delta", "theta", "alpha", "beta", "gamma"]:
        mask = masks[band_name]
        band_agg[f"{band_name}_power_mean"] = power[:, :, mask].sum(axis=-1).mean(axis=1)

    low_power = power[:, :, masks["lowfreq"]].sum(axis=-1).mean(axis=1)
    broadband_power = power[:, :, masks["broadband"]].sum(axis=-1).mean(axis=1)
    broadband_lowfreq_ratio = np.divide(
        broadband_power,
        np.maximum(low_power, 1e-6),
        out=np.zeros_like(broadband_power, dtype=float),
        where=np.maximum(low_power, 1e-6) > 0,
    )

    sync_values = []
    for idx_a, idx_b in get_synchrony_index_pairs(cfg):
        sig_a = epochs_centered[:, idx_a, :]
        sig_b = epochs_centered[:, idx_b, :]
        numerator = np.sum(sig_a * sig_b, axis=1)
        denominator = np.sqrt(np.sum(sig_a ** 2, axis=1) * np.sum(sig_b ** 2, axis=1))
        corr = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 0)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
        sync_values.append(corr)
    synchrony_mean = np.mean(np.stack(sync_values, axis=1), axis=1) if sync_values else np.zeros((n_epochs,), dtype=float)

    band_matrix = np.vstack(
        [
            band_agg["delta_power_mean"],
            band_agg["theta_power_mean"],
            band_agg["alpha_power_mean"],
            band_agg["beta_power_mean"],
            band_agg["gamma_power_mean"],
        ]
    ).T
    dominant_idx = np.argmax(band_matrix, axis=1)
    dominant_names = np.array(["delta", "theta", "alpha", "beta", "gamma"], dtype=object)[dominant_idx]
    dominant_fraction = np.max(band_matrix, axis=1) / np.maximum(np.sum(band_matrix, axis=1), 1e-6)

    out = pd.DataFrame(
        {
            "patient_id": patient_id,
            "file_name": file_name,
            "epoch_index": np.arange(n_epochs, dtype=int),
            "time_s": np.arange(n_epochs, dtype=float) * cfg.feature.epoch_len_s,
            "epoch_ptp_median": np.median(ptp, axis=1),
            "epoch_ptp_max": np.max(ptp, axis=1),
            "high_amp_channel_count": np.sum(ptp >= 250.0, axis=1),
            "line_length_median": np.median(line_length, axis=1),
            "broadband_lowfreq_ratio": broadband_lowfreq_ratio,
            "synchrony_mean": synchrony_mean,
            "dominant_band": dominant_names,
            "dominant_band_fraction": dominant_fraction,
        }
    )
    for key, values in band_agg.items():
        out[key] = values

    _ANALYSIS_FILE_FEATURE_CACHE[cache_key] = out.copy()
    return out


def build_analysis_epoch_features_for_pairs(
    cfg: ExperimentConfig,
    patient_file_pairs: Sequence[Tuple[str, str]],
) -> pd.DataFrame:
    frames = []
    for patient_id, file_name in sorted(set(patient_file_pairs)):
        frame = build_analysis_epoch_features_for_file(cfg, patient_id, file_name)
        if not frame.empty:
            frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def aggregate_event_feature_rows(
    event_df: pd.DataFrame,
    analysis_epoch_df: pd.DataFrame,
    trace_df: pd.DataFrame,
    event_type: str,
) -> pd.DataFrame:
    rows = []
    if event_df is None or event_df.empty:
        return pd.DataFrame()

    for _, event in event_df.iterrows():
        patient_id = str(event["patient_id"])
        file_name = str(event["file_name"])
        if event_type in {"TP", "FN"}:
            start_epoch = int(event["true_start_epoch"])
            end_epoch = int(event["true_end_epoch"])
            event_id = str(event["true_event_id"])
        else:
            start_epoch = int(event["pred_start_epoch"])
            end_epoch = int(event["pred_end_epoch"])
            event_id = str(event["pred_event_id"])

        feature_slice = analysis_epoch_df[
            (analysis_epoch_df["patient_id"] == patient_id)
            & (analysis_epoch_df["file_name"] == file_name)
            & (analysis_epoch_df["epoch_index"] >= start_epoch)
            & (analysis_epoch_df["epoch_index"] < end_epoch)
        ].copy()
        score_slice = trace_df[
            (trace_df["patient_id"] == patient_id)
            & (trace_df["file_name"] == file_name)
            & (trace_df["epoch_index"] >= start_epoch)
            & (trace_df["epoch_index"] < end_epoch)
        ].copy()
        if feature_slice.empty:
            continue

        dominant_mode = feature_slice["dominant_band"].mode()
        row = {
            "EventType": event_type,
            "patient_id": patient_id,
            "file_name": file_name,
            "event_id": event_id,
            "duration_s": float((end_epoch - start_epoch) * deploy_cfg.feature.epoch_len_s),
            "score_mean": float(score_slice["y_score"].mean()) if not score_slice.empty else np.nan,
            "score_peak": float(score_slice["y_score"].max()) if not score_slice.empty else np.nan,
            "dominant_band_mode": str(dominant_mode.iloc[0]) if len(dominant_mode) > 0 else "unknown",
            "dominant_band_fraction_mean": float(feature_slice["dominant_band_fraction"].mean()),
        }
        passthrough_cols = [
            "true_onset_time_s",
            "true_offset_time_s",
            "onset_time_s",
            "offset_time_s",
            "peak_prob",
        ]
        for passthrough_col in passthrough_cols:
            if passthrough_col in event.index:
                row[passthrough_col] = event[passthrough_col]
        numeric_cols = [
            "delta_power_mean",
            "theta_power_mean",
            "alpha_power_mean",
            "beta_power_mean",
            "gamma_power_mean",
            "synchrony_mean",
            "epoch_ptp_median",
            "epoch_ptp_max",
            "high_amp_channel_count",
            "line_length_median",
            "broadband_lowfreq_ratio",
        ]
        for col in numeric_cols:
            row[f"{col}_mean"] = float(feature_slice[col].mean())
            row[f"{col}_median"] = float(feature_slice[col].median())
        if "delay_s" in event.index:
            row["delay_s"] = float(event["delay_s"]) if not pd.isna(event["delay_s"]) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def _robust_standardized_diff(series_a: pd.Series, series_b: pd.Series) -> float:
    a = pd.to_numeric(series_a, errors="coerce").dropna().to_numpy(dtype=float)
    b = pd.to_numeric(series_b, errors="coerce").dropna().to_numpy(dtype=float)
    if len(a) == 0 or len(b) == 0:
        return np.nan
    pooled = np.sqrt((np.var(a) + np.var(b)) / 2.0)
    if pooled <= 1e-9:
        return np.nan
    return float((np.mean(a) - np.mean(b)) / pooled)


def build_feature_summary_table(event_feature_df: pd.DataFrame, features: Sequence[str]) -> pd.DataFrame:
    rows = []
    for event_type, group in event_feature_df.groupby("EventType", sort=True):
        for feature in features:
            vals = pd.to_numeric(group[feature], errors="coerce").dropna()
            if vals.empty:
                continue
            rows.append(
                {
                    "EventType": event_type,
                    "Feature": feature,
                    "N": int(len(vals)),
                    "Mean": float(vals.mean()),
                    "Median": float(vals.median()),
                    "IQR": float(vals.quantile(0.75) - vals.quantile(0.25)),
                }
            )
    return pd.DataFrame(rows)


def build_effect_size_table(event_feature_df: pd.DataFrame, features: Sequence[str]) -> pd.DataFrame:
    comparisons = [("TP", "FP"), ("TP", "FN")]
    rows = []
    for left, right in comparisons:
        left_df = event_feature_df[event_feature_df["EventType"] == left]
        right_df = event_feature_df[event_feature_df["EventType"] == right]
        for feature in features:
            rows.append(
                {
                    "Comparison": f"{left} vs {right}",
                    "Feature": feature,
                    "Mean_Left": pd.to_numeric(left_df.get(feature), errors="coerce").mean() if not left_df.empty else np.nan,
                    "Mean_Right": pd.to_numeric(right_df.get(feature), errors="coerce").mean() if not right_df.empty else np.nan,
                    "StdDiff": _robust_standardized_diff(left_df.get(feature, pd.Series(dtype=float)), right_df.get(feature, pd.Series(dtype=float))),
                }
            )
    effect_df = pd.DataFrame(rows)
    if effect_df.empty:
        return effect_df
    effect_df["AbsStdDiff"] = effect_df["StdDiff"].abs()
    return effect_df.sort_values(["Comparison", "AbsStdDiff"], ascending=[True, False]).reset_index(drop=True)


def classify_fp_reason(
    fp_row: pd.Series,
    true_events_patient_file: pd.DataFrame,
    file_duration_s: float,
    thresholds: Dict[str, float],
    transition_margin_s: int,
) -> str:
    onset_s = float(fp_row["onset_time_s"])
    offset_s = float(fp_row["offset_time_s"])
    near_boundary = onset_s <= transition_margin_s or (file_duration_s - offset_s) <= transition_margin_s
    near_true_transition = False
    if true_events_patient_file is not None and not true_events_patient_file.empty:
        boundary_times = np.concatenate(
            [
                true_events_patient_file["true_onset_time_s"].to_numpy(dtype=float),
                true_events_patient_file["true_offset_time_s"].to_numpy(dtype=float),
            ]
        )
        if boundary_times.size > 0:
            near_true_transition = bool(np.min(np.abs(boundary_times - onset_s)) <= transition_margin_s)
    if near_boundary or near_true_transition:
        return "transition_state"

    artifact_like = (
        (float(fp_row["epoch_ptp_max_mean"]) >= thresholds["epoch_ptp_max_q95"] and float(fp_row["line_length_median_mean"]) >= thresholds["line_length_q95"])
        or (float(fp_row["high_amp_channel_count_mean"]) >= thresholds["high_amp_q90"])
        or (float(fp_row["broadband_lowfreq_ratio_mean"]) >= thresholds["ratio_q90"] and float(fp_row["epoch_ptp_max_mean"]) >= thresholds["epoch_ptp_max_q75"])
    )
    if artifact_like:
        return "artifact_like"

    rhythmic_like = (
        str(fp_row.get("dominant_band_mode", "")) in {"theta", "alpha"}
        and float(fp_row.get("dominant_band_fraction_mean", np.nan)) >= 0.45
        and float(fp_row.get("line_length_median_mean", np.nan)) < thresholds["line_length_q95"]
        and float(fp_row.get("broadband_lowfreq_ratio_mean", np.nan)) < thresholds["ratio_q90"]
    )
    if rhythmic_like:
        return "rhythmic_non_ictal"
    return "uncertain"


event_file_pairs = set()
for frame in [
    baseline_true_events_df,
    baseline_pred_events_df[baseline_pred_events_df["event_role"] == "FP"],
]:
    if frame is None or frame.empty:
        continue
    for _, row in frame.iterrows():
        event_file_pairs.add((str(row["patient_id"]), str(row["file_name"])))

analysis_epoch_df = build_analysis_epoch_features_for_pairs(deploy_cfg, sorted(event_file_pairs))
baseline_trace_analysis_df = baseline_prediction_traces.merge(
    analysis_epoch_df,
    on=["patient_id", "file_name", "epoch_index", "time_s"],
    how="left",
)

tp_event_feature_df = aggregate_event_feature_rows(
    event_df=baseline_true_events_df[baseline_true_events_df["event_role"] == "TP"],
    analysis_epoch_df=analysis_epoch_df,
    trace_df=baseline_prediction_traces,
    event_type="TP",
)
fp_event_feature_df = aggregate_event_feature_rows(
    event_df=baseline_pred_events_df[baseline_pred_events_df["event_role"] == "FP"],
    analysis_epoch_df=analysis_epoch_df,
    trace_df=baseline_prediction_traces,
    event_type="FP",
)
fn_event_feature_df = aggregate_event_feature_rows(
    event_df=baseline_true_events_df[baseline_true_events_df["event_role"] == "FN"],
    analysis_epoch_df=analysis_epoch_df,
    trace_df=baseline_prediction_traces,
    event_type="FN",
)
event_feature_df = pd.concat(
    [tp_event_feature_df, fp_event_feature_df, fn_event_feature_df],
    ignore_index=True,
)

proxy_thresholds = {
    "epoch_ptp_max_q95": float(analysis_epoch_df["epoch_ptp_max"].quantile(0.95)) if not analysis_epoch_df.empty else np.nan,
    "epoch_ptp_max_q75": float(analysis_epoch_df["epoch_ptp_max"].quantile(0.75)) if not analysis_epoch_df.empty else np.nan,
    "line_length_q95": float(analysis_epoch_df["line_length_median"].quantile(0.95)) if not analysis_epoch_df.empty else np.nan,
    "ratio_q90": float(analysis_epoch_df["broadband_lowfreq_ratio"].quantile(0.90)) if not analysis_epoch_df.empty else np.nan,
    "high_amp_q90": float(analysis_epoch_df["high_amp_channel_count"].quantile(0.90)) if not analysis_epoch_df.empty else np.nan,
}

chb08_fp_feature_df = fp_event_feature_df[
    (fp_event_feature_df["patient_id"] == "chb08")
].copy()
fp_reason_rows = []
for _, row in chb08_fp_feature_df.iterrows():
    patient_true_events = baseline_true_events_df[
        (baseline_true_events_df["patient_id"] == row["patient_id"])
        & (baseline_true_events_df["file_name"] == row["file_name"])
    ].copy()
    file_trace = baseline_prediction_traces[
        (baseline_prediction_traces["patient_id"] == row["patient_id"])
        & (baseline_prediction_traces["file_name"] == row["file_name"])
    ]
    file_duration_s = float(file_trace["time_s"].max() + deploy_cfg.feature.epoch_len_s) if not file_trace.empty else np.nan
    fp_reason_rows.append(
        {
            **row.to_dict(),
            "fp_reason": classify_fp_reason(
                fp_row=row,
                true_events_patient_file=patient_true_events,
                file_duration_s=file_duration_s,
                thresholds=proxy_thresholds,
                transition_margin_s=ERROR_ANALYSIS_CFG.transition_margin_s,
            ),
        }
    )

chb08_fp_labeled_df = pd.DataFrame(fp_reason_rows)
if not chb08_fp_labeled_df.empty:
    chb08_fp_labeled_df = chb08_fp_labeled_df.sort_values(
        ["score_peak", "duration_s"], ascending=[False, False]
    ).reset_index(drop=True)
    fp_reason_summary_df = (
        chb08_fp_labeled_df.groupby("fp_reason", dropna=False)
        .size()
        .reset_index(name="Count")
        .sort_values(["Count", "fp_reason"], ascending=[False, True])
        .reset_index(drop=True)
    )
else:
    fp_reason_summary_df = pd.DataFrame(columns=["fp_reason", "Count", "Share"])
if not fp_reason_summary_df.empty:
    fp_reason_summary_df["Share"] = fp_reason_summary_df["Count"] / max(1, fp_reason_summary_df["Count"].sum())

summary_features = [
    "delta_power_mean_mean",
    "theta_power_mean_mean",
    "alpha_power_mean_mean",
    "beta_power_mean_mean",
    "gamma_power_mean_mean",
    "synchrony_mean_mean",
    "epoch_ptp_median_mean",
    "epoch_ptp_max_mean",
    "high_amp_channel_count_mean",
    "line_length_median_mean",
    "broadband_lowfreq_ratio_mean",
    "score_mean",
    "score_peak",
    "dominant_band_fraction_mean",
    "duration_s",
]
event_type_feature_summary_df = build_feature_summary_table(event_feature_df, summary_features)
event_type_effect_size_df = build_effect_size_table(event_feature_df, summary_features)

PIPELINE_RESULTS["analysis_epoch_df"] = analysis_epoch_df
PIPELINE_RESULTS["baseline_trace_analysis_df"] = baseline_trace_analysis_df
PIPELINE_RESULTS["chb08_fp_labeled_df"] = chb08_fp_labeled_df
PIPELINE_RESULTS["fp_reason_summary_df"] = fp_reason_summary_df
PIPELINE_RESULTS["event_feature_df"] = event_feature_df
PIPELINE_RESULTS["event_type_feature_summary_df"] = event_type_feature_summary_df
PIPELINE_RESULTS["event_type_effect_size_df"] = event_type_effect_size_df

display(chb08_fp_labeled_df)
display(fp_reason_summary_df)
display(event_type_feature_summary_df)
display(event_type_effect_size_df)
print("✅ Block 30 完成：误报分类与事件特征对比已生成。")


,EventType,patient_id,file_name,event_id,duration_s,score_mean,score_peak,dominant_band_mode,dominant_band_fraction_mean,onset_time_s,offset_time_s,peak_prob,delta_power_mean_mean,delta_power_mean_median,theta_power_mean_mean,theta_power_mean_median,alpha_power_mean_mean,alpha_power_mean_median,beta_power_mean_mean,beta_power_mean_median,gamma_power_mean_mean,gamma_power_mean_median,synchrony_mean_mean,synchrony_mean_median,epoch_ptp_median_mean,epoch_ptp_median_median,epoch_ptp_max_mean,epoch_ptp_max_median,high_amp_channel_count_mean,high_amp_channel_count_median,line_length_median_mean,line_length_median_median,broadband_lowfreq_ratio_mean,broadband_lowfreq_ratio_median,fp_reason
0,FP,chb08,chb08_15.edf,chb08|chb08_15.edf|P08,26.0,0.675047,0.851369,delta,0.820107,1136.0,1162.0,0.851369,1.356959e+09,1.313265e+09,2.677667e+08,1.829578e+08,2.760758e+07,3.169177e+07,1.411268e+07,1.320552e+07,4.283507e+06,2.032739e+06,0.410100,0.439754,487.193911,455.634918,847.188833,866.495726,21.076923,21.0,6.704381,6.357308,0.007890,0.003585,uncertain
1,FP,chb08,chb08_11.edf,chb08|chb08_11.edf|P02,12.0,0.623874,0.742955,delta,0.793382,3140.0,3152.0,0.742955,9.513201e+08,8.069490e+08,9.769097e+07,9.198646e+07,2.012600e+07,1.991209e+07,6.574430e+07,6.242517e+07,3.477987e+07,3.628695e+07,0.344433,0.309496,455.153930,450.710864,806.999622,802.529795,20.500000,21.0,10.888138,10.357479,0.099572,0.099158,transition_state
2,FP,chb08,chb08_15.edf,chb08|chb08_15.edf|P01,8.0,0.710919,0.735025,delta,0.825778,98.0,106.0,0.735025,1.146281e+09,1.249996e+09,1.877439e+08,1.994561e+08,3.296616e+07,2.948464e+07,1.124488e+07,1.193233e+07,1.775623e+06,1.743119e+06,0.496200,0.467372,473.795291,462.911913,851.502780,849.304811,20.500000,20.5,6.203252,6.193950,0.005238,0.004847,uncertain
3,FP,chb08,chb08_02.edf,chb08|chb08_02.edf|P02,8.0,0.636535,0.706177,delta,0.840395,2898.0,2906.0,0.706177,4.548002e+08,3.507988e+08,5.585972e+07,5.085092e+07,8.351148e+06,7.356996e+06,9.915990e+06,7.774228e+06,4.149616e+06,3.099872e+06,0.513468,0.499873,275.541289,258.374359,503.431657,487.657907,12.500000,12.0,5.374268,5.388335,0.024831,0.023155,uncertain
4,FP,chb08,chb08_15.edf,chb08|chb08_15.edf|P02,6.0,0.643152,0.643152,delta,0.835111,218.0,224.0,0.643152,1.266807e+09,1.395050e+09,1.921083e+08,2.028064e+08,3.838563e+07,3.477636e+07,1.241427e+07,1.403006e+07,2.012848e+06,2.154778e+06,0.277278,0.213746,464.530491,470.662203,737.141722,774.087280,21.333333,21.0,6.490227,6.414928,0.004642,0.004648,uncertain
5,FP,chb08,chb08_15.edf,chb08|chb08_15.edf|P03,8.0,0.629757,0.629757,delta,0.831734,634.0,642.0,0.629757,1.172224e+09,1.322572e+09,2.109331e+08,1.527025e+08,2.762761e+07,2.401260e+07,1.020027e+07,9.257281e+06,1.539612e+06,1.461922e+06,0.468252,0.442162,441.140939,446.799232,862.754512,877.804414,19.750000,20.0,5.942917,5.703333,0.004320,0.003600,uncertain
6,FP,chb08,chb08_15.edf,chb08|chb08_15.edf|P05,8.0,0.605082,0.623803,delta,0.853174,954.0,962.0,0.623803,1.394142e+09,1.532652e+09,2.050388e+08,1.877555e+08,3.094268e+07,3.003808e+07,1.096937e+07,1.126166e+07,1.836737e+06,1.627397e+06,0.400935,0.401166,467.661890,463.963772,836.665080,928.166197,21.000000,21.0,6.085632,5.962357,0.003972,0.004383,uncertain
7,FP,chb08,chb08_15.edf,chb08|chb08_15.edf|P06,6.0,0.609041,0.616274,delta,0.827300,1030.0,1036.0,0.616274,1.593086e+09,1.905251e+09,2.344492e+08,2.355467e+08,4.143511e+07,3.538452e+07,2.182353e+07,2.246760e+07,3.643376e+06,3.777909e+06,0.205605,0.176493,486.308933,490.344629,1150.907574,1122.269978,21.000000,21.0,6.002526,5.585730,0.007829,0.005814,uncertain
8,FP,chb08,chb08_15.edf,chb08|chb08_15.edf|P07,6.0,0.613065,0.613065,delta,0.839342,1092.0,1098.0,0.613065,9.249537e+08,7.361565e+08,1.516748e+08,1.364668e+08,2.178880e+07,1.560931e+07,9.275992e+06,7.862617e+06,1.354759e+06,1.570741e+06,0.388376,0.418218,390.255725,321.890830,626.216727,595.054803,18.333333,18.0,5.383589,4.809735,0.004656,0.004514,uncertain
9,FP,chb08,chb08_02.edf,chb08|chb08_02.edf|P03,10.0,0.520859,0.5

,fp_reason,Count,Share
0,uncertain,14,0.933333
1,transition_state,1,0.066667


,EventType,Feature,N,Mean,Median,IQR
0,FN,delta_power_mean_mean,2,4.065479e+08,4.065479e+08,5.744024e+07
1,FN,theta_power_mean_mean,2,2.094311e+08,2.094311e+08,1.034397e+08
2,FN,alpha_power_mean_mean,2,6.456585e+07,6.456585e+07,2.969789e+07
3,FN,beta_power_mean_mean,2,5.998494e+07,5.998494e+07,3.914384e+07
4,FN,gamma_power_mean_mean,2,2.032659e+07,2.032659e+07,1.709813e+07
5,FN,synchrony_mean_mean,2,3.854062e-01,3.854062e-01,3.602385e-02
6,FN,epoch_ptp_median_mean,2,3.714385e+02,3.714385e+02,1.073242e+02
7,FN,epoch_ptp_max_mean,2,7.259169e+02,7.259169e+02,2.241549e+02
8,FN,high_amp_channel_count_mean,2,1.515584e+01,1.515584e+01,5.701299e+00
9,FN,line_length_median_mean,2,1.010917e+01,1.010917e+01,4.176236e+00


,Comparison,Feature,Mean_Left,Mean_Right,StdDiff,AbsStdDiff
0,TP vs FN,score_peak,9.162750e-01,1.485148e-01,5.465089,5.465089
1,TP vs FN,score_mean,7.525817e-01,9.655815e-02,4.181989,4.181989
2,TP vs FN,duration_s,7.834615e+01,1.800000e+01,1.718695,1.718695
3,TP vs FN,delta_power_mean_mean,1.352605e+09,4.065479e+08,1.538054,1.538054
4,TP vs FN,epoch_ptp_max_mean,1.162657e+03,7.259169e+02,1.128762,1.128762
5,TP vs FN,epoch_ptp_median_mean,6.162786e+02,3.714385e+02,1.097475,1.097475
6,TP vs FN,line_length_median_mean,1.908189e+01,1.010917e+01,1.040021,1.040021
7,TP vs FN,theta_power_mean_mean,7.823652e+08,2.094311e+08,0.914651,0.914651
8,TP vs FN,beta_power_mean_mean,3.060187e+08,5.998494e+07,0.908194,0.908194
9,TP vs FN,high_amp_channel_count_mean,1.921766e+01,1.515584e+01,0.891600,0.891600


✅ Block 30 完成：误报分类与事件特征对比已生成。


### Block 30 结果在论文里怎么用
这一步提供的是“解释层证据”。如果 `artifact_like` 比例高，说明仅靠带通和持续时间约束并没有真正学会排除工件；如果 `rhythmic_non_ictal` 比例高，则说明模型会把节律性但非发作活动当成发作先兆。效应量表则能帮助你把这些判断落到更具体的特征差异上。


## Block 31：定向改进实验与最佳变体选择

### 这个代码块在做什么
这里不再做新的参数搜索，而是固定最终 tuned RF 的 `top_k=30` 和 RF 超参数，只在“特征输入 / 同步特征 / 工件抑制 / 通道子集”层面做小范围实验。先在 `chb04 / chb08` 上比较，再把最佳单一变体拿去全体 10 位病人验证一次。

### 输出怎么解读
- `focus_variant_compare_df`：看 4 个候选在重点病人上的变化。
- `best_variant_global_df`：看最终选中的变体在 10 位病人上的全局结果。
- `best_variant_vs_baseline_df`：看最佳变体是否真的值得纳入主模型，而不是只在个别病人上“局部好看”。


In [31]:
# Block 31. Timing-Focused Threshold Iteration v5
import ast
import io
from pathlib import Path

import nbformat


VARIANT_COMPONENT_FLAGS = {
    "baseline_rf_top30": {
        "UsesProxyFeatures": False,
        "ProxyRecipe": "baseline",
        "ProxyFeatureCount": 0,
        "ThresholdPolicy": "baseline_far_first",
        "MinDurationEpochs": 3,
    },
    "proxy_hybrid_plus_count_rf": {
        "UsesProxyFeatures": True,
        "ProxyRecipe": "hybrid_plus_amp_count",
        "ProxyFeatureCount": 4,
        "ThresholdPolicy": "far_first",
        "MinDurationEpochs": 3,
    },
    "proxy_hybrid_plus_count_delayfirst_rf": {
        "UsesProxyFeatures": True,
        "ProxyRecipe": "hybrid_plus_amp_count",
        "ProxyFeatureCount": 4,
        "ThresholdPolicy": "delay_first",
        "MinDurationEpochs": 3,
    },
    "proxy_hybrid_plus_count_delayfirst_d2_rf": {
        "UsesProxyFeatures": True,
        "ProxyRecipe": "hybrid_plus_amp_count",
        "ProxyFeatureCount": 4,
        "ThresholdPolicy": "delay_first",
        "MinDurationEpochs": 2,
    },
}


def _variant_flag_map(variant_name: str) -> Dict[str, Any]:
    return dict(VARIANT_COMPONENT_FLAGS.get(variant_name, {}))


def _attach_variant_flags(df: pd.DataFrame, variant_name: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df.copy()
    out = df.copy()
    for key, value in _variant_flag_map(variant_name).items():
        out[key] = value
    return out


def _subset_caches(caches: Dict[str, Dict[str, Any]], patient_ids: Sequence[str]) -> Dict[str, Dict[str, Any]]:
    return {pid: caches[pid] for pid in patient_ids if pid in caches}


def _read_notebook(path: Path):
    return nbformat.read(path.open("r", encoding="utf-8"), as_version=4)


def _parse_html_tables_from_output_cell(path: Path, cell_index: int) -> List[pd.DataFrame]:
    try:
        import lxml  # noqa: F401
    except Exception:
        return []
    if not path.exists():
        return []
    nb_local = _read_notebook(path)
    if cell_index >= len(nb_local.cells):
        return []
    tables = []
    for output in nb_local.cells[cell_index].get("outputs", []):
        data = output.get("data", {})
        html = data.get("text/html")
        if not html:
            continue
        try:
            tables.extend(pd.read_html(io.StringIO(html)))
        except ValueError:
            continue
    return tables


def _normalize_summary_table(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    drop_cols = [col for col in out.columns if str(col).startswith("Unnamed:")]
    if drop_cols:
        out = out.drop(columns=drop_cols)
    if "Patient" in out.columns:
        out["Patient"] = out["Patient"].astype(str)
    for col in [
        "TopK",
        "Hours",
        "True_Seizures",
        "Sensitivity",
        "FAR_per_Hour",
        "Mean_Delay_s",
        "Median_Delay_s",
        "Median_Threshold",
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def _extract_function_source_from_notebook(path: Path, func_name: str) -> str:
    if not path.exists():
        return ""
    nb_local = _read_notebook(path)
    module_src = "\n\n".join(cell.source for cell in nb_local.cells if cell.cell_type == "code")
    try:
        module = ast.parse(module_src)
    except SyntaxError:
        return ""
    lines = module_src.splitlines()
    for node in ast.walk(module):
        if isinstance(node, ast.FunctionDef) and node.name == func_name:
            start = node.lineno - 1
            end = getattr(node, "end_lineno", node.lineno)
            return "\n".join(lines[start:end]).strip()
    return ""


def _extract_assignment_value_from_notebook(path: Path, name: str):
    if not path.exists():
        return None
    nb_local = _read_notebook(path)
    module_src = "\n\n".join(cell.source for cell in nb_local.cells if cell.cell_type == "code")
    try:
        module = ast.parse(module_src)
    except SyntaxError:
        return None
    for node in module.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id == name:
                    try:
                        return ast.literal_eval(node.value)
                    except Exception:
                        return None
    return None

def _merge_source_tables(source_tables: Dict[str, pd.DataFrame], current_label: str) -> pd.DataFrame:
    metrics = [
        "Hours",
        "True_Seizures",
        "Sensitivity",
        "FAR_per_Hour",
        "Mean_Delay_s",
        "Median_Delay_s",
        "Median_Threshold",
    ]
    merged = None
    for source_name, table in source_tables.items():
        table = _normalize_summary_table(table)
        if table.empty:
            continue
        keep_cols = [col for col in ["Patient", *metrics] if col in table.columns]
        cur = table[keep_cols].copy()
        cur = cur.rename(columns={metric: f"{source_name}_{metric}" for metric in metrics if metric in cur.columns})
        merged = cur if merged is None else merged.merge(cur, on="Patient", how="outer")
    if merged is None:
        return pd.DataFrame()
    for source_name in ["legacy_baseline", "legacy_clinical"]:
        for metric in metrics:
            ref_col = f"{source_name}_{metric}"
            cur_col = f"{current_label}_{metric}"
            delta_col = f"{source_name}_minus_{current_label}_{metric}"
            if ref_col in merged.columns and cur_col in merged.columns:
                merged[delta_col] = pd.to_numeric(merged[ref_col], errors="coerce") - pd.to_numeric(merged[cur_col], errors="coerce")
    sort_key = merged["Patient"].map(lambda x: (x != "MACRO", x))
    return merged.assign(_sort_key=sort_key).sort_values("_sort_key").drop(columns="_sort_key").reset_index(drop=True)


def choose_threshold_from_validation(
    y_true: np.ndarray,
    y_score: np.ndarray,
    cfg: ExperimentConfig,
) -> Dict[str, float]:
    best = None
    fallback = None

    threshold_grid = cfg.eval.threshold_grid
    min_duration = cfg.eval.min_duration_epochs
    epoch_len_s = cfg.feature.epoch_len_s
    min_sens = cfg.eval.min_acceptable_sensitivity
    threshold_policy = str(getattr(cfg.eval, "threshold_selection_mode", "far_first"))

    for thr in threshold_grid:
        y_pred = (y_score >= thr).astype(np.int8)
        y_pred = apply_duration_constraint(y_pred, min_duration)
        metrics = compute_event_metrics(y_true, y_pred, epoch_len_s)

        row = {
            "threshold": float(thr),
            "sensitivity": metrics["sensitivity"],
            "far_per_hour": metrics["far_per_hour"],
            "median_delay_s": metrics["median_delay_s"],
        }

        _s = row["sensitivity"] if not np.isnan(row["sensitivity"]) else -1.0
        _f = row["far_per_hour"] if not np.isnan(row["far_per_hour"]) else float("inf")
        _d = row["median_delay_s"] if not np.isnan(row["median_delay_s"]) else float("inf")

        if threshold_policy == "delay_first":
            row_rank = (-_s, _d, _f, float(thr))
            best_row_rank = (_d, _f, -_s, float(thr))
        else:
            row_rank = (-_s, _f, _d, float(thr))
            best_row_rank = (_f, _d, -_s, float(thr))

        if fallback is None:
            fallback = row
            fallback_rank = row_rank
        else:
            if row_rank < fallback_rank:
                fallback = row
                fallback_rank = row_rank

        if (
            not np.isnan(metrics["sensitivity"])
            and metrics["sensitivity"] >= min_sens
        ):
            if best is None:
                best = row
                best_rank = best_row_rank
            else:
                if best_row_rank < best_rank:
                    best = row
                    best_rank = best_row_rank

    return best if best is not None else fallback


def _build_proxy_feature_stack(cfg: ExperimentConfig, patient_id: str, file_name: str, proxy_cols: Sequence[str]) -> np.ndarray:
    analysis_df = build_analysis_epoch_features_for_file(cfg, patient_id, file_name)
    X_aux = analysis_df[list(proxy_cols)].to_numpy(dtype=np.float32)
    y_aux = np.zeros((len(X_aux),), dtype=np.int8)
    X_aux_stacked, _ = temporal_stack_features(X_base=X_aux, y=y_aux, history_epochs=cfg.feature.history_epochs)
    return X_aux_stacked.astype(np.float32)


def _augment_payload_with_proxy_features(payload: Dict[str, Any], cfg: ExperimentConfig, proxy_cols: Sequence[str]) -> Dict[str, Any]:
    new_payload = {"meta": dict(payload["meta"]), "files": {}}
    for file_name, item in payload["files"].items():
        X_proxy = _build_proxy_feature_stack(cfg, payload["meta"]["patient_id"], file_name, proxy_cols=proxy_cols)
        new_item = dict(item)
        new_item["X"] = np.hstack([item["X"], X_proxy]).astype(np.float32)
        new_payload["files"][file_name] = new_item
    new_payload["meta"]["stacked_feature_dim"] = int(next(iter(new_payload["files"].values()))["X"].shape[1]) if new_payload["files"] else 0
    new_payload["meta"]["proxy_feature_recipe"] = tuple(proxy_cols)
    return new_payload


def prepare_variant_context(
    variant_name: str,
    base_cfg: ExperimentConfig,
    base_caches: Dict[str, Dict[str, Any]],
    patient_ids: Sequence[str],
) -> Tuple[ExperimentConfig, Dict[str, Dict[str, Any]], Any]:
    patient_ids = tuple(patient_ids)
    if variant_name == "baseline_rf_top30":
        cfg_local = base_cfg.variant()
        cfg_local.eval.patient_ids = patient_ids
        cfg_local.eval.threshold_selection_mode = "far_first"
        return cfg_local, _subset_caches(base_caches, patient_ids), None
    if variant_name in ERROR_ANALYSIS_CFG.proxy_feature_recipes:
        cfg_local = base_cfg.variant()
        cfg_local.eval.patient_ids = patient_ids
        cfg_local.eval.threshold_selection_mode = "far_first"
        cfg_local.eval.min_duration_epochs = 3
        if variant_name in {"proxy_hybrid_plus_count_delayfirst_rf", "proxy_hybrid_plus_count_delayfirst_d2_rf"}:
            cfg_local.eval.threshold_selection_mode = "delay_first"
        if variant_name == "proxy_hybrid_plus_count_delayfirst_d2_rf":
            cfg_local.eval.min_duration_epochs = 2
        base_subset = _subset_caches(base_caches, patient_ids)
        proxy_cols = ERROR_ANALYSIS_CFG.proxy_feature_recipes[variant_name]
        augmented_caches = {}
        for pid in patient_ids:
            if pid in base_subset:
                augmented_caches[pid] = _augment_payload_with_proxy_features(base_subset[pid], cfg_local, proxy_cols=proxy_cols)
        return cfg_local, augmented_caches, None
    raise ValueError(f"Unknown variant_name: {variant_name}")


def summarize_variant_bundle(bundle: Dict[str, Any], variant_name: str) -> pd.DataFrame:
    summary_df = bundle["summary_df"].copy()
    fp_counts = (
        bundle["pred_events_df"][bundle["pred_events_df"]["event_role"] == "FP"].groupby("patient_id").size().to_dict()
        if not bundle["pred_events_df"].empty
        else {}
    )
    summary_df["FP_events"] = summary_df["Patient"].map(fp_counts).fillna(0).astype(int)
    summary_df["Variant"] = variant_name
    return _attach_variant_flags(summary_df, variant_name)


def build_macro_variant_row(summary_df: pd.DataFrame, variant_name: str) -> pd.DataFrame:
    if summary_df.empty:
        return pd.DataFrame()
    macro_df = pd.DataFrame([
        {
            "Variant": variant_name,
            "Patient": "MACRO",
            "Patients": int(len(summary_df)),
            "Sensitivity": float(summary_df["Sensitivity"].mean()),
            "FAR_per_Hour": float(summary_df["FAR_per_Hour"].median()),
            "Mean_Delay_s": float(summary_df["Mean_Delay_s"].mean()),
            "Median_Delay_s": float(summary_df["Median_Delay_s"].median()),
            "FP_events": int(summary_df["FP_events"].sum()),
        }
    ])
    return _attach_variant_flags(macro_df, variant_name)

V2_EXPORT_ROOT = NOTEBOOK_WORK_ROOT / "clinical_error_analysis_exports_v2"
v2_reconciliation_path = V2_EXPORT_ROOT / "baseline_reconciliation_df.csv"
v2_notes_path = V2_EXPORT_ROOT / "baseline_reconciliation_notes_df.csv"

if v2_reconciliation_path.exists():
    baseline_reconciliation_df = pd.read_csv(v2_reconciliation_path)
    baseline_reconciliation_df = baseline_reconciliation_df.rename(
        columns={
            col: col.replace("current_v2_", "current_v5_")
            .replace("minus_current_v2_", "minus_current_v5_")
            for col in baseline_reconciliation_df.columns
        }
    )
    baseline_reconciliation_notes_df = pd.read_csv(v2_notes_path) if v2_notes_path.exists() else pd.DataFrame()
else:
    legacy_baseline_tables = _parse_html_tables_from_output_cell(LEGACY_BASELINE_NOTEBOOK_PATH, 26)
    legacy_clinical_tables = _parse_html_tables_from_output_cell(LEGACY_CLINICAL_NOTEBOOK_PATH, 52)

    legacy_baseline_patient_df = _normalize_summary_table(legacy_baseline_tables[0] if len(legacy_baseline_tables) >= 1 else pd.DataFrame())
    legacy_baseline_macro_df = _normalize_summary_table(legacy_baseline_tables[1] if len(legacy_baseline_tables) >= 2 else pd.DataFrame())
    legacy_clinical_patient_df = _normalize_summary_table(legacy_clinical_tables[0] if len(legacy_clinical_tables) >= 1 else pd.DataFrame())
    legacy_clinical_macro_df = _normalize_summary_table(legacy_clinical_tables[1] if len(legacy_clinical_tables) >= 2 else pd.DataFrame())
    current_v5_patient_df = _normalize_summary_table(rf10_loso_df.copy())
    current_v5_macro_df = _normalize_summary_table(rf10_macro_df.copy())

    baseline_reconciliation_sources = {
        "legacy_baseline": pd.concat([legacy_baseline_patient_df, legacy_baseline_macro_df], ignore_index=True),
        "legacy_clinical": pd.concat([legacy_clinical_patient_df, legacy_clinical_macro_df], ignore_index=True),
        "current_v5": pd.concat([current_v5_patient_df, current_v5_macro_df], ignore_index=True),
    }
    baseline_reconciliation_df = _merge_source_tables(baseline_reconciliation_sources, current_label="current_v5")

    notes_rows = []
    cache_schema_baseline = _extract_assignment_value_from_notebook(LEGACY_BASELINE_NOTEBOOK_PATH, "CACHE_SCHEMA_VERSION")
    cache_schema_clinical = _extract_assignment_value_from_notebook(LEGACY_CLINICAL_NOTEBOOK_PATH, "CACHE_SCHEMA_VERSION")
    notes_rows.append(
        {
            "Check": "CACHE_SCHEMA_VERSION",
            "LegacyBaseline": cache_schema_baseline,
            "LegacyClinical": cache_schema_clinical,
            "Match": bool(cache_schema_baseline == cache_schema_clinical),
            "Interpretation": "Same schema assignment argues against a parser-schema mismatch.",
        }
    )
    for func_name, interpretation in [
        ("parse_summary_to_dict", "Summary parser alignment"),
        ("build_patient_cache", "Cache construction alignment"),
        ("sample_rows_from_matrix_index", "Training row sampling helper alignment"),
        ("collect_rows_from_matrix_index", "Evaluation row collection helper alignment"),
        ("evaluate_patient_loso", "Fold evaluation logic alignment"),
    ]:
        baseline_src = _extract_function_source_from_notebook(LEGACY_BASELINE_NOTEBOOK_PATH, func_name)
        clinical_src = _extract_function_source_from_notebook(LEGACY_CLINICAL_NOTEBOOK_PATH, func_name)
        notes_rows.append(
            {
                "Check": func_name,
                "LegacyBaseline": "present" if baseline_src else "missing",
                "LegacyClinical": "present" if clinical_src else "missing",
                "Match": bool(baseline_src == clinical_src and baseline_src != ""),
                "Interpretation": interpretation,
            }
        )
    baseline_reconciliation_notes_df = pd.DataFrame(notes_rows)

focus_variant_names = [
    "baseline_rf_top30",
    "proxy_hybrid_plus_count_rf",
    "proxy_hybrid_plus_count_delayfirst_rf",
    "proxy_hybrid_plus_count_delayfirst_d2_rf",
]
focus_variant_rows = []
focus_variant_macro_rows = []
if ERROR_ANALYSIS_CFG.run_variant_ablation:
    for variant_name in focus_variant_names:
        print(f"\n===== Focus Variant: {variant_name} =====")
        variant_cfg, variant_caches, post_fn = prepare_variant_context(
            variant_name=variant_name,
            base_cfg=deploy_cfg,
            base_caches=deploy_caches,
            patient_ids=ERROR_ANALYSIS_CFG.focus_patients,
        )
        bundle = evaluate_many_patients_with_traces(
            caches=variant_caches,
            cfg=variant_cfg,
            model_name="random_forest",
            top_k=final_top_k,
            patient_ids=ERROR_ANALYSIS_CFG.focus_patients,
            fold_topk_cache={},
            alarm_postprocess_fn=post_fn,
        )
        patient_df = summarize_variant_bundle(bundle, variant_name)
        focus_variant_rows.append(patient_df)
        focus_variant_macro_rows.append(build_macro_variant_row(patient_df, variant_name))

focus_variant_compare_df = pd.concat(focus_variant_rows, ignore_index=True) if focus_variant_rows else pd.DataFrame()
focus_variant_macro_df = pd.concat(focus_variant_macro_rows, ignore_index=True) if focus_variant_macro_rows else pd.DataFrame()
if not focus_variant_compare_df.empty:
    baseline_focus_df = focus_variant_compare_df[focus_variant_compare_df["Variant"] == "baseline_rf_top30"].set_index("Patient").copy()
    variant_component_ablation_df = focus_variant_compare_df.merge(
        baseline_focus_df[["Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "FP_events"]].rename(
            columns={
                "Sensitivity": "Baseline_Sensitivity",
                "FAR_per_Hour": "Baseline_FAR_per_Hour",
                "Mean_Delay_s": "Baseline_Mean_Delay_s",
                "Median_Delay_s": "Baseline_Median_Delay_s",
                "FP_events": "Baseline_FP_events",
            }
        ),
        left_on="Patient",
        right_index=True,
        how="left",
    )
    variant_component_ablation_df["SensitivityDelta_vs_Baseline"] = pd.to_numeric(variant_component_ablation_df["Sensitivity"], errors="coerce") - pd.to_numeric(variant_component_ablation_df["Baseline_Sensitivity"], errors="coerce")
    variant_component_ablation_df["FARDelta_vs_Baseline"] = pd.to_numeric(variant_component_ablation_df["FAR_per_Hour"], errors="coerce") - pd.to_numeric(variant_component_ablation_df["Baseline_FAR_per_Hour"], errors="coerce")
    variant_component_ablation_df["DelayDelta_vs_Baseline"] = pd.to_numeric(variant_component_ablation_df["Mean_Delay_s"], errors="coerce") - pd.to_numeric(variant_component_ablation_df["Baseline_Mean_Delay_s"], errors="coerce")
    variant_component_ablation_df["FPEventDelta_vs_Baseline"] = pd.to_numeric(variant_component_ablation_df["FP_events"], errors="coerce") - pd.to_numeric(variant_component_ablation_df["Baseline_FP_events"], errors="coerce")
else:
    baseline_focus_df = pd.DataFrame()
    variant_component_ablation_df = pd.DataFrame()

best_focus_variant_name = "baseline_rf_top30"
variant_ranking_df = pd.DataFrame()
if not focus_variant_compare_df.empty and not baseline_focus_df.empty:
    candidate_rows = []
    for variant_name in [v for v in focus_variant_names if v != "baseline_rf_top30"]:
        variant_df = focus_variant_compare_df[focus_variant_compare_df["Variant"] == variant_name].set_index("Patient")
        if not all(pid in variant_df.index for pid in ERROR_ANALYSIS_CFG.focus_patients):
            continue
        row = {
            "Variant": variant_name,
            "passes_focus_sensitivity_guard": all(float(variant_df.loc[pid, "Sensitivity"]) >= float(baseline_focus_df.loc[pid, "Sensitivity"]) - 1e-12 for pid in ERROR_ANALYSIS_CFG.focus_patients),
            "focus_fp_event_improvement": int(baseline_focus_df["FP_events"].sum() - variant_df["FP_events"].sum()),
            "chb08_far_improvement": float(baseline_focus_df.loc["chb08", "FAR_per_Hour"] - variant_df.loc["chb08", "FAR_per_Hour"]),
            "chb04_delay_improvement": float(baseline_focus_df.loc["chb04", "Mean_Delay_s"] - variant_df.loc["chb04", "Mean_Delay_s"]),
            "focus_macro_sensitivity_delta": float(variant_df["Sensitivity"].mean() - baseline_focus_df["Sensitivity"].mean()),
            "focus_macro_far_delta": float(variant_df["FAR_per_Hour"].median() - baseline_focus_df["FAR_per_Hour"].median()),
            "focus_macro_delay_delta": float(variant_df["Mean_Delay_s"].mean() - baseline_focus_df["Mean_Delay_s"].mean()),
        }
        row.update(_variant_flag_map(variant_name))
        candidate_rows.append(row)
    variant_ranking_df = pd.DataFrame(candidate_rows)
    if not variant_ranking_df.empty:
        variant_ranking_df = variant_ranking_df.sort_values(
            ["passes_focus_sensitivity_guard", "chb08_far_improvement", "focus_fp_event_improvement", "chb04_delay_improvement"],
            ascending=[False, False, False, False],
        ).reset_index(drop=True)
        eligible_focus_df = variant_ranking_df[variant_ranking_df["passes_focus_sensitivity_guard"]]
        if not eligible_focus_df.empty:
            best_focus_variant_name = str(eligible_focus_df.iloc[0]["Variant"])

global_variant_names = [
    "proxy_hybrid_plus_count_rf",
    "proxy_hybrid_plus_count_delayfirst_rf",
    "proxy_hybrid_plus_count_delayfirst_d2_rf",
]
global_variant_rows = [summarize_variant_bundle(baseline_trace_bundle, "baseline_rf_top30")]
global_variant_macro_rows = [build_macro_variant_row(global_variant_rows[0], "baseline_rf_top30")]
if ERROR_ANALYSIS_CFG.run_global_variant_validation:
    for variant_name in global_variant_names:
        print(f"\n===== Global Variant Validation: {variant_name} =====")
        variant_cfg, variant_caches, post_fn = prepare_variant_context(
            variant_name=variant_name,
            base_cfg=deploy_cfg,
            base_caches=deploy_caches,
            patient_ids=deploy_cfg.eval.patient_ids,
        )
        bundle = evaluate_many_patients_with_traces(
            caches=variant_caches,
            cfg=variant_cfg,
            model_name="random_forest",
            top_k=final_top_k,
            patient_ids=deploy_cfg.eval.patient_ids,
            fold_topk_cache={},
            alarm_postprocess_fn=post_fn,
        )
        patient_df = summarize_variant_bundle(bundle, variant_name)
        global_variant_rows.append(patient_df)
        global_variant_macro_rows.append(build_macro_variant_row(patient_df, variant_name))

global_variant_compare_df = pd.concat(global_variant_rows, ignore_index=True) if global_variant_rows else pd.DataFrame()
global_variant_macro_df = pd.concat(global_variant_macro_rows, ignore_index=True) if global_variant_macro_rows else pd.DataFrame()
baseline_global_df = global_variant_compare_df[global_variant_compare_df["Variant"] == "baseline_rf_top30"].copy()
baseline_global_macro_df = global_variant_macro_df[global_variant_macro_df["Variant"] == "baseline_rf_top30"].copy()

global_guard_rows = []
adopted_global_variant_name = "baseline_rf_top30"
if not baseline_global_df.empty and not baseline_global_macro_df.empty:
    baseline_global_patient_df = baseline_global_df.set_index("Patient")
    baseline_macro_row = baseline_global_macro_df.iloc[0]
    for variant_name in global_variant_names:
        variant_df = global_variant_compare_df[global_variant_compare_df["Variant"] == variant_name].copy()
        variant_macro_df = global_variant_macro_df[global_variant_macro_df["Variant"] == variant_name].copy()
        if variant_df.empty or variant_macro_df.empty:
            continue
        variant_patient_df = variant_df.set_index("Patient")
        variant_macro_row = variant_macro_df.iloc[0]
        zero_sensitivity_patients = []
        for patient_id in baseline_global_patient_df.index:
            if patient_id in variant_patient_df.index:
                base_sens = float(baseline_global_patient_df.loc[patient_id, "Sensitivity"])
                cur_sens = float(variant_patient_df.loc[patient_id, "Sensitivity"])
                if base_sens > 0.0 and cur_sens <= 1e-12:
                    zero_sensitivity_patients.append(str(patient_id))
        macro_sensitivity_delta = float(variant_macro_row["Sensitivity"] - baseline_macro_row["Sensitivity"])
        macro_far_delta = float(variant_macro_row["FAR_per_Hour"] - baseline_macro_row["FAR_per_Hour"])
        chb08_far_delta = float(variant_patient_df.loc["chb08", "FAR_per_Hour"] - baseline_global_patient_df.loc["chb08", "FAR_per_Hour"])
        chb04_delay_delta = float(variant_patient_df.loc["chb04", "Mean_Delay_s"] - baseline_global_patient_df.loc["chb04", "Mean_Delay_s"])
        passes_zero_guard = len(zero_sensitivity_patients) == 0
        passes_macro_sensitivity_guard = macro_sensitivity_delta >= -float(ERROR_ANALYSIS_CFG.max_macro_sensitivity_drop) - 1e-12
        passes_macro_far_guard = macro_far_delta < -1e-12
        passes_chb08_far_guard = chb08_far_delta < -1e-12
        passes_chb04_delay_guard = chb04_delay_delta <= float(ERROR_ANALYSIS_CFG.max_focus_delay_increase_s) + 1e-12
        row = {
            "Variant": variant_name,
            "MacroSensitivity": float(variant_macro_row["Sensitivity"]),
            "MacroFAR_per_Hour": float(variant_macro_row["FAR_per_Hour"]),
            "MacroMeanDelay_s": float(variant_macro_row["Mean_Delay_s"]),
            "MacroSensitivityDelta": macro_sensitivity_delta,
            "MacroFARDelta": macro_far_delta,
            "chb08_far_delta": chb08_far_delta,
            "chb04_delay_delta": chb04_delay_delta,
            "new_zero_sensitivity_count": int(len(zero_sensitivity_patients)),
            "new_zero_sensitivity_patients": ", ".join(zero_sensitivity_patients),
            "passes_zero_guard": bool(passes_zero_guard),
            "passes_macro_sensitivity_guard": bool(passes_macro_sensitivity_guard),
            "passes_macro_far_guard": bool(passes_macro_far_guard),
            "passes_chb08_far_guard": bool(passes_chb08_far_guard),
            "passes_chb04_delay_guard": bool(passes_chb04_delay_guard),
            "passes_global_guard": bool(passes_zero_guard and passes_macro_sensitivity_guard and passes_macro_far_guard and passes_chb08_far_guard and passes_chb04_delay_guard),
        }
        row.update(_variant_flag_map(variant_name))
        global_guard_rows.append(row)
    global_guard_df = pd.DataFrame(global_guard_rows)
    if not global_guard_df.empty:
        global_guard_df = global_guard_df.sort_values(
            ["passes_global_guard", "MacroFARDelta", "MacroSensitivityDelta", "ProxyFeatureCount"],
            ascending=[False, True, False, True],
        ).reset_index(drop=True)
        eligible_global_df = global_guard_df[global_guard_df["passes_global_guard"]]
        if not eligible_global_df.empty:
            adopted_global_variant_name = str(eligible_global_df.iloc[0]["Variant"])
else:
    global_guard_df = pd.DataFrame()

focus_to_global_transfer_rows = []
if not variant_ranking_df.empty:
    for _, row in variant_ranking_df.iterrows():
        variant_name = str(row["Variant"])
        guard_match = global_guard_df[global_guard_df["Variant"] == variant_name]
        if guard_match.empty:
            continue
        global_row = guard_match.iloc[0]
        focus_to_global_transfer_rows.append(
            {
                "Variant": variant_name,
                "is_best_focus_variant": bool(variant_name == best_focus_variant_name),
                "is_adopted_global_variant": bool(variant_name == adopted_global_variant_name),
                "focus_passes_sensitivity_guard": bool(row["passes_focus_sensitivity_guard"]),
                "focus_chb08_far_improvement": float(row["chb08_far_improvement"]),
                "focus_chb04_delay_improvement": float(row["chb04_delay_improvement"]),
                "focus_fp_event_improvement": int(row["focus_fp_event_improvement"]),
                "focus_macro_sensitivity_delta": float(row["focus_macro_sensitivity_delta"]),
                "focus_macro_far_delta": float(row["focus_macro_far_delta"]),
                "global_macro_sensitivity_delta": float(global_row["MacroSensitivityDelta"]),
                "global_macro_far_delta": float(global_row["MacroFARDelta"]),
                "global_chb08_far_delta": float(global_row["chb08_far_delta"]),
                "global_chb04_delay_delta": float(global_row["chb04_delay_delta"]),
                "global_new_zero_sensitivity_count": int(global_row["new_zero_sensitivity_count"]),
                "passes_global_guard": bool(global_row["passes_global_guard"]),
                "ProxyRecipe": global_row["ProxyRecipe"],
                "ProxyFeatureCount": int(global_row["ProxyFeatureCount"]),
            }
        )
focus_to_global_transfer_df = pd.DataFrame(focus_to_global_transfer_rows)

best_variant_vs_baseline_df = global_variant_macro_df[global_variant_macro_df["Variant"].isin(["baseline_rf_top30", best_focus_variant_name])].copy()
if not best_variant_vs_baseline_df.empty:
    best_variant_vs_baseline_df["Selected_Best_Focus_Variant"] = best_variant_vs_baseline_df["Variant"].eq(best_focus_variant_name)
adopted_global_variant_vs_baseline_df = global_variant_macro_df[global_variant_macro_df["Variant"].isin(["baseline_rf_top30", adopted_global_variant_name])].copy()
if not adopted_global_variant_vs_baseline_df.empty:
    adopted_global_variant_vs_baseline_df["Selected_Adopted_Global_Variant"] = adopted_global_variant_vs_baseline_df["Variant"].eq(adopted_global_variant_name)

adopted_global_patient_delta_df = pd.DataFrame()
if adopted_global_variant_name != "baseline_rf_top30" and not global_variant_compare_df.empty:
    baseline_patient_df = global_variant_compare_df[global_variant_compare_df["Variant"] == "baseline_rf_top30"].set_index("Patient")
    adopted_patient_df = global_variant_compare_df[global_variant_compare_df["Variant"] == adopted_global_variant_name].set_index("Patient")
    joined = adopted_patient_df[["Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "FP_events"]].join(
        baseline_patient_df[["Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "FP_events"]],
        lsuffix="_adopted",
        rsuffix="_baseline",
    )
    for metric in ["Sensitivity", "FAR_per_Hour", "Mean_Delay_s", "Median_Delay_s", "FP_events"]:
        joined[f"{metric}_delta"] = pd.to_numeric(joined[f"{metric}_adopted"], errors="coerce") - pd.to_numeric(joined[f"{metric}_baseline"], errors="coerce")
    adopted_global_patient_delta_df = joined.reset_index()

PIPELINE_RESULTS["baseline_reconciliation_df"] = baseline_reconciliation_df
PIPELINE_RESULTS["baseline_reconciliation_notes_df"] = baseline_reconciliation_notes_df
PIPELINE_RESULTS["focus_variant_compare_df"] = focus_variant_compare_df
PIPELINE_RESULTS["focus_variant_macro_df"] = focus_variant_macro_df
PIPELINE_RESULTS["variant_component_ablation_df"] = variant_component_ablation_df
PIPELINE_RESULTS["variant_ranking_df"] = variant_ranking_df
PIPELINE_RESULTS["global_variant_compare_df"] = global_variant_compare_df
PIPELINE_RESULTS["global_variant_macro_df"] = global_variant_macro_df
PIPELINE_RESULTS["global_guard_df"] = global_guard_df
PIPELINE_RESULTS["focus_to_global_transfer_df"] = focus_to_global_transfer_df
PIPELINE_RESULTS["best_variant_vs_baseline_df"] = best_variant_vs_baseline_df
PIPELINE_RESULTS["adopted_global_variant_vs_baseline_df"] = adopted_global_variant_vs_baseline_df
PIPELINE_RESULTS["adopted_global_patient_delta_df"] = adopted_global_patient_delta_df
PIPELINE_RESULTS["best_focus_variant_name"] = best_focus_variant_name
PIPELINE_RESULTS["adopted_global_variant_name"] = adopted_global_variant_name

display(baseline_reconciliation_df)
display(baseline_reconciliation_notes_df)
display(variant_component_ablation_df)
display(variant_ranking_df)
display(global_guard_df)
display(focus_to_global_transfer_df)
display(adopted_global_patient_delta_df)
print(f"Block 31 v5 complete. best_focus_variant_name = {best_focus_variant_name}")
print(f"Block 31 v5 complete. adopted_global_variant_name = {adopted_global_variant_name}")



===== Focus Variant: baseline_rf_top30 =====


[chb04] trace-ready random_forest | Sens=100.00% | FAR/hr=0.5919 | MedianThr=0.551


[chb08] trace-ready random_forest | Sens=100.00% | FAR/hr=0.7498 | MedianThr=0.464

===== Focus Variant: proxy_hybrid_plus_count_rf =====


[chb04] trace-ready random_forest | Sens=100.00% | FAR/hr=0.5064 | MedianThr=0.568


[chb08] trace-ready random_forest | Sens=100.00% | FAR/hr=0.2499 | MedianThr=0.499

===== Focus Variant: proxy_hybrid_plus_count_delayfirst_rf =====


...

,Variant,passes_focus_sensitivity_guard,focus_fp_event_improvement,chb08_far_improvement,chb04_delay_improvement,focus_macro_sensitivity_delta,focus_macro_far_delta,focus_macro_delay_delta,UsesProxyFeatures,ProxyRecipe,ProxyFeatureCount,ThresholdPolicy,MinDurationEpochs
0,proxy_hybrid_plus_count_rf,True,22,0.499847,-17.5,0.0,-0.292671,8.55,True,hybrid_plus_amp_count,4,far_first,3
1,proxy_hybrid_plus_count_delayfirst_rf,True,-196,-2.599206,3.0,0.0,1.779689,-3.30,True,hybrid_plus_amp_count,4,delay_first,3
2,proxy_hybrid_plus_count_delayfirst_d2_rf,True,-232,-3.298992,3.0,0.0,2.201924,-3.30,True,hybrid_plus_amp_count,4,delay_first,2


,Variant,MacroSensitivity,MacroFAR_per_Hour,MacroMeanDelay_s,MacroSensitivityDelta,MacroFARDelta,chb08_far_delta,chb04_delay_delta,new_zero_sensitivity_count,new_zero_sensitivity_patients,passes_zero_guard,passes_macro_sensitivity_guard,passes_macro_far_guard,passes_chb08_far_guard,passes_chb04_delay_guard,passes_global_guard,UsesProxyFeatures,ProxyRecipe,ProxyFeatureCount,ThresholdPolicy,MinDurationEpochs
0,proxy_hybrid_plus_count_rf,0.966667,0.280952,11.074762,-0.011111,-0.109796,-0.499847,17.5,0,,True,True,True,True,False,False,True,hybrid_plus_amp_count,4,far_first,3
1,proxy_hybrid_plus_count_delayfirst_rf,0.966667,1.189708,5.868571,-0.011111,0.798961,2.599206,-3.0,0,,True,True,False,False,True,False,True,hybrid_plus_amp_count,4,delay_first,3
2,proxy_hybrid_plus_count_delayfirst_d2_rf,0.977778,1.242338,6.032857,0.000000,0.851590,3.298992,-3.0,0,,True,True,False,False,True,False,True,hybrid_plus_amp_count,4,delay_first,2


,Variant,is_best_focus_variant,is_adopted_global_variant,focus_passes_sensitivity_guard,focus_chb08_far_improvement,focus_chb04_delay_improvement,focus_fp_event_improvement,focus_macro_sensitivity_delta,focus_macro_far_delta,global_macro_sensitivity_delta,global_macro_far_delta,global_chb08_far_delta,global_chb04_delay_delta,global_new_zero_sensitivity_count,passes_global_guard,ProxyRecipe,ProxyFeatureCount
0,proxy_hybrid_plus_count_rf,True,False,True,0.499847,-17.5,22,0.0,-0.292671,-0.011111,-0.109796,-0.499847,17.5,0,False,hybrid_plus_amp_count,4
1,proxy_hybrid_plus_count_delayfirst_rf,False,False,True,-2.599206,3.0,-196,0.0,1.779689,-0.011111,0.798961,2.599206,-3.0,0,False,hybrid_plus_amp_count,4
2,proxy_hybrid_plus_count_delayfirst_d2_rf,False,False,True,-3.298992,3.0,-232,0.0,2.201924,0.000000,0.851590,3.298992,-3.0,0,False,hybrid_plus_amp_count,4


""


Block 31 v5 complete. best_focus_variant_name = proxy_hybrid_plus_count_rf
Block 31 v5 complete. adopted_global_variant_name = baseline_rf_top30


### Block 31 结果在论文里怎么用
这一步回答的是“改进值不值得正式纳入主模型”。如果某个变体只在 `chb04 / chb08` 上局部好看，但放回 10 位病人后整体 FAR 或延迟变差，就更适合作为讨论中的方向性建议；如果它在全局验证里也保持住优势，才更适合写成主模型升级方案。


## Block 32：导出临床误差分析报告与表格

### 这个代码块在做什么
这里把前面得到的重点表格、病例图索引和最终结论统一导出到新的 `clinical_error_analysis_exports` 目录，并额外生成一份 Markdown 报告，方便你直接作为论文讨论部分的草稿基础。

### 输出怎么解读
- CSV：适合后续画图、汇总或附录整理；
- Markdown 报告：适合直接转写成论文中的“Error Analysis / Discussion”章节；
- 图片索引：帮助你快速找到导出的病例图。


In [32]:
# Block 32. Clinical Error Analysis v5 Export
def _export_df_csv(df: pd.DataFrame, stem: str) -> Optional[Path]:
    if df is None or df.empty:
        return None
    path = CLINICAL_ERROR_EXPORT_ROOT / f"{stem}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def _df_to_markdown(df: pd.DataFrame, max_rows: int = 12) -> str:
    if df is None or df.empty:
        return "_empty_"
    preview = df.head(max_rows).copy()
    try:
        return preview.to_markdown(index=False)
    except Exception:
        return preview.to_csv(index=False)


exported_tables = {}
for name in [
    "chb04_delay_events_df",
    "chb08_fp_events_df",
    "clinical_case_export_df",
    "chb08_fp_labeled_df",
    "fp_reason_summary_df",
    "event_type_feature_summary_df",
    "event_type_effect_size_df",
    "baseline_reconciliation_df",
    "baseline_reconciliation_notes_df",
    "focus_variant_compare_df",
    "focus_variant_macro_df",
    "variant_component_ablation_df",
    "variant_ranking_df",
    "global_variant_compare_df",
    "global_variant_macro_df",
    "global_guard_df",
    "focus_to_global_transfer_df",
    "best_variant_vs_baseline_df",
    "adopted_global_variant_vs_baseline_df",
    "adopted_global_patient_delta_df",
]:
    exported_tables[name] = _export_df_csv(PIPELINE_RESULTS.get(name), name)

baseline_reconciliation_df = PIPELINE_RESULTS.get("baseline_reconciliation_df", pd.DataFrame())
baseline_reconciliation_notes_df = PIPELINE_RESULTS.get("baseline_reconciliation_notes_df", pd.DataFrame())
variant_component_ablation_df = PIPELINE_RESULTS.get("variant_component_ablation_df", pd.DataFrame())
global_guard_df = PIPELINE_RESULTS.get("global_guard_df", pd.DataFrame())
focus_to_global_transfer_df = PIPELINE_RESULTS.get("focus_to_global_transfer_df", pd.DataFrame())
variant_ranking_df = PIPELINE_RESULTS.get("variant_ranking_df", pd.DataFrame())
best_variant_vs_baseline_df = PIPELINE_RESULTS.get("best_variant_vs_baseline_df", pd.DataFrame())
adopted_global_variant_vs_baseline_df = PIPELINE_RESULTS.get("adopted_global_variant_vs_baseline_df", pd.DataFrame())
adopted_global_patient_delta_df = PIPELINE_RESULTS.get("adopted_global_patient_delta_df", pd.DataFrame())
fp_reason_summary_df = PIPELINE_RESULTS.get("fp_reason_summary_df", pd.DataFrame())
event_type_effect_size_df = PIPELINE_RESULTS.get("event_type_effect_size_df", pd.DataFrame())
best_focus_variant_name = PIPELINE_RESULTS.get("best_focus_variant_name", "baseline_rf_top30")
adopted_global_variant_name = PIPELINE_RESULTS.get("adopted_global_variant_name", "baseline_rf_top30")
uncertain_fp_share = PIPELINE_RESULTS.get("uncertain_fp_share", np.nan)
fp_taxonomy_is_descriptive_only = bool(PIPELINE_RESULTS.get("fp_taxonomy_is_descriptive_only", True))

chb04_delay_count = int(len(chb04_delay_events_df)) if "chb04_delay_events_df" in PIPELINE_RESULTS else 0
chb08_fp_count = int(len(chb08_fp_events_df)) if "chb08_fp_events_df" in PIPELINE_RESULTS else 0
top_effect_rows = event_type_effect_size_df.head(8) if event_type_effect_size_df is not None else pd.DataFrame()

notes_lookup = {}
if baseline_reconciliation_notes_df is not None and not baseline_reconciliation_notes_df.empty:
    notes_lookup = {str(row["Check"]): row.to_dict() for _, row in baseline_reconciliation_notes_df.iterrows()}

eval_drift_likely = bool(
    notes_lookup.get("parse_summary_to_dict", {}).get("Match", False)
    and notes_lookup.get("build_patient_cache", {}).get("Match", False)
    and not notes_lookup.get("evaluate_patient_loso", {}).get("Match", True)
)

report_lines = []
report_lines.append("# EE6019 Clinical Error Analysis v5")
report_lines.append("")
report_lines.append("## 1. Scope")
report_lines.append("- This v5 notebook keeps the v4 feature recipe fixed and only iterates on timing behaviour.")
report_lines.append("- The main question is whether a validation-only early-threshold policy can recover earlier seizure onset on `chb04` without destroying the FAR gains already achieved on `chb08`.")
report_lines.append("- No DL branch, no parameter scan, and no new multi-model comparison are introduced here.")
report_lines.append("")
report_lines.append("## 2. Baseline reconciliation")
report_lines.append("- Three baselines are compared side by side: the legacy baseline notebook output, the original clinical notebook output, and the freshly executed v5 baseline.")
report_lines.append(_df_to_markdown(baseline_reconciliation_df, max_rows=16) if not baseline_reconciliation_df.empty else "- Baseline reconciliation table could not be built.")
report_lines.append("")
if eval_drift_likely:
    report_lines.append("- Inference: the mismatch is still most consistent with evaluation-path drift rather than cache-schema drift, because the parser and cache builder appear aligned while `evaluate_patient_loso` differs between notebooks.")
else:
    report_lines.append("- The exact mismatch source remains partly unresolved, so the new variants should still be interpreted relative to the reproducible clinical baseline.")
report_lines.append("- Diagnostic checks:")
report_lines.append(_df_to_markdown(baseline_reconciliation_notes_df, max_rows=16))
report_lines.append("")
report_lines.append("## 3. Error-analysis anchors")
report_lines.append(f"- `chb04` late-TP cases above the delay threshold: **{chb04_delay_count}**")
report_lines.append(f"- `chb08` FP events under review: **{chb08_fp_count}**")
report_lines.append("- Interpretation target: `chb04` is still treated as a delayed-detection problem, while `chb08` is treated as an FP-burden problem.")
report_lines.append("")
report_lines.append("## 4. FP taxonomy quality")
report_lines.append(_df_to_markdown(fp_reason_summary_df, max_rows=10))
if pd.notna(uncertain_fp_share):
    report_lines.append(f"- `uncertain` share among labeled chb08 FP events: **{float(uncertain_fp_share):.2%}**")
if fp_taxonomy_is_descriptive_only:
    report_lines.append("- The taxonomy is still descriptive only, because the uncertain share remains too high for direct model-selection use.")
report_lines.append("")
report_lines.append("## 5. Feature-level contrast between TP / FP / FN")
report_lines.append("- v5 keeps the v4 count-based proxy recipe and changes only threshold-selection policy and minimum alarm duration.")
report_lines.append(_df_to_markdown(top_effect_rows, max_rows=8))
report_lines.append("")
report_lines.append("## 6. Proxy subset ablation on focus patients")
report_lines.append("- `proxy_hybrid_plus_count_rf` is the v4 control branch with FAR-first threshold selection and `min_duration_epochs = 3`.")
report_lines.append("- `proxy_hybrid_plus_count_delayfirst_rf` keeps the same features but ranks validation thresholds by delay first, then FAR.")
report_lines.append("- `proxy_hybrid_plus_count_delayfirst_d2_rf` adds the same delay-first threshold policy and relaxes the minimum alarm duration from 3 epochs to 2.")
report_lines.append(_df_to_markdown(variant_component_ablation_df, max_rows=20))
report_lines.append("")
report_lines.append("## 7. Focus ranking and global transfer")
report_lines.append(f"- Best focus-only candidate: **{best_focus_variant_name}**")
report_lines.append(_df_to_markdown(variant_ranking_df, max_rows=12))
report_lines.append("")
report_lines.append("### 7.1 Global guard results")
report_lines.append(_df_to_markdown(global_guard_df, max_rows=12))
report_lines.append("")
report_lines.append("### 7.2 Focus-to-global transfer table")
report_lines.append(_df_to_markdown(focus_to_global_transfer_df, max_rows=12))
report_lines.append("")
report_lines.append("## 8. Adoption decision")
report_lines.append(f"- Adopted global variant: **{adopted_global_variant_name}**")
report_lines.append("- Guard policy: no new zero-sensitivity patient, macro sensitivity drop <= 0.02, macro FAR must decrease, chb08 FAR must decrease, and chb04 delay must not worsen.")
if adopted_global_variant_name == "baseline_rf_top30":
    report_lines.append("- No proxy subset passed the full global guard, so the present conclusion remains diagnostic rather than deployable.")
else:
    report_lines.append("- At least one v5 timing variant passed the full global guard, so the threshold-policy direction remains viable.")
report_lines.append("")
report_lines.append("### 8.1 Baseline vs best focus variant")
report_lines.append(_df_to_markdown(best_variant_vs_baseline_df, max_rows=6))
report_lines.append("")
report_lines.append("### 8.2 Baseline vs adopted global variant")
report_lines.append(_df_to_markdown(adopted_global_variant_vs_baseline_df, max_rows=6))
report_lines.append("")
report_lines.append("### 8.3 Per-patient deltas for the adopted variant")
report_lines.append(_df_to_markdown(adopted_global_patient_delta_df, max_rows=12))
report_lines.append("")
report_lines.append("## 9. Code validity and leakage checks")
report_lines.append("- `top-k` features are still selected from inner-train files only.")
report_lines.append("- Threshold selection is still based on the inner-validation split only.")
report_lines.append("- No test-event labels are used to define the proxy subsets or the post-processing path.")
report_lines.append("- The main methodological risk in v5 is still not leakage but repeated focus-driven iteration around `chb04/chb08`.")
report_lines.append("")
report_lines.append("## 10. Export index")
for name, path in exported_tables.items():
    report_lines.append(f"- {name}: {path if path is not None else 'not exported'}")
report_lines.append(f"- Markdown memo: {CLINICAL_ERROR_REPORT_PATH}")

ensure_dir(CLINICAL_ERROR_REPORT_PATH.parent)
CLINICAL_ERROR_REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")
PIPELINE_RESULTS["clinical_error_report_lines"] = report_lines

print("Block 32 v5 complete.")
print(f"  report_path = {CLINICAL_ERROR_REPORT_PATH}")


Block 32 v5 complete.


  report_path = <LOCAL_DATA_ROOT>\EE6019 新代码修改\EE6019_Clinical_Error_Analysis_v5.md


### Block 32 结果在论文里怎么用
到这里，这份 notebook 不再只是“会跑的实验代码”，而是已经把病例图、误差归因、变体对比和文字报告整合起来了。你后面写论文时，可以直接把这里生成的 Markdown 报告作为讨论章节的初稿，再根据导师要求做语言和结构上的润色。
